In [1]:
# =============================================================================
# CELL 1 — GOLD BUSINESS MODEL
# Configuration + Silver Dependency Discovery & Validation
# =============================================================================

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

# -----------------------------------------------------------------------------
# 1. RUN METADATA
# -----------------------------------------------------------------------------

GOLD_RUN_ID = f"gold_business_model_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}_{uuid.uuid4().hex[:8]}"
GOLD_RUN_STARTED_UTC = datetime.now(timezone.utc)

SILVER_LAKEHOUSE = "lh_global_finance_silver"
GOLD_LAKEHOUSE   = "lh_global_finance_gold"

SILVER_SCHEMA = "dbo"
GOLD_SCHEMA   = "dbo"

# -----------------------------------------------------------------------------
# 2. EXPECTED SILVER DEPENDENCIES
#    These are the business entities required by the Gold model.
# -----------------------------------------------------------------------------

EXPECTED_DIMENSIONS = [
    "dim_date",
    "dim_company",
    "dim_customer",
    "dim_product"
]

EXPECTED_FACTS = [
    "fact_sales_order_line",
    "fact_invoice_line",
    "fact_payment",
    "fact_inventory_transaction",
    "fact_monthly_budget"
]

EXPECTED_SILVER_TABLES = EXPECTED_DIMENSIONS + EXPECTED_FACTS

# -----------------------------------------------------------------------------
# 3. DISCOVER ACTUAL SILVER TABLES
# -----------------------------------------------------------------------------

silver_tables_df = spark.sql(
    f"SHOW TABLES IN {SILVER_LAKEHOUSE}.{SILVER_SCHEMA}"
)

silver_table_names = [
    row["tableName"]
    for row in silver_tables_df.collect()
    if not row["isTemporary"]
]

silver_table_set = set(silver_table_names)

# -----------------------------------------------------------------------------
# 4. VALIDATE REQUIRED TABLE EXISTENCE
# -----------------------------------------------------------------------------

available_required_tables = [
    table
    for table in EXPECTED_SILVER_TABLES
    if table in silver_table_set
]

missing_required_tables = [
    table
    for table in EXPECTED_SILVER_TABLES
    if table not in silver_table_set
]

# -----------------------------------------------------------------------------
# 5. PROFILE REQUIRED TABLE ROW COUNTS
# -----------------------------------------------------------------------------

dependency_results = []

for table_name in EXPECTED_SILVER_TABLES:

    full_table_name = (
        f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.{table_name}"
    )

    if table_name in silver_table_set:

        try:
            row_count = spark.table(full_table_name).count()

            dependency_results.append({
                "table_name": table_name,
                "table_type": (
                    "DIMENSION"
                    if table_name in EXPECTED_DIMENSIONS
                    else "FACT"
                ),
                "exists": True,
                "row_count": int(row_count),
                "is_empty": row_count == 0,
                "status": (
                    "READY"
                    if row_count > 0
                    else "EMPTY"
                )
            })

        except Exception as e:

            dependency_results.append({
                "table_name": table_name,
                "table_type": (
                    "DIMENSION"
                    if table_name in EXPECTED_DIMENSIONS
                    else "FACT"
                ),
                "exists": True,
                "row_count": None,
                "is_empty": None,
                "status": f"READ_FAILED: {str(e)[:200]}"
            })

    else:

        dependency_results.append({
            "table_name": table_name,
            "table_type": (
                "DIMENSION"
                if table_name in EXPECTED_DIMENSIONS
                else "FACT"
            ),
            "exists": False,
            "row_count": None,
            "is_empty": None,
            "status": "MISSING"
        })

# -----------------------------------------------------------------------------
# 6. SUMMARISE DEPENDENCY HEALTH
# -----------------------------------------------------------------------------

missing_count = sum(
    1 for x in dependency_results
    if x["status"] == "MISSING"
)

empty_count = sum(
    1 for x in dependency_results
    if x["status"] == "EMPTY"
)

read_failure_count = sum(
    1 for x in dependency_results
    if str(x["status"]).startswith("READ_FAILED")
)

ready_count = sum(
    1 for x in dependency_results
    if x["status"] == "READY"
)

# -----------------------------------------------------------------------------
# 7. OUTPUT
# -----------------------------------------------------------------------------

print("=" * 100)
print("GOLD BUSINESS MODEL — SOURCE CONFIGURATION")
print("=" * 100)

print(f"Gold pipeline run ID          : {GOLD_RUN_ID}")
print(f"Run started UTC               : {GOLD_RUN_STARTED_UTC}")
print(f"Silver source                 : {SILVER_LAKEHOUSE}.{SILVER_SCHEMA}")
print(f"Gold target                   : {GOLD_LAKEHOUSE}.{GOLD_SCHEMA}")
print("-" * 100)

print(f"Silver dimensions expected    : {len(EXPECTED_DIMENSIONS)}")
print(f"Silver facts expected         : {len(EXPECTED_FACTS)}")
print(f"Total dependencies expected   : {len(EXPECTED_SILVER_TABLES)}")
print(f"Dependencies available        : {len(available_required_tables)}")
print(f"Dependencies ready            : {ready_count}")
print(f"Dependencies missing          : {missing_count}")
print(f"Empty dependencies            : {empty_count}")
print(f"Read failures                 : {read_failure_count}")
print(f"Total Silver tables discovered: {len(silver_table_names)}")

print("=" * 100)

# -----------------------------------------------------------------------------
# 8. FAIL FAST
# -----------------------------------------------------------------------------

critical_failure_count = (
    missing_count
    + empty_count
    + read_failure_count
)

if critical_failure_count > 0:

    print("DEPENDENCY DETAILS:")

    for item in dependency_results:
        print(
            f"{item['table_type']:<10} | "
            f"{item['table_name']:<32} | "
            f"{str(item['row_count']):<12} | "
            f"{item['status']}"
        )

    raise RuntimeError(
        f"GOLD BUSINESS MODEL NOT READY — "
        f"{critical_failure_count} critical Silver dependency issue(s) found."
    )

print("GOLD BUSINESS MODEL SOURCE CONFIGURATION: READY")
print("=" * 100)

# -----------------------------------------------------------------------------
# 9. DISPLAY DEPENDENCIES
# -----------------------------------------------------------------------------

for item in dependency_results:
    print(
        f"{item['table_type']:<10} | "
        f"{item['table_name']:<32} | "
        f"rows = {item['row_count']:<10} | "
        f"{item['status']}"
    )

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 3, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — SOURCE CONFIGURATION
Gold pipeline run ID          : gold_business_model_20260814T135536_ed789cef
Run started UTC               : 2026-08-14 13:55:36.808303+00:00
Silver source                 : lh_global_finance_silver.dbo
Gold target                   : lh_global_finance_gold.dbo
----------------------------------------------------------------------------------------------------
Silver dimensions expected    : 4
Silver facts expected         : 5
Total dependencies expected   : 9
Dependencies available        : 9
Dependencies ready            : 9
Dependencies missing          : 0
Empty dependencies            : 0
Read failures                 : 0
Total Silver tables discovered: 12
GOLD BUSINESS MODEL SOURCE CONFIGURATION: READY
DIMENSION  | dim_date                         | rows = 5844       | READY
DIMENSION  | dim_company                      | rows = 3          | READY
DIMENSION  | dim_customer                     | rows = 1961       | READY
DIMENSION  | dim_

In [2]:
# =============================================================================
# CELL 2 — GOLD BUSINESS MODEL
# Silver Source Profiling + Grain Inspection
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, IntegerType, LongType, ShortType,
    DecimalType, DoubleType, FloatType,
    DateType, TimestampType
)

print("=" * 100)
print("GOLD BUSINESS MODEL — SILVER SOURCE PROFILING")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. PROFILE CONFIGURATION
# -----------------------------------------------------------------------------

PROFILE_TABLES = EXPECTED_SILVER_TABLES

profile_results = []
schema_results = []

# -----------------------------------------------------------------------------
# 2. PROFILE EACH SILVER TABLE
# -----------------------------------------------------------------------------

for table_name in PROFILE_TABLES:

    full_name = f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.{table_name}"

    try:
        df = spark.table(full_name)

        row_count = df.count()
        column_count = len(df.columns)

        # ---------------------------------------------------------------------
        # Identify likely business/surrogate key columns
        # ---------------------------------------------------------------------

        key_columns = [
            c for c in df.columns
            if c.endswith("_key")
        ]

        # ---------------------------------------------------------------------
        # Identify date/timestamp columns
        # ---------------------------------------------------------------------

        date_columns = [
            field.name
            for field in df.schema.fields
            if isinstance(field.dataType, (DateType, TimestampType))
        ]

        # ---------------------------------------------------------------------
        # Null counts for key columns
        # ---------------------------------------------------------------------

        null_key_count = 0

        if key_columns:

            null_expressions = [
                F.sum(
                    F.when(F.col(c).isNull(), 1).otherwise(0)
                ).alias(c)
                for c in key_columns
            ]

            null_result = (
                df
                .agg(*null_expressions)
                .collect()[0]
                .asDict()
            )

            null_key_count = sum(
                int(v or 0)
                for v in null_result.values()
            )

        # ---------------------------------------------------------------------
        # Determine approximate grain candidates
        # ---------------------------------------------------------------------

        likely_primary_keys = []

        for c in key_columns:

            distinct_count = (
                df.select(c)
                  .where(F.col(c).isNotNull())
                  .distinct()
                  .count()
            )

            if distinct_count == row_count:
                likely_primary_keys.append(c)

        # ---------------------------------------------------------------------
        # Date range
        # ---------------------------------------------------------------------

        min_date = None
        max_date = None
        primary_date_column = None

        preferred_date_columns = [
            c for c in [
                "date",
                "full_date",
                "order_date",
                "invoice_date",
                "payment_date",
                "transaction_date",
                "budget_month"
            ]
            if c in df.columns
        ]

        if preferred_date_columns:
            primary_date_column = preferred_date_columns[0]

        elif date_columns:
            primary_date_column = date_columns[0]

        if primary_date_column:

            date_stats = (
                df.agg(
                    F.min(F.col(primary_date_column)).alias("min_date"),
                    F.max(F.col(primary_date_column)).alias("max_date")
                )
                .collect()[0]
            )

            min_date = date_stats["min_date"]
            max_date = date_stats["max_date"]

        # ---------------------------------------------------------------------
        # Save table profile
        # ---------------------------------------------------------------------

        profile_results.append({
            "table_name": table_name,
            "table_type":
                "DIMENSION"
                if table_name in EXPECTED_DIMENSIONS
                else "FACT",
            "row_count": row_count,
            "column_count": column_count,
            "key_columns": ", ".join(key_columns),
            "likely_primary_key": ", ".join(likely_primary_keys),
            "null_key_values": null_key_count,
            "primary_date_column": primary_date_column,
            "min_date": str(min_date) if min_date is not None else None,
            "max_date": str(max_date) if max_date is not None else None,
            "status": "PROFILED"
        })

        # ---------------------------------------------------------------------
        # Save complete schema inventory
        # ---------------------------------------------------------------------

        for field in df.schema.fields:

            schema_results.append({
                "table_name": table_name,
                "column_name": field.name,
                "data_type": field.dataType.simpleString(),
                "nullable": field.nullable
            })

    except Exception as e:

        profile_results.append({
            "table_name": table_name,
            "table_type":
                "DIMENSION"
                if table_name in EXPECTED_DIMENSIONS
                else "FACT",
            "row_count": None,
            "column_count": None,
            "key_columns": None,
            "likely_primary_key": None,
            "null_key_values": None,
            "primary_date_column": None,
            "min_date": None,
            "max_date": None,
            "status": f"FAILED: {str(e)[:250]}"
        })


# -----------------------------------------------------------------------------
# 3. VALIDATION
# -----------------------------------------------------------------------------

profiling_failures = [
    x for x in profile_results
    if x["status"] != "PROFILED"
]

empty_tables = [
    x for x in profile_results
    if x["row_count"] == 0
]

tables_without_unique_key = [
    x for x in profile_results
    if x["row_count"] not in (None, 0)
    and not x["likely_primary_key"]
]

# -----------------------------------------------------------------------------
# 4. PRINT SUMMARY
# -----------------------------------------------------------------------------

print(f"Silver tables expected       : {len(PROFILE_TABLES)}")
print(f"Silver tables profiled       : {len(PROFILE_TABLES) - len(profiling_failures)}")
print(f"Profiling failures           : {len(profiling_failures)}")
print(f"Empty Silver tables          : {len(empty_tables)}")
print(f"Tables without unique *_key  : {len(tables_without_unique_key)}")
print(f"Schema columns discovered    : {len(schema_results)}")

print("=" * 100)

# -----------------------------------------------------------------------------
# 5. PRINT TABLE-BY-TABLE PROFILE
# -----------------------------------------------------------------------------

for x in profile_results:

    print()
    print("-" * 100)
    print(f"TABLE                : {x['table_name']}")
    print(f"TYPE                 : {x['table_type']}")
    print(f"ROWS                 : {x['row_count']}")
    print(f"COLUMNS              : {x['column_count']}")
    print(f"KEY COLUMNS          : {x['key_columns']}")
    print(f"UNIQUE KEY CANDIDATE : {x['likely_primary_key']}")
    print(f"NULL KEY VALUES      : {x['null_key_values']}")
    print(f"DATE COLUMN          : {x['primary_date_column']}")
    print(f"MIN DATE             : {x['min_date']}")
    print(f"MAX DATE             : {x['max_date']}")
    print(f"STATUS               : {x['status']}")

# -----------------------------------------------------------------------------
# 6. FAIL FAST ONLY FOR ACTUAL SOURCE PROBLEMS
# -----------------------------------------------------------------------------

if profiling_failures:

    raise RuntimeError(
        f"GOLD SOURCE PROFILING FAILED — "
        f"{len(profiling_failures)} Silver table(s) could not be profiled."
    )

if empty_tables:

    raise RuntimeError(
        f"GOLD SOURCE PROFILING FAILED — "
        f"{len(empty_tables)} required Silver table(s) are empty."
    )

print()
print("=" * 100)
print("GOLD BUSINESS MODEL — SILVER SOURCE PROFILING: SUCCEEDED")
print("=" * 100)

# -----------------------------------------------------------------------------
# 7. IMPORTANT: PRINT COMPLETE SILVER SCHEMAS
#    We need this output before writing Gold transformation logic.
# -----------------------------------------------------------------------------

for table_name in PROFILE_TABLES:

    print()
    print("=" * 100)
    print(f"SCHEMA — {table_name.upper()}")
    print("=" * 100)

    spark.table(
        f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.{table_name}"
    ).printSchema()

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 4, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — SILVER SOURCE PROFILING
Silver tables expected       : 9
Silver tables profiled       : 9
Profiling failures           : 0
Empty Silver tables          : 0
Tables without unique *_key  : 0
Schema columns discovered    : 302

----------------------------------------------------------------------------------------------------
TABLE                : dim_date
TYPE                 : DIMENSION
ROWS                 : 5844
COLUMNS              : 59
KEY COLUMNS          : date_key
UNIQUE KEY CANDIDATE : date_key
NULL KEY VALUES      : 0
DATE COLUMN          : calendar_date
MIN DATE             : 2020-01-01
MAX DATE             : 2035-12-31
STATUS               : PROFILED

----------------------------------------------------------------------------------------------------
TABLE                : dim_company
TYPE                 : DIMENSION
ROWS                 : 3
COLUMNS              : 23
KEY COLUMNS          : company_key, company_business_key
UNIQUE KEY CANDIDATE : compan

In [3]:
# =============================================================================
# CELL 3 — GOLD BUSINESS MODEL
# Grain, Relationship & Referential Integrity Validation
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. LOAD SILVER DIMENSIONS
# =============================================================================

DIM_DATE_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.dim_date"
)

DIM_COMPANY_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.dim_company"
)

DIM_CUSTOMER_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.dim_customer"
)

DIM_PRODUCT_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.dim_product"
)


# =============================================================================
# 2. LOAD SILVER FACTS
# =============================================================================

FACT_SALES_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.fact_sales_order_line"
)

FACT_INVOICE_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.fact_invoice_line"
)

FACT_PAYMENT_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.fact_payment"
)

FACT_INVENTORY_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.fact_inventory_transaction"
)

FACT_BUDGET_DF = spark.table(
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}.fact_monthly_budget"
)


# =============================================================================
# 3. DIMENSION KEY SETS
#
# IMPORTANT:
# We validate facts against ALL valid dimension surrogate keys,
# not only is_current = true.
#
# This preserves correct historical SCD2 relationships.
# =============================================================================

DATE_KEYS_DF = (
    DIM_DATE_DF
    .select(
        F.col("date_key").cast("int").alias("date_key")
    )
    .distinct()
)

COMPANY_KEYS_DF = (
    DIM_COMPANY_DF
    .select("company_key")
    .distinct()
)

CUSTOMER_KEYS_DF = (
    DIM_CUSTOMER_DF
    .select("customer_key")
    .distinct()
)

PRODUCT_KEYS_DF = (
    DIM_PRODUCT_DF
    .select("product_key")
    .distinct()
)


# =============================================================================
# 4. HELPER — DUPLICATE KEY COUNT
# =============================================================================

def duplicate_key_count(df, key_column):

    return (
        df
        .groupBy(key_column)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )


# =============================================================================
# 5. HELPER — NULL COUNT
# =============================================================================

def null_count(df, column_name):

    return (
        df
        .filter(
            F.col(column_name).isNull()
        )
        .count()
    )


# =============================================================================
# 6. HELPER — ORPHAN COUNT
# =============================================================================

def orphan_count(
    fact_df,
    fact_key,
    dimension_key_df,
    dimension_key
):

    return (
        fact_df
        .filter(
            F.col(fact_key).isNotNull()
        )
        .select(
            F.col(fact_key).alias("_fact_key")
        )
        .distinct()
        .join(
            dimension_key_df.select(
                F.col(dimension_key).alias("_dimension_key")
            ),
            F.col("_fact_key") == F.col("_dimension_key"),
            "left_anti"
        )
        .count()
    )


# =============================================================================
# 7. DIMENSION PRIMARY-KEY VALIDATION
# =============================================================================

dim_date_duplicate_keys = duplicate_key_count(
    DIM_DATE_DF,
    "date_key"
)

dim_company_duplicate_keys = duplicate_key_count(
    DIM_COMPANY_DF,
    "company_key"
)

dim_customer_duplicate_keys = duplicate_key_count(
    DIM_CUSTOMER_DF,
    "customer_key"
)

dim_product_duplicate_keys = duplicate_key_count(
    DIM_PRODUCT_DF,
    "product_key"
)


# =============================================================================
# 8. FACT BUSINESS / SURROGATE KEY UNIQUENESS
# =============================================================================

sales_duplicate_fact_keys = duplicate_key_count(
    FACT_SALES_DF,
    "sales_order_line_key"
)

sales_duplicate_business_keys = duplicate_key_count(
    FACT_SALES_DF,
    "sales_order_line_business_key"
)

invoice_duplicate_fact_keys = duplicate_key_count(
    FACT_INVOICE_DF,
    "invoice_line_key"
)

invoice_duplicate_business_keys = duplicate_key_count(
    FACT_INVOICE_DF,
    "invoice_line_business_key"
)

payment_duplicate_fact_keys = duplicate_key_count(
    FACT_PAYMENT_DF,
    "payment_key"
)

payment_duplicate_business_keys = duplicate_key_count(
    FACT_PAYMENT_DF,
    "payment_business_key"
)

inventory_duplicate_fact_keys = duplicate_key_count(
    FACT_INVENTORY_DF,
    "inventory_transaction_key"
)

inventory_duplicate_business_keys = duplicate_key_count(
    FACT_INVENTORY_DF,
    "inventory_transaction_business_key"
)

budget_duplicate_fact_keys = duplicate_key_count(
    FACT_BUDGET_DF,
    "monthly_budget_key"
)

budget_duplicate_business_keys = duplicate_key_count(
    FACT_BUDGET_DF,
    "monthly_budget_business_key"
)


# =============================================================================
# 9. SALES FOREIGN-KEY NULL ANALYSIS
#
# requested_delivery_date_key is treated as OPTIONAL.
# All other Gold relationship keys are REQUIRED.
# =============================================================================

sales_null_company_keys = null_count(
    FACT_SALES_DF,
    "company_key"
)

sales_null_customer_keys = null_count(
    FACT_SALES_DF,
    "customer_key"
)

sales_null_product_keys = null_count(
    FACT_SALES_DF,
    "product_key"
)

sales_null_order_date_keys = null_count(
    FACT_SALES_DF,
    "order_date_key"
)

sales_null_requested_delivery_keys = null_count(
    FACT_SALES_DF,
    "requested_delivery_date_key"
)


# =============================================================================
# 10. SALES ORPHAN VALIDATION
# =============================================================================

sales_orphan_company_keys = orphan_count(
    FACT_SALES_DF,
    "company_key",
    COMPANY_KEYS_DF,
    "company_key"
)

sales_orphan_customer_keys = orphan_count(
    FACT_SALES_DF,
    "customer_key",
    CUSTOMER_KEYS_DF,
    "customer_key"
)

sales_orphan_product_keys = orphan_count(
    FACT_SALES_DF,
    "product_key",
    PRODUCT_KEYS_DF,
    "product_key"
)

sales_orphan_order_date_keys = orphan_count(
    FACT_SALES_DF,
    "order_date_key",
    DATE_KEYS_DF,
    "date_key"
)

sales_orphan_requested_delivery_keys = orphan_count(
    FACT_SALES_DF,
    "requested_delivery_date_key",
    DATE_KEYS_DF,
    "date_key"
)


# =============================================================================
# 11. INVOICE FOREIGN-KEY VALIDATION
# =============================================================================

invoice_null_company_keys = null_count(
    FACT_INVOICE_DF,
    "company_key"
)

invoice_null_customer_keys = null_count(
    FACT_INVOICE_DF,
    "customer_key"
)

invoice_null_product_keys = null_count(
    FACT_INVOICE_DF,
    "product_key"
)

invoice_null_invoice_date_keys = null_count(
    FACT_INVOICE_DF,
    "invoice_date_key"
)

invoice_null_due_date_keys = null_count(
    FACT_INVOICE_DF,
    "due_date_key"
)


invoice_orphan_company_keys = orphan_count(
    FACT_INVOICE_DF,
    "company_key",
    COMPANY_KEYS_DF,
    "company_key"
)

invoice_orphan_customer_keys = orphan_count(
    FACT_INVOICE_DF,
    "customer_key",
    CUSTOMER_KEYS_DF,
    "customer_key"
)

invoice_orphan_product_keys = orphan_count(
    FACT_INVOICE_DF,
    "product_key",
    PRODUCT_KEYS_DF,
    "product_key"
)

invoice_orphan_invoice_date_keys = orphan_count(
    FACT_INVOICE_DF,
    "invoice_date_key",
    DATE_KEYS_DF,
    "date_key"
)

invoice_orphan_due_date_keys = orphan_count(
    FACT_INVOICE_DF,
    "due_date_key",
    DATE_KEYS_DF,
    "date_key"
)


# =============================================================================
# 12. PAYMENT FOREIGN-KEY VALIDATION
# =============================================================================

payment_null_company_keys = null_count(
    FACT_PAYMENT_DF,
    "company_key"
)

payment_null_customer_keys = null_count(
    FACT_PAYMENT_DF,
    "customer_key"
)

payment_null_date_keys = null_count(
    FACT_PAYMENT_DF,
    "payment_date_key"
)


payment_orphan_company_keys = orphan_count(
    FACT_PAYMENT_DF,
    "company_key",
    COMPANY_KEYS_DF,
    "company_key"
)

payment_orphan_customer_keys = orphan_count(
    FACT_PAYMENT_DF,
    "customer_key",
    CUSTOMER_KEYS_DF,
    "customer_key"
)

payment_orphan_date_keys = orphan_count(
    FACT_PAYMENT_DF,
    "payment_date_key",
    DATE_KEYS_DF,
    "date_key"
)


# =============================================================================
# 13. INVENTORY FOREIGN-KEY VALIDATION
# =============================================================================

inventory_null_company_keys = null_count(
    FACT_INVENTORY_DF,
    "company_key"
)

inventory_null_product_keys = null_count(
    FACT_INVENTORY_DF,
    "product_key"
)

inventory_null_date_keys = null_count(
    FACT_INVENTORY_DF,
    "transaction_date_key"
)


inventory_orphan_company_keys = orphan_count(
    FACT_INVENTORY_DF,
    "company_key",
    COMPANY_KEYS_DF,
    "company_key"
)

inventory_orphan_product_keys = orphan_count(
    FACT_INVENTORY_DF,
    "product_key",
    PRODUCT_KEYS_DF,
    "product_key"
)

inventory_orphan_date_keys = orphan_count(
    FACT_INVENTORY_DF,
    "transaction_date_key",
    DATE_KEYS_DF,
    "date_key"
)


# =============================================================================
# 14. BUDGET FOREIGN-KEY VALIDATION
# =============================================================================

budget_null_company_keys = null_count(
    FACT_BUDGET_DF,
    "company_key"
)

budget_null_date_keys = null_count(
    FACT_BUDGET_DF,
    "budget_month_date_key"
)


budget_orphan_company_keys = orphan_count(
    FACT_BUDGET_DF,
    "company_key",
    COMPANY_KEYS_DF,
    "company_key"
)

budget_orphan_date_keys = orphan_count(
    FACT_BUDGET_DF,
    "budget_month_date_key",
    DATE_KEYS_DF,
    "date_key"
)


# =============================================================================
# 15. DATE KEY / DATE VALUE ALIGNMENT
# =============================================================================

sales_date_alignment_failures = (
    FACT_SALES_DF
    .filter(
        F.col("order_date_key")
        !=
        F.date_format(
            F.col("order_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .count()
)

invoice_date_alignment_failures = (
    FACT_INVOICE_DF
    .filter(
        F.col("invoice_date_key")
        !=
        F.date_format(
            F.col("invoice_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .count()
)

payment_date_alignment_failures = (
    FACT_PAYMENT_DF
    .filter(
        F.col("payment_date_key")
        !=
        F.date_format(
            F.col("payment_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .count()
)

inventory_date_alignment_failures = (
    FACT_INVENTORY_DF
    .filter(
        F.col("transaction_date_key")
        !=
        F.date_format(
            F.col("transaction_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .count()
)

budget_date_alignment_failures = (
    FACT_BUDGET_DF
    .filter(
        F.col("budget_month_date_key")
        !=
        F.date_format(
            F.col("budget_month"),
            "yyyyMMdd"
        ).cast("int")
    )
    .count()
)


# =============================================================================
# 16. BUDGET CATEGORY VS PRODUCT CATEGORY COMPATIBILITY
#
# This is INFORMATIONAL for now.
#
# It determines whether Budget vs Actual can use a conformed
# dim_budget_category / product-category mapping.
# =============================================================================

BUDGET_CATEGORIES_DF = (
    FACT_BUDGET_DF
    .select(
        F.upper(
            F.trim(
                F.col("budget_category")
            )
        ).alias("category")
    )
    .filter(
        F.col("category").isNotNull()
    )
    .distinct()
)


PRODUCT_CATEGORIES_DF = (
    DIM_PRODUCT_DF
    .select(
        F.upper(
            F.trim(
                F.col("product_category")
            )
        ).alias("category")
    )
    .filter(
        F.col("category").isNotNull()
    )
    .distinct()
)


budget_category_count = (
    BUDGET_CATEGORIES_DF.count()
)

product_category_count = (
    PRODUCT_CATEGORIES_DF.count()
)


BUDGET_CATEGORIES_NOT_IN_PRODUCT_DF = (
    BUDGET_CATEGORIES_DF
    .join(
        PRODUCT_CATEGORIES_DF,
        on="category",
        how="left_anti"
    )
)

budget_categories_not_in_product_count = (
    BUDGET_CATEGORIES_NOT_IN_PRODUCT_DF.count()
)


PRODUCT_CATEGORIES_NOT_IN_BUDGET_DF = (
    PRODUCT_CATEGORIES_DF
    .join(
        BUDGET_CATEGORIES_DF,
        on="category",
        how="left_anti"
    )
)

product_categories_not_in_budget_count = (
    PRODUCT_CATEGORIES_NOT_IN_BUDGET_DF.count()
)


# =============================================================================
# 17. BUILD CRITICAL FAILURE COUNT
# =============================================================================

critical_checks = {

    # Dimensions
    "dim_date_duplicate_key": dim_date_duplicate_keys,
    "dim_company_duplicate_key": dim_company_duplicate_keys,
    "dim_customer_duplicate_key": dim_customer_duplicate_keys,
    "dim_product_duplicate_key": dim_product_duplicate_keys,

    # Fact key uniqueness
    "sales_duplicate_fact_key": sales_duplicate_fact_keys,
    "sales_duplicate_business_key": sales_duplicate_business_keys,

    "invoice_duplicate_fact_key": invoice_duplicate_fact_keys,
    "invoice_duplicate_business_key": invoice_duplicate_business_keys,

    "payment_duplicate_fact_key": payment_duplicate_fact_keys,
    "payment_duplicate_business_key": payment_duplicate_business_keys,

    "inventory_duplicate_fact_key": inventory_duplicate_fact_keys,
    "inventory_duplicate_business_key": inventory_duplicate_business_keys,

    "budget_duplicate_fact_key": budget_duplicate_fact_keys,
    "budget_duplicate_business_key": budget_duplicate_business_keys,

    # Required Sales FKs
    "sales_null_company_key": sales_null_company_keys,
    "sales_null_customer_key": sales_null_customer_keys,
    "sales_null_product_key": sales_null_product_keys,
    "sales_null_order_date_key": sales_null_order_date_keys,

    "sales_orphan_company_key": sales_orphan_company_keys,
    "sales_orphan_customer_key": sales_orphan_customer_keys,
    "sales_orphan_product_key": sales_orphan_product_keys,
    "sales_orphan_order_date_key": sales_orphan_order_date_keys,

    # Invoice
    "invoice_null_company_key": invoice_null_company_keys,
    "invoice_null_customer_key": invoice_null_customer_keys,
    "invoice_null_product_key": invoice_null_product_keys,
    "invoice_null_invoice_date_key": invoice_null_invoice_date_keys,
    "invoice_null_due_date_key": invoice_null_due_date_keys,

    "invoice_orphan_company_key": invoice_orphan_company_keys,
    "invoice_orphan_customer_key": invoice_orphan_customer_keys,
    "invoice_orphan_product_key": invoice_orphan_product_keys,
    "invoice_orphan_invoice_date_key": invoice_orphan_invoice_date_keys,
    "invoice_orphan_due_date_key": invoice_orphan_due_date_keys,

    # Payment
    "payment_null_company_key": payment_null_company_keys,
    "payment_null_customer_key": payment_null_customer_keys,
    "payment_null_date_key": payment_null_date_keys,

    "payment_orphan_company_key": payment_orphan_company_keys,
    "payment_orphan_customer_key": payment_orphan_customer_keys,
    "payment_orphan_date_key": payment_orphan_date_keys,

    # Inventory
    "inventory_null_company_key": inventory_null_company_keys,
    "inventory_null_product_key": inventory_null_product_keys,
    "inventory_null_date_key": inventory_null_date_keys,

    "inventory_orphan_company_key": inventory_orphan_company_keys,
    "inventory_orphan_product_key": inventory_orphan_product_keys,
    "inventory_orphan_date_key": inventory_orphan_date_keys,

    # Budget
    "budget_null_company_key": budget_null_company_keys,
    "budget_null_date_key": budget_null_date_keys,

    "budget_orphan_company_key": budget_orphan_company_keys,
    "budget_orphan_date_key": budget_orphan_date_keys,

    # Date alignment
    "sales_date_alignment": sales_date_alignment_failures,
    "invoice_date_alignment": invoice_date_alignment_failures,
    "payment_date_alignment": payment_date_alignment_failures,
    "inventory_date_alignment": inventory_date_alignment_failures,
    "budget_date_alignment": budget_date_alignment_failures,
}


failed_critical_checks = {
    k: v
    for k, v in critical_checks.items()
    if v > 0
}


# =============================================================================
# 18. PRINT SALES NULL-KEY DIAGNOSTIC
# =============================================================================

print("=" * 100)
print("SALES ORDER LINE KEY DIAGNOSTIC")
print("=" * 100)

print(
    f"Null company keys                    : "
    f"{sales_null_company_keys}"
)

print(
    f"Null customer keys                   : "
    f"{sales_null_customer_keys}"
)

print(
    f"Null product keys                    : "
    f"{sales_null_product_keys}"
)

print(
    f"Null order date keys                 : "
    f"{sales_null_order_date_keys}"
)

print(
    f"Null requested delivery date keys    : "
    f"{sales_null_requested_delivery_keys}"
)

print(
    f"Orphan requested delivery date keys  : "
    f"{sales_orphan_requested_delivery_keys}"
)

print("=" * 100)


# =============================================================================
# 19. PRINT RELATIONSHIP VALIDATION SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD BUSINESS MODEL — RELATIONSHIP & GRAIN VALIDATION")
print("=" * 100)

print(f"Dimension duplicate key failures : "
      f"{dim_date_duplicate_keys + dim_company_duplicate_keys + dim_customer_duplicate_keys + dim_product_duplicate_keys}")

print(f"Sales orphan required keys       : "
      f"{sales_orphan_company_keys + sales_orphan_customer_keys + sales_orphan_product_keys + sales_orphan_order_date_keys}")

print(f"Invoice orphan keys              : "
      f"{invoice_orphan_company_keys + invoice_orphan_customer_keys + invoice_orphan_product_keys + invoice_orphan_invoice_date_keys + invoice_orphan_due_date_keys}")

print(f"Payment orphan keys              : "
      f"{payment_orphan_company_keys + payment_orphan_customer_keys + payment_orphan_date_keys}")

print(f"Inventory orphan keys            : "
      f"{inventory_orphan_company_keys + inventory_orphan_product_keys + inventory_orphan_date_keys}")

print(f"Budget orphan keys               : "
      f"{budget_orphan_company_keys + budget_orphan_date_keys}")

print(f"Date alignment failures          : "
      f"{sales_date_alignment_failures + invoice_date_alignment_failures + payment_date_alignment_failures + inventory_date_alignment_failures + budget_date_alignment_failures}")

print(f"Critical checks failed           : "
      f"{len(failed_critical_checks)}")

print("=" * 100)


# =============================================================================
# 20. PRINT CATEGORY COMPATIBILITY
# =============================================================================

print("=" * 100)
print("BUDGET / PRODUCT CATEGORY COMPATIBILITY")
print("=" * 100)

print(
    f"Budget categories                 : "
    f"{budget_category_count}"
)

print(
    f"Product categories                : "
    f"{product_category_count}"
)

print(
    f"Budget categories not in products : "
    f"{budget_categories_not_in_product_count}"
)

print(
    f"Product categories not in budgets : "
    f"{product_categories_not_in_budget_count}"
)

print("=" * 100)


# =============================================================================
# 21. DISPLAY CATEGORY MISMATCHES IF PRESENT
# =============================================================================

if budget_categories_not_in_product_count > 0:

    print(
        "Budget categories without matching product category:"
    )

    display(
        BUDGET_CATEGORIES_NOT_IN_PRODUCT_DF
        .orderBy("category")
    )


if product_categories_not_in_budget_count > 0:

    print(
        "Product categories without matching budget category:"
    )

    display(
        PRODUCT_CATEGORIES_NOT_IN_BUDGET_DF
        .orderBy("category")
    )


# =============================================================================
# 22. FAIL FAST ON TRUE MODEL INTEGRITY PROBLEMS
# =============================================================================

if failed_critical_checks:

    print("=" * 100)
    print("CRITICAL GOLD MODEL FAILURES")
    print("=" * 100)

    for rule, count in failed_critical_checks.items():

        print(
            f"{rule:<45} : {count}"
        )

    print("=" * 100)

    raise RuntimeError(
        f"GOLD MODEL RELATIONSHIP VALIDATION FAILED — "
        f"{len(failed_critical_checks)} critical check(s) failed."
    )


# =============================================================================
# 23. FINAL STATUS
# =============================================================================

print(
    "GOLD BUSINESS MODEL — RELATIONSHIP & GRAIN VALIDATION: SUCCEEDED"
)

print("=" * 100)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 5, Finished, Available, Finished, False)

SALES ORDER LINE KEY DIAGNOSTIC
Null company keys                    : 0
Null customer keys                   : 0
Null product keys                    : 0
Null order date keys                 : 0
Null requested delivery date keys    : 0
Orphan requested delivery date keys  : 0
GOLD BUSINESS MODEL — RELATIONSHIP & GRAIN VALIDATION
Dimension duplicate key failures : 0
Sales orphan required keys       : 0
Invoice orphan keys              : 0
Payment orphan keys              : 0
Inventory orphan keys            : 0
Budget orphan keys               : 0
Date alignment failures          : 0
Critical checks failed           : 0
BUDGET / PRODUCT CATEGORY COMPATIBILITY
Budget categories                 : 9
Product categories                : 22
Budget categories not in products : 5
Product categories not in budgets : 18
Budget categories without matching product category:


SynapseWidget(Synapse.DataFrame, cf1a7067-765a-4d0d-8926-726e7cc31532)

Product categories without matching budget category:


SynapseWidget(Synapse.DataFrame, d88f4e32-1d03-4d16-a080-2df6ff150f12)

GOLD BUSINESS MODEL — RELATIONSHIP & GRAIN VALIDATION: SUCCEEDED


In [4]:
# =============================================================================
# CELL 4 — GOLD BUSINESS MODEL
# Category Conformance Analysis
#
# Purpose:
# - Profile Budget categories.
# - Profile Product categories.
# - Identify exact matches.
# - Identify unmatched categories.
# - Quantify business significance of each category.
# - Prepare controlled mapping for Gold.
#
# IMPORTANT:
# This cell does NOT write any Gold tables.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. BUDGET CATEGORY PROFILE
# =============================================================================

BUDGET_CATEGORY_PROFILE_DF = (
    FACT_BUDGET_DF

    .withColumn(
        "category_normalised",
        F.upper(
            F.trim(
                F.col("budget_category")
            )
        )
    )

    .groupBy(
        "category_normalised"
    )

    .agg(
        F.count("*").alias(
            "budget_rows"
        ),

        F.countDistinct(
            "company_key"
        ).alias(
            "budget_companies"
        ),

        F.countDistinct(
            "budget_month"
        ).alias(
            "budget_months"
        ),

        F.sum(
            "revenue_budget"
        ).alias(
            "revenue_budget"
        ),

        F.sum(
            "cost_budget"
        ).alias(
            "cost_budget"
        ),

        F.sum(
            "gross_margin_budget"
        ).alias(
            "gross_margin_budget"
        )
    )
)


# =============================================================================
# 2. PRODUCT CATEGORY PROFILE
# =============================================================================

PRODUCT_CATEGORY_PROFILE_DF = (
    DIM_PRODUCT_DF

    .filter(
        F.col(
            "product_category"
        ).isNotNull()
    )

    .withColumn(
        "category_normalised",
        F.upper(
            F.trim(
                F.col("product_category")
            )
        )
    )

    .groupBy(
        "category_normalised"
    )

    .agg(
        F.count("*").alias(
            "product_rows"
        ),

        F.countDistinct(
            "product_key"
        ).alias(
            "products"
        ),

        F.countDistinct(
            "company_key"
        ).alias(
            "product_companies"
        )
    )
)


# =============================================================================
# 3. PRODUCT CATEGORY ACTUAL SALES PROFILE
#
# Use invoiced revenue as the Actual side.
# =============================================================================

PRODUCT_ACTUAL_PROFILE_DF = (
    FACT_INVOICE_DF.alias("f")

    .join(
        DIM_PRODUCT_DF.alias("p"),

        F.col("f.product_key")
        ==
        F.col("p.product_key"),

        how="left"
    )

    .withColumn(
        "category_normalised",
        F.upper(
            F.trim(
                F.col(
                    "p.product_category"
                )
            )
        )
    )

    .groupBy(
        "category_normalised"
    )

    .agg(
        F.count("*").alias(
            "invoice_lines"
        ),

        F.sum(
            F.col(
                "f.net_amount"
            )
        ).alias(
            "actual_revenue"
        ),

        F.sum(
            F.col(
                "f.cost_amount"
            )
        ).alias(
            "actual_cost"
        ),

        F.sum(
            F.col(
                "f.margin_amount"
            )
        ).alias(
            "actual_margin"
        )
    )
)


# =============================================================================
# 4. CREATE MASTER CATEGORY COMPARISON
# =============================================================================

CATEGORY_COMPARISON_DF = (
    BUDGET_CATEGORY_PROFILE_DF.alias("b")

    .join(
        PRODUCT_CATEGORY_PROFILE_DF.alias("p"),

        F.col(
            "b.category_normalised"
        )
        ==
        F.col(
            "p.category_normalised"
        ),

        how="full_outer"
    )

    .select(
        F.coalesce(
            F.col(
                "b.category_normalised"
            ),
            F.col(
                "p.category_normalised"
            )
        ).alias(
            "category"
        ),

        F.col(
            "b.category_normalised"
        ).alias(
            "budget_category"
        ),

        F.col(
            "p.category_normalised"
        ).alias(
            "product_category"
        ),

        "budget_rows",
        "budget_companies",
        "budget_months",
        "revenue_budget",
        "cost_budget",
        "gross_margin_budget",

        "product_rows",
        "products",
        "product_companies"
    )

    .join(
        PRODUCT_ACTUAL_PROFILE_DF,

        on=(
            F.col("category")
            ==
            F.col(
                "category_normalised"
            )
        ),

        how="left"
    )

    .drop(
        "category_normalised"
    )

    .withColumn(
        "match_status",

        F.when(
            F.col(
                "budget_category"
            ).isNotNull()
            &
            F.col(
                "product_category"
            ).isNotNull(),

            F.lit(
                "EXACT_MATCH"
            )
        )

        .when(
            F.col(
                "budget_category"
            ).isNotNull(),

            F.lit(
                "BUDGET_ONLY"
            )
        )

        .otherwise(
            F.lit(
                "PRODUCT_ONLY"
            )
        )
    )
)


# =============================================================================
# 5. DISPLAY MASTER CATEGORY PROFILE
# =============================================================================

print("=" * 100)
print("GOLD CATEGORY CONFORMANCE — ALL CATEGORIES")
print("=" * 100)

display(
    CATEGORY_COMPARISON_DF

    .orderBy(
        "match_status",
        "category"
    )
)


# =============================================================================
# 6. DISPLAY EXACT MATCHES
# =============================================================================

EXACT_CATEGORY_MATCHES_DF = (
    CATEGORY_COMPARISON_DF

    .filter(
        F.col(
            "match_status"
        )
        ==
        "EXACT_MATCH"
    )
)


print("=" * 100)
print("EXACT CATEGORY MATCHES")
print("=" * 100)

display(
    EXACT_CATEGORY_MATCHES_DF
    .orderBy(
        "category"
    )
)


# =============================================================================
# 7. DISPLAY BUDGET-ONLY CATEGORIES
# =============================================================================

BUDGET_ONLY_CATEGORIES_DF = (
    CATEGORY_COMPARISON_DF

    .filter(
        F.col(
            "match_status"
        )
        ==
        "BUDGET_ONLY"
    )
)


print("=" * 100)
print("BUDGET-ONLY CATEGORIES")
print("=" * 100)

display(
    BUDGET_ONLY_CATEGORIES_DF
    .orderBy(
        F.col(
            "revenue_budget"
        ).desc_nulls_last()
    )
)


# =============================================================================
# 8. DISPLAY PRODUCT-ONLY CATEGORIES
# =============================================================================

PRODUCT_ONLY_CATEGORIES_DF = (
    CATEGORY_COMPARISON_DF

    .filter(
        F.col(
            "match_status"
        )
        ==
        "PRODUCT_ONLY"
    )
)


print("=" * 100)
print("PRODUCT-ONLY CATEGORIES")
print("=" * 100)

display(
    PRODUCT_ONLY_CATEGORIES_DF

    .orderBy(
        F.col(
            "actual_revenue"
        ).desc_nulls_last()
    )
)


# =============================================================================
# 9. COUNTS
# =============================================================================

exact_match_count = (
    EXACT_CATEGORY_MATCHES_DF.count()
)

budget_only_count = (
    BUDGET_ONLY_CATEGORIES_DF.count()
)

product_only_count = (
    PRODUCT_ONLY_CATEGORIES_DF.count()
)


# =============================================================================
# 10. BUSINESS IMPACT OF UNMATCHED PRODUCT CATEGORIES
# =============================================================================

unmatched_product_actual_revenue = (
    PRODUCT_ONLY_CATEGORIES_DF

    .agg(
        F.coalesce(
            F.sum(
                "actual_revenue"
            ),
            F.lit(0)
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


total_actual_revenue = (
    PRODUCT_ACTUAL_PROFILE_DF

    .agg(
        F.coalesce(
            F.sum(
                "actual_revenue"
            ),
            F.lit(0)
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


# =============================================================================
# 11. PRINT SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD CATEGORY CONFORMANCE SUMMARY")
print("=" * 100)

print(
    f"Exact category matches             : "
    f"{exact_match_count}"
)

print(
    f"Budget-only categories             : "
    f"{budget_only_count}"
)

print(
    f"Product-only categories            : "
    f"{product_only_count}"
)

print(
    f"Total actual invoiced revenue      : "
    f"{total_actual_revenue}"
)

print(
    f"Revenue in unmatched categories    : "
    f"{unmatched_product_actual_revenue}"
)

print("=" * 100)


# =============================================================================
# 12. FINAL STATUS
# =============================================================================

print(
    "GOLD BUSINESS MODEL — CATEGORY CONFORMANCE ANALYSIS: SUCCEEDED"
)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 6, Finished, Available, Finished, False)

GOLD CATEGORY CONFORMANCE — ALL CATEGORIES


SynapseWidget(Synapse.DataFrame, 8c60d44c-0ba5-425d-aae4-a86ec9c296a8)

EXACT CATEGORY MATCHES


SynapseWidget(Synapse.DataFrame, 46eb9e33-9919-45ee-b07b-a0ce0e39656c)

BUDGET-ONLY CATEGORIES


SynapseWidget(Synapse.DataFrame, 9523ddbb-fd13-4223-99b0-dff3e313f043)

PRODUCT-ONLY CATEGORIES


SynapseWidget(Synapse.DataFrame, 883c09a8-b810-40c3-a06e-346562a4a87d)

GOLD CATEGORY CONFORMANCE SUMMARY
Exact category matches             : 4
Budget-only categories             : 5
Product-only categories            : 18
Total actual invoiced revenue      : 70509104.7300
Revenue in unmatched categories    : 69872854.7300
GOLD BUSINESS MODEL — CATEGORY CONFORMANCE ANALYSIS: SUCCEEDED


In [5]:
# =============================================================================
# CELL 5 — GOLD BUSINESS MODEL
# Gold Dimension Preparation
#
# Purpose:
# - Prepare business-facing Gold dimensions.
# - Retain conformed surrogate keys from Silver.
# - Remove Silver engineering metadata not needed by Power BI.
# - Create a dedicated dim_budget_category.
#
# No Gold tables are written yet.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. GOLD DIM_DATE
# =============================================================================

GOLD_DIM_DATE_DF = (
    DIM_DATE_DF

    .select(
        "date_key",
        "calendar_date",
        "full_date_description",

        "day_of_month",
        "day_of_year",
        "iso_day_of_week",
        "day_name",
        "day_short_name",

        "iso_week_number",
        "iso_week_year",
        "iso_year_week",
        "week_start_date",
        "week_end_date",

        "month_number",
        "month_name",
        "month_short_name",
        "year_month_number",
        "year_month_name",
        "month_start_date",
        "month_end_date",

        "calendar_quarter",
        "calendar_quarter_name",
        "calendar_year_quarter",

        "calendar_year",

        "fiscal_period_number",
        "fiscal_period_name",
        "fiscal_quarter_number",
        "fiscal_quarter_name",
        "fiscal_year_quarter",
        "fiscal_year_name",

        "is_weekday",
        "is_weekend",
        "is_business_day",
        "is_bank_holiday",
        "bank_holiday_name",

        "is_month_start",
        "is_month_end",
        "is_quarter_start",
        "is_quarter_end",
        "is_year_start",
        "is_year_end"
    )
)


# =============================================================================
# 2. GOLD DIM_COMPANY
# =============================================================================

GOLD_DIM_COMPANY_DF = (
    DIM_COMPANY_DF

    .filter(
        F.col("is_current") == True
    )

    .select(
        "company_key",
        "company_business_key",
        "company_code",
        "company_name",
        "country_code",
        "country_name",
        "city",
        "postal_code",
        "base_currency_code",
        "local_currency_code",
        "is_active"
    )
)


# =============================================================================
# 3. GOLD DIM_CUSTOMER
#
# Keep SCD surrogate customer_key because facts already reference it.
# Retain reporting-useful attributes only.
# =============================================================================

GOLD_DIM_CUSTOMER_DF = (
    DIM_CUSTOMER_DF

    .select(
        "customer_key",
        "customer_business_key",
        "customer_id",
        "customer_name",
        "customer_type",

        "company_key",
        "company_code",

        "country_code",
        "city",
        "postal_code",

        "currency_code",
        "payment_terms",
        "credit_limit",
        "current_balance",

        "is_active",
        "parent_customer_id",
        "is_job",

        "effective_from_utc",
        "effective_to_utc",
        "is_current"
    )
)


# =============================================================================
# 4. GOLD DIM_PRODUCT
# =============================================================================

GOLD_DIM_PRODUCT_DF = (
    DIM_PRODUCT_DF

    .select(
        "product_key",
        "product_business_key",
        "product_id",
        "product_name",
        "product_type",
        "product_category",
        "sku",
        "product_description",
        "unit_of_measure",

        "company_key",
        "company_code",

        "currency_code",
        "unit_price",
        "purchase_cost",

        "is_inventory_item",
        "is_sales_item",
        "is_purchase_item",
        "is_active",

        "effective_from_utc",
        "effective_to_utc",
        "is_current"
    )
)


# =============================================================================
# 5. CREATE DIM_BUDGET_CATEGORY
#
# Budget categories are intentionally independent of product categories.
# =============================================================================

GOLD_DIM_BUDGET_CATEGORY_DF = (
    FACT_BUDGET_DF

    .select(
        F.trim(
            F.col("budget_category")
        ).alias(
            "budget_category"
        )
    )

    .filter(
        F.col("budget_category").isNotNull()
    )

    .distinct()

    .withColumn(
        "budget_category_key",

        F.sha2(
            F.upper(
                F.trim(
                    F.col("budget_category")
                )
            ),
            256
        )
    )

    .withColumn(
        "budget_category_name",
        F.col("budget_category")
    )

    .drop(
        "budget_category"
    )

    .select(
        "budget_category_key",
        "budget_category_name"
    )
)


# =============================================================================
# 6. VALIDATE DIMENSION UNIQUENESS
# =============================================================================

dimension_validation = {}


for name, df, key in [

    (
        "dim_date",
        GOLD_DIM_DATE_DF,
        "date_key"
    ),

    (
        "dim_company",
        GOLD_DIM_COMPANY_DF,
        "company_key"
    ),

    (
        "dim_customer",
        GOLD_DIM_CUSTOMER_DF,
        "customer_key"
    ),

    (
        "dim_product",
        GOLD_DIM_PRODUCT_DF,
        "product_key"
    ),

    (
        "dim_budget_category",
        GOLD_DIM_BUDGET_CATEGORY_DF,
        "budget_category_key"
    )
]:

    rows = df.count()

    distinct_keys = (
        df
        .select(key)
        .distinct()
        .count()
    )

    null_keys = (
        df
        .filter(
            F.col(key).isNull()
        )
        .count()
    )

    duplicates = (
        rows - distinct_keys
    )

    dimension_validation[name] = {
        "rows": rows,
        "distinct_keys": distinct_keys,
        "null_keys": null_keys,
        "duplicates": duplicates
    }


# =============================================================================
# 7. PRINT VALIDATION
# =============================================================================

print("=" * 100)
print("GOLD BUSINESS MODEL — DIMENSION PREPARATION")
print("=" * 100)


for name, values in dimension_validation.items():

    print(
        f"{name:<25} | "
        f"rows={values['rows']:<7} | "
        f"keys={values['distinct_keys']:<7} | "
        f"null_keys={values['null_keys']:<5} | "
        f"duplicates={values['duplicates']}"
    )


print("=" * 100)


# =============================================================================
# 8. FAIL FAST
# =============================================================================

dimension_failures = [

    name
    for name, values
    in dimension_validation.items()

    if (
        values["null_keys"] > 0
        or
        values["duplicates"] > 0
    )
]


if dimension_failures:

    raise RuntimeError(
        "GOLD DIMENSION PREPARATION FAILED: "
        + ", ".join(
            dimension_failures
        )
    )


# =============================================================================
# 9. DISPLAY BUDGET CATEGORY DIMENSION
# =============================================================================

print("=" * 100)
print("GOLD DIM_BUDGET_CATEGORY")
print("=" * 100)

display(
    GOLD_DIM_BUDGET_CATEGORY_DF
    .orderBy(
        "budget_category_name"
    )
)


# =============================================================================
# 10. FINAL STATUS
# =============================================================================

print(
    "GOLD BUSINESS MODEL — DIMENSION PREPARATION: SUCCEEDED"
)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 7, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — DIMENSION PREPARATION
dim_date                  | rows=5844    | keys=5844    | null_keys=0     | duplicates=0
dim_company               | rows=3       | keys=3       | null_keys=0     | duplicates=0
dim_customer              | rows=1961    | keys=1961    | null_keys=0     | duplicates=0
dim_product               | rows=1353    | keys=1353    | null_keys=0     | duplicates=0
dim_budget_category       | rows=9       | keys=9       | null_keys=0     | duplicates=0
GOLD DIM_BUDGET_CATEGORY


SynapseWidget(Synapse.DataFrame, d3465637-ab3d-49ba-9945-911eb5d2df04)

GOLD BUSINESS MODEL — DIMENSION PREPARATION: SUCCEEDED


In [6]:
# =============================================================================
# CELL 6 — GOLD BUSINESS MODEL
# Gold Fact Preparation
#
# Purpose:
# - Prepare business-facing Gold facts from Silver.
# - Preserve established surrogate keys.
# - Remove unnecessary Silver engineering metadata.
# - Add budget_category_key to fact_budget.
# - Validate Gold fact grain and FK completeness.
# - Reconcile Gold row counts dynamically against Silver source facts.
#
# Important:
# - No Gold tables are written in this cell.
# - Expected row counts are derived dynamically from Silver source facts.
# - This avoids stale hard-coded row-count failures after valid incremental loads.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. GOLD FACT SALES
#
# Grain:
# One row per sales order line.
# =============================================================================

GOLD_FACT_SALES_DF = (
    FACT_SALES_DF

    .select(
        "sales_order_line_key",
        "company_key",
        "customer_key",
        "product_key",

        "order_date_key",
        "requested_delivery_date_key",

        "sales_order_number",
        "sales_order_line_number",

        "order_date",
        "requested_delivery_date",
        "order_status",
        "currency_code",

        "ordered_quantity",
        "unit_price",
        "discount_percentage",
        "discount_amount",

        "gross_amount",
        "net_amount",

        "unit_cost",
        "cost_amount",

        "margin_amount",
        "margin_percentage"
    )
)


# =============================================================================
# 2. GOLD FACT INVOICE
#
# Grain:
# One row per invoice line.
# =============================================================================

GOLD_FACT_INVOICE_DF = (
    FACT_INVOICE_DF

    .select(
        "invoice_line_key",
        "company_key",
        "customer_key",
        "product_key",

        "invoice_date_key",
        "due_date_key",

        "invoice_number",
        "invoice_line_number",

        "invoice_date",
        "due_date",
        "invoice_status",
        "currency_code",

        "quantity",
        "unit_price",

        "discount_percentage",
        "discount_amount",

        "gross_amount",
        "net_amount",

        "unit_cost",
        "cost_amount",

        "margin_amount",
        "margin_percentage"
    )
)


# =============================================================================
# 3. GOLD FACT PAYMENT
#
# Grain:
# One row per payment.
# =============================================================================

GOLD_FACT_PAYMENT_DF = (
    FACT_PAYMENT_DF

    .select(
        "payment_key",
        "company_key",
        "customer_key",

        "payment_date_key",

        "payment_number",

        "source_invoice_id",
        "linked_invoice_number",

        "payment_date",
        "currency_code",
        "payment_method",
        "reference_number",

        "payment_amount",
        "applied_invoice_amount",

        "credit_memo_application_amount",
        "other_non_invoice_application_amount",

        "invoice_application_count",
        "credit_memo_application_count",
        "other_non_invoice_application_count"
    )
)


# =============================================================================
# 4. GOLD FACT INVENTORY
#
# Grain:
# One row per inventory movement transaction.
# =============================================================================

GOLD_FACT_INVENTORY_DF = (
    FACT_INVENTORY_DF

    .select(
        "inventory_transaction_key",
        "company_key",
        "product_key",

        "transaction_date_key",

        "warehouse_code",
        "transaction_date",

        "transaction_type",
        "movement_direction",

        "signed_quantity",
        "quantity_in",
        "quantity_out",

        "unit_cost",
        "transaction_value",

        "reference_document_id"
    )
)


# =============================================================================
# 5. PREPARE BUDGET CATEGORY LOOKUP
# =============================================================================

BUDGET_CATEGORY_LOOKUP_DF = (
    GOLD_DIM_BUDGET_CATEGORY_DF

    .select(
        "budget_category_key",

        F.upper(
            F.trim(
                F.col(
                    "budget_category_name"
                )
            )
        ).alias(
            "_budget_category_normalised"
        )
    )
)


# =============================================================================
# 6. GOLD FACT BUDGET
#
# Grain:
# One row per company + month + budget category.
# =============================================================================

GOLD_FACT_BUDGET_DF = (
    FACT_BUDGET_DF.alias("b")

    .withColumn(
        "_budget_category_normalised",

        F.upper(
            F.trim(
                F.col(
                    "budget_category"
                )
            )
        )
    )

    .join(
        BUDGET_CATEGORY_LOOKUP_DF.alias("bc"),

        on="_budget_category_normalised",

        how="left"
    )

    .select(
        F.col(
            "b.monthly_budget_key"
        ).alias(
            "monthly_budget_key"
        ),

        F.col(
            "b.company_key"
        ).alias(
            "company_key"
        ),

        F.col(
            "b.budget_month_date_key"
        ).alias(
            "budget_month_date_key"
        ),

        F.col(
            "bc.budget_category_key"
        ).alias(
            "budget_category_key"
        ),

        F.col(
            "b.budget_month"
        ).alias(
            "budget_month"
        ),

        F.col(
            "b.budget_year"
        ).alias(
            "budget_year"
        ),

        F.col(
            "b.budget_month_number"
        ).alias(
            "budget_month_number"
        ),

        F.col(
            "b.currency_code"
        ).alias(
            "currency_code"
        ),

        F.col(
            "b.revenue_budget"
        ).alias(
            "revenue_budget"
        ),

        F.col(
            "b.cost_budget"
        ).alias(
            "cost_budget"
        ),

        F.col(
            "b.gross_margin_budget"
        ).alias(
            "gross_margin_budget"
        )
    )
)


# =============================================================================
# 7. DYNAMIC SOURCE COUNTS
#
# Gold should preserve the row grain of each corresponding Silver fact.
# These counts replace stale hard-coded expectations.
# =============================================================================

EXPECTED_FACT_SALES_ROWS = (
    FACT_SALES_DF.count()
)

EXPECTED_FACT_INVOICE_ROWS = (
    FACT_INVOICE_DF.count()
)

EXPECTED_FACT_PAYMENT_ROWS = (
    FACT_PAYMENT_DF.count()
)

EXPECTED_FACT_INVENTORY_ROWS = (
    FACT_INVENTORY_DF.count()
)

EXPECTED_FACT_BUDGET_ROWS = (
    FACT_BUDGET_DF.count()
)


# =============================================================================
# 8. FACT VALIDATION CONFIGURATION
# =============================================================================

fact_validation_config = [

    (
        "fact_sales",
        GOLD_FACT_SALES_DF,
        "sales_order_line_key",
        EXPECTED_FACT_SALES_ROWS
    ),

    (
        "fact_invoice",
        GOLD_FACT_INVOICE_DF,
        "invoice_line_key",
        EXPECTED_FACT_INVOICE_ROWS
    ),

    (
        "fact_payment",
        GOLD_FACT_PAYMENT_DF,
        "payment_key",
        EXPECTED_FACT_PAYMENT_ROWS
    ),

    (
        "fact_inventory",
        GOLD_FACT_INVENTORY_DF,
        "inventory_transaction_key",
        EXPECTED_FACT_INVENTORY_ROWS
    ),

    (
        "fact_budget",
        GOLD_FACT_BUDGET_DF,
        "monthly_budget_key",
        EXPECTED_FACT_BUDGET_ROWS
    )
]


# =============================================================================
# 9. VALIDATE FACT GRAIN
# =============================================================================

fact_validation_results = {}


for (
    fact_name,
    df,
    primary_key,
    expected_rows
) in fact_validation_config:

    row_count = (
        df.count()
    )

    distinct_key_count = (
        df
        .select(
            primary_key
        )
        .distinct()
        .count()
    )

    null_key_count = (
        df
        .filter(
            F.col(
                primary_key
            ).isNull()
        )
        .count()
    )

    duplicate_key_count = (
        row_count
        -
        distinct_key_count
    )

    fact_validation_results[fact_name] = {

        "rows":
            row_count,

        "expected_rows":
            expected_rows,

        "distinct_keys":
            distinct_key_count,

        "null_keys":
            null_key_count,

        "duplicate_keys":
            duplicate_key_count
    }


# =============================================================================
# 10. REQUIRED FK NULL VALIDATION
# =============================================================================

fk_validation = {

    "sales_company":
        GOLD_FACT_SALES_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "sales_customer":
        GOLD_FACT_SALES_DF
        .filter(
            F.col(
                "customer_key"
            ).isNull()
        )
        .count(),

    "sales_product":
        GOLD_FACT_SALES_DF
        .filter(
            F.col(
                "product_key"
            ).isNull()
        )
        .count(),

    "sales_order_date":
        GOLD_FACT_SALES_DF
        .filter(
            F.col(
                "order_date_key"
            ).isNull()
        )
        .count(),

    "invoice_company":
        GOLD_FACT_INVOICE_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "invoice_customer":
        GOLD_FACT_INVOICE_DF
        .filter(
            F.col(
                "customer_key"
            ).isNull()
        )
        .count(),

    "invoice_product":
        GOLD_FACT_INVOICE_DF
        .filter(
            F.col(
                "product_key"
            ).isNull()
        )
        .count(),

    "invoice_date":
        GOLD_FACT_INVOICE_DF
        .filter(
            F.col(
                "invoice_date_key"
            ).isNull()
        )
        .count(),

    "payment_company":
        GOLD_FACT_PAYMENT_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "payment_customer":
        GOLD_FACT_PAYMENT_DF
        .filter(
            F.col(
                "customer_key"
            ).isNull()
        )
        .count(),

    "payment_date":
        GOLD_FACT_PAYMENT_DF
        .filter(
            F.col(
                "payment_date_key"
            ).isNull()
        )
        .count(),

    "inventory_company":
        GOLD_FACT_INVENTORY_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "inventory_product":
        GOLD_FACT_INVENTORY_DF
        .filter(
            F.col(
                "product_key"
            ).isNull()
        )
        .count(),

    "inventory_date":
        GOLD_FACT_INVENTORY_DF
        .filter(
            F.col(
                "transaction_date_key"
            ).isNull()
        )
        .count(),

    "budget_company":
        GOLD_FACT_BUDGET_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "budget_date":
        GOLD_FACT_BUDGET_DF
        .filter(
            F.col(
                "budget_month_date_key"
            ).isNull()
        )
        .count(),

    "budget_category":
        GOLD_FACT_BUDGET_DF
        .filter(
            F.col(
                "budget_category_key"
            ).isNull()
        )
        .count()
}


# =============================================================================
# 11. OPTIONAL SALES DELIVERY DATE DIAGNOSTIC
# =============================================================================

optional_delivery_date_nulls = (
    GOLD_FACT_SALES_DF

    .filter(
        F.col(
            "requested_delivery_date_key"
        ).isNull()
    )

    .count()
)


# =============================================================================
# 12. PRINT FACT VALIDATION
# =============================================================================

print("=" * 100)
print("GOLD BUSINESS MODEL — FACT PREPARATION")
print("=" * 100)


for fact_name, result in fact_validation_results.items():

    print(
        f"{fact_name:<25} | "
        f"rows={result['rows']:<8} | "
        f"expected={result['expected_rows']:<8} | "
        f"keys={result['distinct_keys']:<8} | "
        f"null_keys={result['null_keys']:<5} | "
        f"duplicates={result['duplicate_keys']}"
    )


print("=" * 100)


# =============================================================================
# 13. PRINT FK VALIDATION
# =============================================================================

print("=" * 100)
print("GOLD FACT — REQUIRED FOREIGN KEY VALIDATION")
print("=" * 100)


for fk_name, count in fk_validation.items():

    print(
        f"{fk_name:<35} : "
        f"{count}"
    )


print(
    f"{'sales_requested_delivery_optional':<35} : "
    f"{optional_delivery_date_nulls}"
)

print("=" * 100)


# =============================================================================
# 14. FAIL FAST
# =============================================================================

fact_failures = []


for fact_name, result in fact_validation_results.items():

    if (
        result["rows"]
        !=
        result["expected_rows"]
    ):

        fact_failures.append(
            f"{fact_name}_ROW_COUNT"
        )

    if result["null_keys"] > 0:

        fact_failures.append(
            f"{fact_name}_NULL_PK"
        )

    if result["duplicate_keys"] > 0:

        fact_failures.append(
            f"{fact_name}_DUPLICATE_PK"
        )


for fk_name, count in fk_validation.items():

    if count > 0:

        fact_failures.append(
            f"{fk_name}_NULL_FK"
        )


if fact_failures:

    raise RuntimeError(
        "GOLD FACT PREPARATION FAILED: "
        + ", ".join(
            fact_failures
        )
    )


# =============================================================================
# 15. DISPLAY BUDGET FACT SAMPLE
# =============================================================================

print("=" * 100)
print("GOLD FACT_BUDGET — SAMPLE")
print("=" * 100)


display(
    GOLD_FACT_BUDGET_DF

    .orderBy(
        "company_key",
        "budget_month",
        "budget_category_key"
    )

    .limit(
        50
    )
)


# =============================================================================
# 16. FINAL STATUS
# =============================================================================

print(
    "GOLD BUSINESS MODEL — FACT PREPARATION: SUCCEEDED"
)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 8, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — FACT PREPARATION
fact_sales                | rows=15817    | expected=15817    | keys=15817    | null_keys=0     | duplicates=0
fact_invoice              | rows=13722    | expected=13722    | keys=13722    | null_keys=0     | duplicates=0
fact_payment              | rows=3020     | expected=3020     | keys=3020     | null_keys=0     | duplicates=0
fact_inventory            | rows=9007     | expected=9007     | keys=9007     | null_keys=0     | duplicates=0
fact_budget               | rows=368      | expected=368      | keys=368      | null_keys=0     | duplicates=0
GOLD FACT — REQUIRED FOREIGN KEY VALIDATION
sales_company                       : 0
sales_customer                      : 0
sales_product                       : 0
sales_order_date                    : 0
invoice_company                     : 0
invoice_customer                    : 0
invoice_product                     : 0
invoice_date                        : 0
payment_company                     : 0
pa

SynapseWidget(Synapse.DataFrame, 7d8f0912-53fd-4ef2-9846-2200b0ee92c6)

GOLD BUSINESS MODEL — FACT PREPARATION: SUCCEEDED


In [7]:
# =============================================================================
# CELL 7 — GOLD BUSINESS MODEL
# Business Aggregates Preparation
#
# Purpose:
# - Build Power BI-ready business aggregates.
# - Keep calculations deterministic and reusable.
# - Align Budget vs Actual at COMPANY + MONTH grain.
# - Prepare customer, product, cash and inventory performance.
#
# IMPORTANT:
# - No Gold tables are written yet.
# - This cell only prepares and validates aggregate DataFrames.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. HELPER — MONTH KEY
# =============================================================================

def month_key_from_date(date_column):
    return F.date_format(
        F.trunc(
            date_column,
            "month"
        ),
        "yyyyMMdd"
    ).cast("int")


# =============================================================================
# 2. MONTHLY SALES AGGREGATE
# =============================================================================

MONTHLY_SALES_DF = (
    GOLD_FACT_SALES_DF

    .withColumn(
        "month_date_key",
        month_key_from_date(
            F.col("order_date")
        )
    )

    .groupBy(
        "company_key",
        "month_date_key"
    )

    .agg(

        F.countDistinct(
            "sales_order_number"
        ).alias(
            "sales_order_count"
        ),

        F.count(
            "*"
        ).alias(
            "sales_order_line_count"
        ),

        F.countDistinct(
            "customer_key"
        ).alias(
            "sales_customer_count"
        ),

        F.countDistinct(
            "product_key"
        ).alias(
            "sales_product_count"
        ),

        F.sum(
            "ordered_quantity"
        ).alias(
            "ordered_quantity"
        ),

        F.sum(
            "gross_amount"
        ).alias(
            "sales_gross_amount"
        ),

        F.sum(
            "discount_amount"
        ).alias(
            "sales_discount_amount"
        ),

        F.sum(
            "net_amount"
        ).alias(
            "sales_net_amount"
        ),

        F.sum(
            "cost_amount"
        ).alias(
            "sales_cost_amount"
        ),

        F.sum(
            "margin_amount"
        ).alias(
            "sales_margin_amount"
        )
    )
)


# =============================================================================
# 3. MONTHLY INVOICE / ACTUAL AGGREGATE
# =============================================================================

MONTHLY_INVOICE_DF = (
    GOLD_FACT_INVOICE_DF

    .withColumn(
        "month_date_key",
        month_key_from_date(
            F.col("invoice_date")
        )
    )

    .groupBy(
        "company_key",
        "month_date_key"
    )

    .agg(

        F.countDistinct(
            "invoice_number"
        ).alias(
            "invoice_count"
        ),

        F.count(
            "*"
        ).alias(
            "invoice_line_count"
        ),

        F.countDistinct(
            "customer_key"
        ).alias(
            "invoiced_customer_count"
        ),

        F.countDistinct(
            "product_key"
        ).alias(
            "invoiced_product_count"
        ),

        F.sum(
            "quantity"
        ).alias(
            "invoiced_quantity"
        ),

        F.sum(
            "gross_amount"
        ).alias(
            "actual_gross_revenue"
        ),

        F.sum(
            "discount_amount"
        ).alias(
            "actual_discount_amount"
        ),

        F.sum(
            "net_amount"
        ).alias(
            "actual_revenue"
        ),

        F.sum(
            "cost_amount"
        ).alias(
            "actual_cost"
        ),

        F.sum(
            "margin_amount"
        ).alias(
            "actual_margin"
        )
    )
)


# =============================================================================
# 4. MONTHLY PAYMENT AGGREGATE
# =============================================================================

MONTHLY_PAYMENT_DF = (
    GOLD_FACT_PAYMENT_DF

    .withColumn(
        "month_date_key",
        month_key_from_date(
            F.col("payment_date")
        )
    )

    .groupBy(
        "company_key",
        "month_date_key"
    )

    .agg(

        F.countDistinct(
            "payment_key"
        ).alias(
            "payment_count"
        ),

        F.countDistinct(
            "customer_key"
        ).alias(
            "paying_customer_count"
        ),

        F.sum(
            "payment_amount"
        ).alias(
            "cash_collected"
        ),

        F.sum(
            "applied_invoice_amount"
        ).alias(
            "invoice_cash_applied"
        ),

        F.sum(
            "credit_memo_application_amount"
        ).alias(
            "credit_memo_amount"
        ),

        F.sum(
            "other_non_invoice_application_amount"
        ).alias(
            "other_payment_application_amount"
        )
    )
)


# =============================================================================
# 5. MONTHLY INVENTORY MOVEMENT AGGREGATE
# =============================================================================

MONTHLY_INVENTORY_DF = (
    GOLD_FACT_INVENTORY_DF

    .withColumn(
        "month_date_key",
        month_key_from_date(
            F.col("transaction_date")
        )
    )

    .groupBy(
        "company_key",
        "month_date_key"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "inventory_transaction_count"
        ),

        F.countDistinct(
            "product_key"
        ).alias(
            "inventory_product_count"
        ),

        F.sum(
            "quantity_in"
        ).alias(
            "inventory_quantity_in"
        ),

        F.sum(
            "quantity_out"
        ).alias(
            "inventory_quantity_out"
        ),

        F.sum(
            "signed_quantity"
        ).alias(
            "inventory_net_quantity"
        ),

        F.sum(
            "transaction_value"
        ).alias(
            "inventory_transaction_value"
        )
    )
)


# =============================================================================
# 6. MONTHLY BUDGET AGGREGATE
#
# IMPORTANT:
# Budget is aggregated only to COMPANY + MONTH for Budget vs Actual.
# Budget category and product category are intentionally NOT forced together.
# =============================================================================

MONTHLY_BUDGET_DF = (
    GOLD_FACT_BUDGET_DF

    .groupBy(
        "company_key",
        F.col(
            "budget_month_date_key"
        ).alias(
            "month_date_key"
        )
    )

    .agg(

        F.countDistinct(
            "budget_category_key"
        ).alias(
            "budget_category_count"
        ),

        F.sum(
            "revenue_budget"
        ).alias(
            "budget_revenue"
        ),

        F.sum(
            "cost_budget"
        ).alias(
            "budget_cost"
        ),

        F.sum(
            "gross_margin_budget"
        ).alias(
            "budget_margin"
        )
    )
)


# =============================================================================
# 7. COMPANY MONTHLY PERFORMANCE
#
# Conformed grain:
# ONE ROW = COMPANY + MONTH
# =============================================================================

AGG_COMPANY_MONTHLY_PERFORMANCE_DF = (
    MONTHLY_INVOICE_DF.alias("i")

    .join(
        MONTHLY_SALES_DF.alias("s"),
        on=[
            "company_key",
            "month_date_key"
        ],
        how="full_outer"
    )

    .join(
        MONTHLY_PAYMENT_DF.alias("p"),
        on=[
            "company_key",
            "month_date_key"
        ],
        how="full_outer"
    )

    .join(
        MONTHLY_INVENTORY_DF.alias("inv"),
        on=[
            "company_key",
            "month_date_key"
        ],
        how="full_outer"
    )

    .select(

        "company_key",
        "month_date_key",

        # Sales
        "sales_order_count",
        "sales_order_line_count",
        "sales_customer_count",
        "sales_product_count",
        "ordered_quantity",
        "sales_gross_amount",
        "sales_discount_amount",
        "sales_net_amount",
        "sales_cost_amount",
        "sales_margin_amount",

        # Invoice
        "invoice_count",
        "invoice_line_count",
        "invoiced_customer_count",
        "invoiced_product_count",
        "invoiced_quantity",
        "actual_gross_revenue",
        "actual_discount_amount",
        "actual_revenue",
        "actual_cost",
        "actual_margin",

        # Payments
        "payment_count",
        "paying_customer_count",
        "cash_collected",
        "invoice_cash_applied",
        "credit_memo_amount",
        "other_payment_application_amount",

        # Inventory
        "inventory_transaction_count",
        "inventory_product_count",
        "inventory_quantity_in",
        "inventory_quantity_out",
        "inventory_net_quantity",
        "inventory_transaction_value"
    )
)


# =============================================================================
# 8. DERIVED COMPANY PERFORMANCE KPIs
# =============================================================================

AGG_COMPANY_MONTHLY_PERFORMANCE_DF = (
    AGG_COMPANY_MONTHLY_PERFORMANCE_DF

    .withColumn(
        "actual_margin_percentage",

        F.when(
            F.col("actual_revenue") != 0,

            (
                F.col("actual_margin")
                /
                F.col("actual_revenue")
            )
            * 100
        )
    )

    .withColumn(
        "sales_margin_percentage",

        F.when(
            F.col("sales_net_amount") != 0,

            (
                F.col("sales_margin_amount")
                /
                F.col("sales_net_amount")
            )
            * 100
        )
    )

    .withColumn(
        "average_invoice_value",

        F.when(
            F.col("invoice_count") != 0,

            F.col("actual_revenue")
            /
            F.col("invoice_count")
        )
    )

    .withColumn(
        "average_sales_order_value",

        F.when(
            F.col("sales_order_count") != 0,

            F.col("sales_net_amount")
            /
            F.col("sales_order_count")
        )
    )

    .withColumn(
        "cash_collection_rate",

        F.when(
            F.col("actual_revenue") != 0,

            (
                F.col("cash_collected")
                /
                F.col("actual_revenue")
            )
            * 100
        )
    )
)


# =============================================================================
# 9. BUDGET VS ACTUAL
#
# Grain:
# ONE ROW = COMPANY + MONTH
# =============================================================================

AGG_BUDGET_VS_ACTUAL_DF = (
    MONTHLY_BUDGET_DF.alias("b")

    .join(
        MONTHLY_INVOICE_DF.alias("a"),

        on=[
            "company_key",
            "month_date_key"
        ],

        how="full_outer"
    )

    .select(

        "company_key",
        "month_date_key",

        "budget_category_count",

        "budget_revenue",
        "actual_revenue",

        "budget_cost",
        "actual_cost",

        "budget_margin",
        "actual_margin",

        "invoice_count",
        "invoiced_customer_count"
    )

    .withColumn(
        "revenue_variance",

        F.col("actual_revenue")
        -
        F.col("budget_revenue")
    )

    .withColumn(
        "revenue_variance_percentage",

        F.when(
            F.col("budget_revenue") != 0,

            (
                (
                    F.col("actual_revenue")
                    -
                    F.col("budget_revenue")
                )
                /
                F.col("budget_revenue")
            )
            * 100
        )
    )

    .withColumn(
        "cost_variance",

        F.col("actual_cost")
        -
        F.col("budget_cost")
    )

    .withColumn(
        "cost_variance_percentage",

        F.when(
            F.col("budget_cost") != 0,

            (
                (
                    F.col("actual_cost")
                    -
                    F.col("budget_cost")
                )
                /
                F.col("budget_cost")
            )
            * 100
        )
    )

    .withColumn(
        "margin_variance",

        F.col("actual_margin")
        -
        F.col("budget_margin")
    )

    .withColumn(
        "budget_margin_percentage",

        F.when(
            F.col("budget_revenue") != 0,

            (
                F.col("budget_margin")
                /
                F.col("budget_revenue")
            )
            * 100
        )
    )

    .withColumn(
        "actual_margin_percentage",

        F.when(
            F.col("actual_revenue") != 0,

            (
                F.col("actual_margin")
                /
                F.col("actual_revenue")
            )
            * 100
        )
    )
)


# =============================================================================
# 10. CUSTOMER PERFORMANCE
#
# Grain:
# ONE ROW = COMPANY + CUSTOMER
# =============================================================================

AGG_CUSTOMER_PERFORMANCE_DF = (
    GOLD_FACT_INVOICE_DF

    .groupBy(
        "company_key",
        "customer_key"
    )

    .agg(

        F.countDistinct(
            "invoice_number"
        ).alias(
            "invoice_count"
        ),

        F.countDistinct(
            "product_key"
        ).alias(
            "products_purchased"
        ),

        F.min(
            "invoice_date"
        ).alias(
            "first_invoice_date"
        ),

        F.max(
            "invoice_date"
        ).alias(
            "latest_invoice_date"
        ),

        F.sum(
            "quantity"
        ).alias(
            "quantity_purchased"
        ),

        F.sum(
            "net_amount"
        ).alias(
            "revenue"
        ),

        F.sum(
            "cost_amount"
        ).alias(
            "cost"
        ),

        F.sum(
            "margin_amount"
        ).alias(
            "margin"
        )
    )

    .withColumn(
        "margin_percentage",

        F.when(
            F.col("revenue") != 0,

            (
                F.col("margin")
                /
                F.col("revenue")
            )
            * 100
        )
    )

    .withColumn(
        "average_invoice_value",

        F.when(
            F.col("invoice_count") != 0,

            F.col("revenue")
            /
            F.col("invoice_count")
        )
    )
)


# =============================================================================
# 11. PRODUCT PERFORMANCE
#
# Grain:
# ONE ROW = COMPANY + PRODUCT
# =============================================================================

AGG_PRODUCT_PERFORMANCE_DF = (
    GOLD_FACT_INVOICE_DF

    .groupBy(
        "company_key",
        "product_key"
    )

    .agg(

        F.countDistinct(
            "invoice_number"
        ).alias(
            "invoice_count"
        ),

        F.countDistinct(
            "customer_key"
        ).alias(
            "customer_count"
        ),

        F.sum(
            "quantity"
        ).alias(
            "quantity_sold"
        ),

        F.sum(
            "net_amount"
        ).alias(
            "revenue"
        ),

        F.sum(
            "cost_amount"
        ).alias(
            "cost"
        ),

        F.sum(
            "margin_amount"
        ).alias(
            "margin"
        )
    )

    .withColumn(
        "margin_percentage",

        F.when(
            F.col("revenue") != 0,

            (
                F.col("margin")
                /
                F.col("revenue")
            )
            * 100
        )
    )
)


# =============================================================================
# 12. CASH COLLECTION
#
# Grain:
# ONE ROW = COMPANY + CUSTOMER
# =============================================================================

AGG_CASH_COLLECTION_DF = (
    GOLD_FACT_PAYMENT_DF

    .groupBy(
        "company_key",
        "customer_key"
    )

    .agg(

        F.countDistinct(
            "payment_key"
        ).alias(
            "payment_count"
        ),

        F.min(
            "payment_date"
        ).alias(
            "first_payment_date"
        ),

        F.max(
            "payment_date"
        ).alias(
            "latest_payment_date"
        ),

        F.sum(
            "payment_amount"
        ).alias(
            "cash_collected"
        ),

        F.sum(
            "applied_invoice_amount"
        ).alias(
            "invoice_cash_applied"
        ),

        F.sum(
            "credit_memo_application_amount"
        ).alias(
            "credit_memo_amount"
        ),

        F.sum(
            "other_non_invoice_application_amount"
        ).alias(
            "other_application_amount"
        )
    )
)


# =============================================================================
# 13. INVENTORY MOVEMENT
#
# Grain:
# ONE ROW = COMPANY + PRODUCT + WAREHOUSE
# =============================================================================

AGG_INVENTORY_MOVEMENT_DF = (
    GOLD_FACT_INVENTORY_DF

    .groupBy(
        "company_key",
        "product_key",
        "warehouse_code"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "transaction_count"
        ),

        F.min(
            "transaction_date"
        ).alias(
            "first_transaction_date"
        ),

        F.max(
            "transaction_date"
        ).alias(
            "latest_transaction_date"
        ),

        F.sum(
            "quantity_in"
        ).alias(
            "quantity_in"
        ),

        F.sum(
            "quantity_out"
        ).alias(
            "quantity_out"
        ),

        F.sum(
            "signed_quantity"
        ).alias(
            "net_quantity"
        ),

        F.sum(
            "transaction_value"
        ).alias(
            "transaction_value"
        )
    )
)


# =============================================================================
# 14. VALIDATE COMPANY-MONTH GRAIN
# =============================================================================

company_month_duplicate_count = (
    AGG_COMPANY_MONTHLY_PERFORMANCE_DF

    .groupBy(
        "company_key",
        "month_date_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


budget_actual_duplicate_count = (
    AGG_BUDGET_VS_ACTUAL_DF

    .groupBy(
        "company_key",
        "month_date_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


# =============================================================================
# 15. VALIDATE CUSTOMER / PRODUCT AGGREGATE GRAINS
# =============================================================================

customer_duplicate_count = (
    AGG_CUSTOMER_PERFORMANCE_DF

    .groupBy(
        "company_key",
        "customer_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


product_duplicate_count = (
    AGG_PRODUCT_PERFORMANCE_DF

    .groupBy(
        "company_key",
        "product_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


cash_duplicate_count = (
    AGG_CASH_COLLECTION_DF

    .groupBy(
        "company_key",
        "customer_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


inventory_duplicate_count = (
    AGG_INVENTORY_MOVEMENT_DF

    .groupBy(
        "company_key",
        "product_key",
        "warehouse_code"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


# =============================================================================
# 16. ROW COUNTS
# =============================================================================

aggregate_results = {

    "agg_company_monthly_performance":
        AGG_COMPANY_MONTHLY_PERFORMANCE_DF.count(),

    "agg_budget_vs_actual":
        AGG_BUDGET_VS_ACTUAL_DF.count(),

    "agg_customer_performance":
        AGG_CUSTOMER_PERFORMANCE_DF.count(),

    "agg_product_performance":
        AGG_PRODUCT_PERFORMANCE_DF.count(),

    "agg_cash_collection":
        AGG_CASH_COLLECTION_DF.count(),

    "agg_inventory_movement":
        AGG_INVENTORY_MOVEMENT_DF.count()
}


# =============================================================================
# 17. PRINT SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD BUSINESS MODEL — BUSINESS AGGREGATES")
print("=" * 100)

for name, count in aggregate_results.items():

    print(
        f"{name:<40} : "
        f"{count}"
    )


print("-" * 100)

print(
    f"Company-month duplicate grain  : "
    f"{company_month_duplicate_count}"
)

print(
    f"Budget-vs-actual duplicate grain: "
    f"{budget_actual_duplicate_count}"
)

print(
    f"Customer duplicate grain       : "
    f"{customer_duplicate_count}"
)

print(
    f"Product duplicate grain        : "
    f"{product_duplicate_count}"
)

print(
    f"Cash duplicate grain           : "
    f"{cash_duplicate_count}"
)

print(
    f"Inventory duplicate grain      : "
    f"{inventory_duplicate_count}"
)

print("=" * 100)


# =============================================================================
# 18. FAIL FAST
# =============================================================================

aggregate_failures = []


if company_month_duplicate_count > 0:
    aggregate_failures.append(
        "COMPANY_MONTHLY_DUPLICATE_GRAIN"
    )

if budget_actual_duplicate_count > 0:
    aggregate_failures.append(
        "BUDGET_ACTUAL_DUPLICATE_GRAIN"
    )

if customer_duplicate_count > 0:
    aggregate_failures.append(
        "CUSTOMER_DUPLICATE_GRAIN"
    )

if product_duplicate_count > 0:
    aggregate_failures.append(
        "PRODUCT_DUPLICATE_GRAIN"
    )

if cash_duplicate_count > 0:
    aggregate_failures.append(
        "CASH_DUPLICATE_GRAIN"
    )

if inventory_duplicate_count > 0:
    aggregate_failures.append(
        "INVENTORY_DUPLICATE_GRAIN"
    )


if aggregate_failures:

    raise RuntimeError(
        "GOLD BUSINESS AGGREGATE PREPARATION FAILED: "
        + ", ".join(
            aggregate_failures
        )
    )


# =============================================================================
# 19. DISPLAY BUDGET VS ACTUAL SAMPLE
# =============================================================================

print("=" * 100)
print("GOLD AGG_BUDGET_VS_ACTUAL — SAMPLE")
print("=" * 100)

display(
    AGG_BUDGET_VS_ACTUAL_DF

    .orderBy(
        "company_key",
        "month_date_key"
    )

    .limit(
        100
    )
)


# =============================================================================
# 20. DISPLAY COMPANY MONTHLY PERFORMANCE SAMPLE
# =============================================================================

print("=" * 100)
print("GOLD AGG_COMPANY_MONTHLY_PERFORMANCE — SAMPLE")
print("=" * 100)

display(
    AGG_COMPANY_MONTHLY_PERFORMANCE_DF

    .orderBy(
        "company_key",
        "month_date_key"
    )

    .limit(
        100
    )
)


# =============================================================================
# 21. FINAL STATUS
# =============================================================================

print(
    "GOLD BUSINESS MODEL — BUSINESS AGGREGATES: SUCCEEDED"
)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 9, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — BUSINESS AGGREGATES
agg_company_monthly_performance          : 83
agg_budget_vs_actual                     : 82
agg_customer_performance                 : 913
agg_product_performance                  : 1228
agg_cash_collection                      : 821
agg_inventory_movement                   : 4065
----------------------------------------------------------------------------------------------------
Company-month duplicate grain  : 0
Budget-vs-actual duplicate grain: 0
Customer duplicate grain       : 0
Product duplicate grain        : 0
Cash duplicate grain           : 0
Inventory duplicate grain      : 0
GOLD AGG_BUDGET_VS_ACTUAL — SAMPLE


SynapseWidget(Synapse.DataFrame, 72d2f517-b330-4b7e-9187-405ab21297f0)

GOLD AGG_COMPANY_MONTHLY_PERFORMANCE — SAMPLE


SynapseWidget(Synapse.DataFrame, 36d2b26d-125e-4863-b249-6a235674bb9b)

GOLD BUSINESS MODEL — BUSINESS AGGREGATES: SUCCEEDED


In [8]:
# =============================================================================
# CELL 8 — GOLD BUSINESS MODEL
# Pre-Write Validation + Source-to-Gold Reconciliation
#
# Purpose:
# - Validate Gold dimensions, facts, and aggregates before writing.
# - Reconcile Gold facts back to Silver source row counts.
# - Reconcile key business measures.
# - Validate aggregate totals against Gold facts.
# - Fail fast on critical integrity problems.
#
# No Gold tables are written in this cell.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. DIMENSION ROW COUNTS
# =============================================================================

gold_dim_counts = {
    "dim_date": GOLD_DIM_DATE_DF.count(),
    "dim_company": GOLD_DIM_COMPANY_DF.count(),
    "dim_customer": GOLD_DIM_CUSTOMER_DF.count(),
    "dim_product": GOLD_DIM_PRODUCT_DF.count(),
    "dim_budget_category": GOLD_DIM_BUDGET_CATEGORY_DF.count(),
}


# =============================================================================
# 2. FACT ROW COUNTS
# =============================================================================

gold_fact_counts = {
    "fact_sales": GOLD_FACT_SALES_DF.count(),
    "fact_invoice": GOLD_FACT_INVOICE_DF.count(),
    "fact_payment": GOLD_FACT_PAYMENT_DF.count(),
    "fact_inventory": GOLD_FACT_INVENTORY_DF.count(),
    "fact_budget": GOLD_FACT_BUDGET_DF.count(),
}


# =============================================================================
# 3. SILVER EXPECTED FACT COUNTS
# =============================================================================

silver_fact_counts = {
    "fact_sales": FACT_SALES_DF.count(),
    "fact_invoice": FACT_INVOICE_DF.count(),
    "fact_payment": FACT_PAYMENT_DF.count(),
    "fact_inventory": FACT_INVENTORY_DF.count(),
    "fact_budget": FACT_BUDGET_DF.count(),
}


# =============================================================================
# 4. FACT ROW-COUNT RECONCILIATION
# =============================================================================

fact_row_count_failures = {}

for fact_name in gold_fact_counts:

    expected = silver_fact_counts[fact_name]
    actual = gold_fact_counts[fact_name]

    if actual != expected:
        fact_row_count_failures[fact_name] = {
            "expected": expected,
            "actual": actual
        }


# =============================================================================
# 5. SALES MEASURE RECONCILIATION
# =============================================================================

silver_sales_totals = (
    FACT_SALES_DF
    .agg(
        F.sum("ordered_quantity").alias("ordered_quantity"),
        F.sum("gross_amount").alias("gross_amount"),
        F.sum("discount_amount").alias("discount_amount"),
        F.sum("net_amount").alias("net_amount"),
        F.sum("cost_amount").alias("cost_amount"),
        F.sum("margin_amount").alias("margin_amount"),
    )
    .first()
)

gold_sales_totals = (
    GOLD_FACT_SALES_DF
    .agg(
        F.sum("ordered_quantity").alias("ordered_quantity"),
        F.sum("gross_amount").alias("gross_amount"),
        F.sum("discount_amount").alias("discount_amount"),
        F.sum("net_amount").alias("net_amount"),
        F.sum("cost_amount").alias("cost_amount"),
        F.sum("margin_amount").alias("margin_amount"),
    )
    .first()
)


# =============================================================================
# 6. INVOICE MEASURE RECONCILIATION
# =============================================================================

silver_invoice_totals = (
    FACT_INVOICE_DF
    .agg(
        F.sum("quantity").alias("quantity"),
        F.sum("gross_amount").alias("gross_amount"),
        F.sum("discount_amount").alias("discount_amount"),
        F.sum("net_amount").alias("net_amount"),
        F.sum("cost_amount").alias("cost_amount"),
        F.sum("margin_amount").alias("margin_amount"),
    )
    .first()
)

gold_invoice_totals = (
    GOLD_FACT_INVOICE_DF
    .agg(
        F.sum("quantity").alias("quantity"),
        F.sum("gross_amount").alias("gross_amount"),
        F.sum("discount_amount").alias("discount_amount"),
        F.sum("net_amount").alias("net_amount"),
        F.sum("cost_amount").alias("cost_amount"),
        F.sum("margin_amount").alias("margin_amount"),
    )
    .first()
)


# =============================================================================
# 7. PAYMENT MEASURE RECONCILIATION
# =============================================================================

silver_payment_totals = (
    FACT_PAYMENT_DF
    .agg(
        F.sum("payment_amount").alias("payment_amount"),
        F.sum("applied_invoice_amount").alias("applied_invoice_amount"),
        F.sum("credit_memo_application_amount").alias("credit_memo_amount"),
        F.sum("other_non_invoice_application_amount").alias("other_amount"),
    )
    .first()
)

gold_payment_totals = (
    GOLD_FACT_PAYMENT_DF
    .agg(
        F.sum("payment_amount").alias("payment_amount"),
        F.sum("applied_invoice_amount").alias("applied_invoice_amount"),
        F.sum("credit_memo_application_amount").alias("credit_memo_amount"),
        F.sum("other_non_invoice_application_amount").alias("other_amount"),
    )
    .first()
)


# =============================================================================
# 8. INVENTORY MEASURE RECONCILIATION
# =============================================================================

silver_inventory_totals = (
    FACT_INVENTORY_DF
    .agg(
        F.sum("signed_quantity").alias("signed_quantity"),
        F.sum("quantity_in").alias("quantity_in"),
        F.sum("quantity_out").alias("quantity_out"),
        F.sum("transaction_value").alias("transaction_value"),
    )
    .first()
)

gold_inventory_totals = (
    GOLD_FACT_INVENTORY_DF
    .agg(
        F.sum("signed_quantity").alias("signed_quantity"),
        F.sum("quantity_in").alias("quantity_in"),
        F.sum("quantity_out").alias("quantity_out"),
        F.sum("transaction_value").alias("transaction_value"),
    )
    .first()
)


# =============================================================================
# 9. BUDGET MEASURE RECONCILIATION
# =============================================================================

silver_budget_totals = (
    FACT_BUDGET_DF
    .agg(
        F.sum("revenue_budget").alias("revenue_budget"),
        F.sum("cost_budget").alias("cost_budget"),
        F.sum("gross_margin_budget").alias("gross_margin_budget"),
    )
    .first()
)

gold_budget_totals = (
    GOLD_FACT_BUDGET_DF
    .agg(
        F.sum("revenue_budget").alias("revenue_budget"),
        F.sum("cost_budget").alias("cost_budget"),
        F.sum("gross_margin_budget").alias("gross_margin_budget"),
    )
    .first()
)


# =============================================================================
# 10. HELPER — DECIMAL-SAFE DIFFERENCE
# =============================================================================

def values_differ(a, b, tolerance=0.01):

    a_val = float(a or 0)
    b_val = float(b or 0)

    return abs(a_val - b_val) > tolerance


# =============================================================================
# 11. BUILD MEASURE RECONCILIATION FAILURES
# =============================================================================

measure_failures = []


# Sales
for field in [
    "ordered_quantity",
    "gross_amount",
    "discount_amount",
    "net_amount",
    "cost_amount",
    "margin_amount",
]:

    if values_differ(
        silver_sales_totals[field],
        gold_sales_totals[field]
    ):
        measure_failures.append(
            f"SALES_{field.upper()}"
        )


# Invoice
for field in [
    "quantity",
    "gross_amount",
    "discount_amount",
    "net_amount",
    "cost_amount",
    "margin_amount",
]:

    if values_differ(
        silver_invoice_totals[field],
        gold_invoice_totals[field]
    ):
        measure_failures.append(
            f"INVOICE_{field.upper()}"
        )


# Payment
for field in [
    "payment_amount",
    "applied_invoice_amount",
    "credit_memo_amount",
    "other_amount",
]:

    if values_differ(
        silver_payment_totals[field],
        gold_payment_totals[field]
    ):
        measure_failures.append(
            f"PAYMENT_{field.upper()}"
        )


# Inventory
for field in [
    "signed_quantity",
    "quantity_in",
    "quantity_out",
    "transaction_value",
]:

    if values_differ(
        silver_inventory_totals[field],
        gold_inventory_totals[field]
    ):
        measure_failures.append(
            f"INVENTORY_{field.upper()}"
        )


# Budget
for field in [
    "revenue_budget",
    "cost_budget",
    "gross_margin_budget",
]:

    if values_differ(
        silver_budget_totals[field],
        gold_budget_totals[field]
    ):
        measure_failures.append(
            f"BUDGET_{field.upper()}"
        )


# =============================================================================
# 12. AGGREGATE-TO-FACT RECONCILIATION
# =============================================================================

agg_invoice_total = (
    AGG_COMPANY_MONTHLY_PERFORMANCE_DF
    .agg(
        F.sum("actual_revenue").alias("actual_revenue")
    )
    .first()["actual_revenue"]
)

fact_invoice_total = (
    GOLD_FACT_INVOICE_DF
    .agg(
        F.sum("net_amount").alias("actual_revenue")
    )
    .first()["actual_revenue"]
)


agg_cash_total = (
    AGG_COMPANY_MONTHLY_PERFORMANCE_DF
    .agg(
        F.sum("cash_collected").alias("cash_collected")
    )
    .first()["cash_collected"]
)

fact_cash_total = (
    GOLD_FACT_PAYMENT_DF
    .agg(
        F.sum("payment_amount").alias("cash_collected")
    )
    .first()["cash_collected"]
)


agg_budget_total = (
    AGG_BUDGET_VS_ACTUAL_DF
    .agg(
        F.sum("budget_revenue").alias("budget_revenue")
    )
    .first()["budget_revenue"]
)

fact_budget_total = (
    GOLD_FACT_BUDGET_DF
    .agg(
        F.sum("revenue_budget").alias("budget_revenue")
    )
    .first()["budget_revenue"]
)


agg_inventory_net_total = (
    AGG_COMPANY_MONTHLY_PERFORMANCE_DF
    .agg(
        F.sum("inventory_net_quantity").alias("inventory_net_quantity")
    )
    .first()["inventory_net_quantity"]
)

fact_inventory_net_total = (
    GOLD_FACT_INVENTORY_DF
    .agg(
        F.sum("signed_quantity").alias("inventory_net_quantity")
    )
    .first()["inventory_net_quantity"]
)


# =============================================================================
# 13. AGGREGATE RECONCILIATION FAILURES
# =============================================================================

aggregate_reconciliation_failures = []


if values_differ(
    agg_invoice_total,
    fact_invoice_total
):
    aggregate_reconciliation_failures.append(
        "AGG_INVOICE_REVENUE"
    )


if values_differ(
    agg_cash_total,
    fact_cash_total
):
    aggregate_reconciliation_failures.append(
        "AGG_CASH_COLLECTED"
    )


if values_differ(
    agg_budget_total,
    fact_budget_total
):
    aggregate_reconciliation_failures.append(
        "AGG_BUDGET_REVENUE"
    )


if values_differ(
    agg_inventory_net_total,
    fact_inventory_net_total
):
    aggregate_reconciliation_failures.append(
        "AGG_INVENTORY_NET_QUANTITY"
    )


# =============================================================================
# 14. GOLD FK NULL VALIDATION
# =============================================================================

gold_fk_failures = {
    "sales_company":
        GOLD_FACT_SALES_DF.filter(
            F.col("company_key").isNull()
        ).count(),

    "sales_customer":
        GOLD_FACT_SALES_DF.filter(
            F.col("customer_key").isNull()
        ).count(),

    "sales_product":
        GOLD_FACT_SALES_DF.filter(
            F.col("product_key").isNull()
        ).count(),

    "invoice_company":
        GOLD_FACT_INVOICE_DF.filter(
            F.col("company_key").isNull()
        ).count(),

    "invoice_customer":
        GOLD_FACT_INVOICE_DF.filter(
            F.col("customer_key").isNull()
        ).count(),

    "invoice_product":
        GOLD_FACT_INVOICE_DF.filter(
            F.col("product_key").isNull()
        ).count(),

    "payment_company":
        GOLD_FACT_PAYMENT_DF.filter(
            F.col("company_key").isNull()
        ).count(),

    "payment_customer":
        GOLD_FACT_PAYMENT_DF.filter(
            F.col("customer_key").isNull()
        ).count(),

    "inventory_company":
        GOLD_FACT_INVENTORY_DF.filter(
            F.col("company_key").isNull()
        ).count(),

    "inventory_product":
        GOLD_FACT_INVENTORY_DF.filter(
            F.col("product_key").isNull()
        ).count(),

    "budget_company":
        GOLD_FACT_BUDGET_DF.filter(
            F.col("company_key").isNull()
        ).count(),

    "budget_category":
        GOLD_FACT_BUDGET_DF.filter(
            F.col("budget_category_key").isNull()
        ).count(),
}


# =============================================================================
# 15. COUNT FK FAILURES
# =============================================================================

failed_fk_rules = {
    k: v
    for k, v in gold_fk_failures.items()
    if v > 0
}


# =============================================================================
# 16. PRINT VALIDATION SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD BUSINESS MODEL — PRE-WRITE VALIDATION")
print("=" * 100)

print("DIMENSIONS")
print("-" * 100)

for name, count in gold_dim_counts.items():
    print(
        f"{name:<30} : {count}"
    )

print("-" * 100)
print("FACTS")
print("-" * 100)

for name, count in gold_fact_counts.items():
    print(
        f"{name:<30} : {count}"
    )

print("-" * 100)

print(
    f"Fact row-count failures          : "
    f"{len(fact_row_count_failures)}"
)

print(
    f"Measure reconciliation failures  : "
    f"{len(measure_failures)}"
)

print(
    f"Aggregate reconciliation failures: "
    f"{len(aggregate_reconciliation_failures)}"
)

print(
    f"Foreign-key rules failed         : "
    f"{len(failed_fk_rules)}"
)

print("=" * 100)


# =============================================================================
# 17. PRINT DETAILED FAILURES
# =============================================================================

if fact_row_count_failures:

    print("FACT ROW COUNT FAILURES")

    for fact_name, values in fact_row_count_failures.items():

        print(
            fact_name,
            values
        )


if measure_failures:

    print(
        "MEASURE FAILURES:",
        measure_failures
    )


if aggregate_reconciliation_failures:

    print(
        "AGGREGATE FAILURES:",
        aggregate_reconciliation_failures
    )


if failed_fk_rules:

    print(
        "FK FAILURES:",
        failed_fk_rules
    )


# =============================================================================
# 18. FAIL FAST
# =============================================================================

critical_failure_count = (
    len(fact_row_count_failures)
    +
    len(measure_failures)
    +
    len(aggregate_reconciliation_failures)
    +
    len(failed_fk_rules)
)


if critical_failure_count > 0:

    raise RuntimeError(
        f"GOLD PRE-WRITE VALIDATION FAILED — "
        f"{critical_failure_count} critical validation issue(s)."
    )


# =============================================================================
# 19. FINAL STATUS
# =============================================================================

print(
    "GOLD BUSINESS MODEL — PRE-WRITE VALIDATION: SUCCEEDED"
)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 10, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — PRE-WRITE VALIDATION
DIMENSIONS
----------------------------------------------------------------------------------------------------
dim_date                       : 5844
dim_company                    : 3
dim_customer                   : 1961
dim_product                    : 1353
dim_budget_category            : 9
----------------------------------------------------------------------------------------------------
FACTS
----------------------------------------------------------------------------------------------------
fact_sales                     : 15817
fact_invoice                   : 13722
fact_payment                   : 3020
fact_inventory                 : 9007
fact_budget                    : 368
----------------------------------------------------------------------------------------------------
Fact row-count failures          : 0
Measure reconciliation failures  : 0
Aggregate reconciliation failures: 0
Foreign-key rules failed         : 0
GOLD BUSINESS

In [9]:
# =============================================================================
# CELL 9 — GOLD BUSINESS MODEL
# Write Gold Dimensions, Facts, and Aggregates
#
# Purpose:
# - Persist validated Gold tables into lh_global_finance_gold.
# - Use Delta format.
# - Overwrite deterministically for this portfolio Gold layer.
# - Validate every table after write.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. GOLD TARGET TABLE MAP
# =============================================================================

GOLD_TABLES = {

    # -------------------------------------------------------------------------
    # DIMENSIONS
    # -------------------------------------------------------------------------

    "dim_date":
        GOLD_DIM_DATE_DF,

    "dim_company":
        GOLD_DIM_COMPANY_DF,

    "dim_customer":
        GOLD_DIM_CUSTOMER_DF,

    "dim_product":
        GOLD_DIM_PRODUCT_DF,

    "dim_budget_category":
        GOLD_DIM_BUDGET_CATEGORY_DF,


    # -------------------------------------------------------------------------
    # FACTS
    # -------------------------------------------------------------------------

    "fact_sales":
        GOLD_FACT_SALES_DF,

    "fact_invoice":
        GOLD_FACT_INVOICE_DF,

    "fact_payment":
        GOLD_FACT_PAYMENT_DF,

    "fact_inventory":
        GOLD_FACT_INVENTORY_DF,

    "fact_budget":
        GOLD_FACT_BUDGET_DF,


    # -------------------------------------------------------------------------
    # BUSINESS AGGREGATES
    # -------------------------------------------------------------------------

    "agg_company_monthly_performance":
        AGG_COMPANY_MONTHLY_PERFORMANCE_DF,

    "agg_budget_vs_actual":
        AGG_BUDGET_VS_ACTUAL_DF,

    "agg_customer_performance":
        AGG_CUSTOMER_PERFORMANCE_DF,

    "agg_product_performance":
        AGG_PRODUCT_PERFORMANCE_DF,

    "agg_cash_collection":
        AGG_CASH_COLLECTION_DF,

    "agg_inventory_movement":
        AGG_INVENTORY_MOVEMENT_DF,
}


# =============================================================================
# 2. EXPECTED ROW COUNTS
# =============================================================================

EXPECTED_GOLD_ROW_COUNTS = {

    "dim_date":
        GOLD_DIM_DATE_DF.count(),

    "dim_company":
        GOLD_DIM_COMPANY_DF.count(),

    "dim_customer":
        GOLD_DIM_CUSTOMER_DF.count(),

    "dim_product":
        GOLD_DIM_PRODUCT_DF.count(),

    "dim_budget_category":
        GOLD_DIM_BUDGET_CATEGORY_DF.count(),


    "fact_sales":
        GOLD_FACT_SALES_DF.count(),

    "fact_invoice":
        GOLD_FACT_INVOICE_DF.count(),

    "fact_payment":
        GOLD_FACT_PAYMENT_DF.count(),

    "fact_inventory":
        GOLD_FACT_INVENTORY_DF.count(),

    "fact_budget":
        GOLD_FACT_BUDGET_DF.count(),


    "agg_company_monthly_performance":
        AGG_COMPANY_MONTHLY_PERFORMANCE_DF.count(),

    "agg_budget_vs_actual":
        AGG_BUDGET_VS_ACTUAL_DF.count(),

    "agg_customer_performance":
        AGG_CUSTOMER_PERFORMANCE_DF.count(),

    "agg_product_performance":
        AGG_PRODUCT_PERFORMANCE_DF.count(),

    "agg_cash_collection":
        AGG_CASH_COLLECTION_DF.count(),

    "agg_inventory_movement":
        AGG_INVENTORY_MOVEMENT_DF.count(),
}


# =============================================================================
# 3. WRITE TABLES
# =============================================================================

write_results = []


print("=" * 100)
print("GOLD BUSINESS MODEL — TABLE WRITE")
print("=" * 100)


for table_name, df in GOLD_TABLES.items():

    target_table = (
        f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.{table_name}"
    )

    expected_rows = (
        EXPECTED_GOLD_ROW_COUNTS[
            table_name
        ]
    )

    print(
        f"Writing {target_table} ..."
    )


    (
        df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            target_table
        )
    )


    # -------------------------------------------------------------------------
    # Validate written table
    # -------------------------------------------------------------------------

    written_df = (
        spark.table(
            target_table
        )
    )

    actual_rows = (
        written_df.count()
    )


    row_count_match = (
        actual_rows
        ==
        expected_rows
    )


    write_results.append({

        "table_name":
            table_name,

        "target_table":
            target_table,

        "expected_rows":
            expected_rows,

        "actual_rows":
            actual_rows,

        "row_count_match":
            row_count_match,

        "status":
            (
                "SUCCEEDED"
                if row_count_match
                else "FAILED"
            )
    })


# =============================================================================
# 4. PRINT WRITE RESULTS
# =============================================================================

print("=" * 100)
print("GOLD TABLE WRITE RESULTS")
print("=" * 100)


for result in write_results:

    print(
        f"{result['table_name']:<40} | "
        f"expected={result['expected_rows']:<8} | "
        f"actual={result['actual_rows']:<8} | "
        f"{result['status']}"
    )


print("=" * 100)


# =============================================================================
# 5. VALIDATE ALL WRITES
# =============================================================================

failed_writes = [
    result
    for result in write_results
    if result["status"] != "SUCCEEDED"
]


if failed_writes:

    raise RuntimeError(
        "GOLD TABLE WRITE FAILED FOR: "
        + ", ".join(
            [
                x["table_name"]
                for x in failed_writes
            ]
        )
    )


# =============================================================================
# 6. VERIFY TARGET TABLE DISCOVERY
# =============================================================================

gold_tables_discovered = [
    row["tableName"]
    for row in spark.sql(
        f"SHOW TABLES IN {GOLD_LAKEHOUSE}.{GOLD_SCHEMA}"
    ).collect()
    if not row["isTemporary"]
]


missing_written_tables = [
    table_name
    for table_name in GOLD_TABLES.keys()
    if table_name not in gold_tables_discovered
]


if missing_written_tables:

    raise RuntimeError(
        "Written Gold table(s) not discoverable: "
        + ", ".join(
            missing_written_tables
        )
    )


# =============================================================================
# 7. FINAL SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD BUSINESS MODEL — WRITE SUMMARY")
print("=" * 100)

print(
    f"Gold tables expected : "
    f"{len(GOLD_TABLES)}"
)

print(
    f"Gold tables written  : "
    f"{len(write_results)}"
)

print(
    f"Write failures       : "
    f"{len(failed_writes)}"
)

print(
    f"Missing after write  : "
    f"{len(missing_written_tables)}"
)

print("=" * 100)

print(
    "GOLD BUSINESS MODEL — TABLE WRITE: SUCCEEDED"
)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 11, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — TABLE WRITE
Writing lh_global_finance_gold.dbo.dim_date ...
Writing lh_global_finance_gold.dbo.dim_company ...
Writing lh_global_finance_gold.dbo.dim_customer ...
Writing lh_global_finance_gold.dbo.dim_product ...
Writing lh_global_finance_gold.dbo.dim_budget_category ...
Writing lh_global_finance_gold.dbo.fact_sales ...
Writing lh_global_finance_gold.dbo.fact_invoice ...
Writing lh_global_finance_gold.dbo.fact_payment ...
Writing lh_global_finance_gold.dbo.fact_inventory ...
Writing lh_global_finance_gold.dbo.fact_budget ...
Writing lh_global_finance_gold.dbo.agg_company_monthly_performance ...
Writing lh_global_finance_gold.dbo.agg_budget_vs_actual ...
Writing lh_global_finance_gold.dbo.agg_customer_performance ...
Writing lh_global_finance_gold.dbo.agg_product_performance ...
Writing lh_global_finance_gold.dbo.agg_cash_collection ...
Writing lh_global_finance_gold.dbo.agg_inventory_movement ...
GOLD TABLE WRITE RESULTS
dim_date                                 |

In [10]:
# =============================================================================
# CELL 10 — GOLD BUSINESS MODEL
# Post-Load Validation
#
# Purpose:
# - Validate all persisted Gold dimensions, facts, and aggregates.
# - Reconcile persisted row counts dynamically against the prepared Gold
#   DataFrames from this notebook.
# - Validate PK/FK integrity.
# - Validate referential integrity.
# - Reconcile core business measures.
# - Confirm Gold star-schema readiness for the semantic model / Power BI.
#
# Important:
# - No historical hard-coded row counts are used.
# - Legitimate incremental growth therefore does not create false failures.
# - A persisted table must still reconcile exactly to its prepared DataFrame.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. HELPER — RESOLVE REQUIRED UPSTREAM DATAFRAME
#
# Some aggregate/dimension DataFrame names may differ slightly between
# notebook versions. This helper resolves the first available candidate.
# =============================================================================

def resolve_runtime_dataframe(
    logical_name,
    candidate_names,
):
    """
    Return the first existing Spark DataFrame from candidate_names.

    Raises a clear error when none of the expected upstream DataFrames
    exists in the current notebook session.
    """

    for candidate_name in candidate_names:

        if candidate_name in globals():

            candidate_object = globals()[
                candidate_name
            ]

            if hasattr(
                candidate_object,
                "count",
            ):
                return candidate_object

    raise NameError(
        f"Unable to resolve the prepared DataFrame for "
        f"'{logical_name}'. "
        f"Checked candidates={candidate_names}. "
        "Run the preceding Gold Business Model cells first."
    )


# =============================================================================
# 2. RESOLVE PRE-WRITE GOLD DIMENSION DATAFRAMES
# =============================================================================

EXPECTED_DIM_DATE_DF = resolve_runtime_dataframe(
    "dim_date",
    [
        "GOLD_DIM_DATE_DF",
        "DIM_DATE_GOLD_DF",
    ],
)

EXPECTED_DIM_COMPANY_DF = resolve_runtime_dataframe(
    "dim_company",
    [
        "GOLD_DIM_COMPANY_DF",
        "DIM_COMPANY_GOLD_DF",
    ],
)

EXPECTED_DIM_CUSTOMER_DF = resolve_runtime_dataframe(
    "dim_customer",
    [
        "GOLD_DIM_CUSTOMER_DF",
        "DIM_CUSTOMER_GOLD_DF",
    ],
)

EXPECTED_DIM_PRODUCT_DF = resolve_runtime_dataframe(
    "dim_product",
    [
        "GOLD_DIM_PRODUCT_DF",
        "DIM_PRODUCT_GOLD_DF",
    ],
)

EXPECTED_DIM_BUDGET_CATEGORY_DF = (
    resolve_runtime_dataframe(
        "dim_budget_category",
        [
            "GOLD_DIM_BUDGET_CATEGORY_DF",
            "DIM_BUDGET_CATEGORY_GOLD_DF",
        ],
    )
)


# =============================================================================
# 3. RESOLVE PRE-WRITE GOLD FACT DATAFRAMES
# =============================================================================

EXPECTED_FACT_SALES_DF = resolve_runtime_dataframe(
    "fact_sales",
    [
        "GOLD_FACT_SALES_DF",
    ],
)

EXPECTED_FACT_INVOICE_DF = resolve_runtime_dataframe(
    "fact_invoice",
    [
        "GOLD_FACT_INVOICE_DF",
    ],
)

EXPECTED_FACT_PAYMENT_DF = resolve_runtime_dataframe(
    "fact_payment",
    [
        "GOLD_FACT_PAYMENT_DF",
    ],
)

EXPECTED_FACT_INVENTORY_DF = resolve_runtime_dataframe(
    "fact_inventory",
    [
        "GOLD_FACT_INVENTORY_DF",
    ],
)

EXPECTED_FACT_BUDGET_DF = resolve_runtime_dataframe(
    "fact_budget",
    [
        "GOLD_FACT_BUDGET_DF",
    ],
)


# =============================================================================
# 4. RESOLVE PRE-WRITE GOLD AGGREGATE DATAFRAMES
# =============================================================================

EXPECTED_AGG_COMPANY_MONTHLY_DF = (
    resolve_runtime_dataframe(
        "agg_company_monthly_performance",
        [
            "GOLD_AGG_COMPANY_MONTHLY_DF",
            "GOLD_AGG_COMPANY_MONTHLY_PERFORMANCE_DF",
            "AGG_COMPANY_MONTHLY_PERFORMANCE_DF",
        ],
    )
)

EXPECTED_AGG_BUDGET_ACTUAL_DF = (
    resolve_runtime_dataframe(
        "agg_budget_vs_actual",
        [
            "GOLD_AGG_BUDGET_ACTUAL_DF",
            "GOLD_AGG_BUDGET_VS_ACTUAL_DF",
            "AGG_BUDGET_VS_ACTUAL_DF",
        ],
    )
)

EXPECTED_AGG_CUSTOMER_DF = (
    resolve_runtime_dataframe(
        "agg_customer_performance",
        [
            "GOLD_AGG_CUSTOMER_DF",
            "GOLD_AGG_CUSTOMER_PERFORMANCE_DF",
            "AGG_CUSTOMER_PERFORMANCE_DF",
        ],
    )
)

EXPECTED_AGG_PRODUCT_DF = (
    resolve_runtime_dataframe(
        "agg_product_performance",
        [
            "GOLD_AGG_PRODUCT_DF",
            "GOLD_AGG_PRODUCT_PERFORMANCE_DF",
            "AGG_PRODUCT_PERFORMANCE_DF",
        ],
    )
)

EXPECTED_AGG_CASH_DF = (
    resolve_runtime_dataframe(
        "agg_cash_collection",
        [
            "GOLD_AGG_CASH_DF",
            "GOLD_AGG_CASH_COLLECTION_DF",
            "AGG_CASH_COLLECTION_DF",
        ],
    )
)

EXPECTED_AGG_INVENTORY_DF = (
    resolve_runtime_dataframe(
        "agg_inventory_movement",
        [
            "GOLD_AGG_INVENTORY_DF",
            "GOLD_AGG_INVENTORY_MOVEMENT_DF",
            "AGG_INVENTORY_MOVEMENT_DF",
        ],
    )
)


# =============================================================================
# 5. LOAD PERSISTED GOLD TABLES
# =============================================================================

GOLD_DIM_DATE_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.dim_date"
)

GOLD_DIM_COMPANY_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.dim_company"
)

GOLD_DIM_CUSTOMER_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.dim_customer"
)

GOLD_DIM_PRODUCT_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.dim_product"
)

GOLD_DIM_BUDGET_CATEGORY_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.dim_budget_category"
)


GOLD_FACT_SALES_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_sales"
)

GOLD_FACT_INVOICE_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_invoice"
)

GOLD_FACT_PAYMENT_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_payment"
)

GOLD_FACT_INVENTORY_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_inventory"
)

GOLD_FACT_BUDGET_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_budget"
)


GOLD_AGG_COMPANY_MONTHLY_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}."
    "agg_company_monthly_performance"
)

GOLD_AGG_BUDGET_ACTUAL_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}."
    "agg_budget_vs_actual"
)

GOLD_AGG_CUSTOMER_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}."
    "agg_customer_performance"
)

GOLD_AGG_PRODUCT_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}."
    "agg_product_performance"
)

GOLD_AGG_CASH_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}."
    "agg_cash_collection"
)

GOLD_AGG_INVENTORY_PERSISTED_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}."
    "agg_inventory_movement"
)


# =============================================================================
# 6. DYNAMIC EXPECTED ROW COUNTS
#
# Expected = DataFrame prepared earlier in this notebook.
# Actual   = persisted Gold Delta table after write.
# =============================================================================

EXPECTED_COUNTS = {

    "dim_date":
        EXPECTED_DIM_DATE_DF.count(),

    "dim_company":
        EXPECTED_DIM_COMPANY_DF.count(),

    "dim_customer":
        EXPECTED_DIM_CUSTOMER_DF.count(),

    "dim_product":
        EXPECTED_DIM_PRODUCT_DF.count(),

    "dim_budget_category":
        EXPECTED_DIM_BUDGET_CATEGORY_DF.count(),


    "fact_sales":
        EXPECTED_FACT_SALES_DF.count(),

    "fact_invoice":
        EXPECTED_FACT_INVOICE_DF.count(),

    "fact_payment":
        EXPECTED_FACT_PAYMENT_DF.count(),

    "fact_inventory":
        EXPECTED_FACT_INVENTORY_DF.count(),

    "fact_budget":
        EXPECTED_FACT_BUDGET_DF.count(),


    "agg_company_monthly_performance":
        EXPECTED_AGG_COMPANY_MONTHLY_DF.count(),

    "agg_budget_vs_actual":
        EXPECTED_AGG_BUDGET_ACTUAL_DF.count(),

    "agg_customer_performance":
        EXPECTED_AGG_CUSTOMER_DF.count(),

    "agg_product_performance":
        EXPECTED_AGG_PRODUCT_DF.count(),

    "agg_cash_collection":
        EXPECTED_AGG_CASH_DF.count(),

    "agg_inventory_movement":
        EXPECTED_AGG_INVENTORY_DF.count(),
}


# =============================================================================
# 7. ACTUAL PERSISTED ROW COUNTS
# =============================================================================

ACTUAL_COUNTS = {

    "dim_date":
        GOLD_DIM_DATE_PERSISTED_DF.count(),

    "dim_company":
        GOLD_DIM_COMPANY_PERSISTED_DF.count(),

    "dim_customer":
        GOLD_DIM_CUSTOMER_PERSISTED_DF.count(),

    "dim_product":
        GOLD_DIM_PRODUCT_PERSISTED_DF.count(),

    "dim_budget_category":
        GOLD_DIM_BUDGET_CATEGORY_PERSISTED_DF.count(),


    "fact_sales":
        GOLD_FACT_SALES_PERSISTED_DF.count(),

    "fact_invoice":
        GOLD_FACT_INVOICE_PERSISTED_DF.count(),

    "fact_payment":
        GOLD_FACT_PAYMENT_PERSISTED_DF.count(),

    "fact_inventory":
        GOLD_FACT_INVENTORY_PERSISTED_DF.count(),

    "fact_budget":
        GOLD_FACT_BUDGET_PERSISTED_DF.count(),


    "agg_company_monthly_performance":
        GOLD_AGG_COMPANY_MONTHLY_PERSISTED_DF.count(),

    "agg_budget_vs_actual":
        GOLD_AGG_BUDGET_ACTUAL_PERSISTED_DF.count(),

    "agg_customer_performance":
        GOLD_AGG_CUSTOMER_PERSISTED_DF.count(),

    "agg_product_performance":
        GOLD_AGG_PRODUCT_PERSISTED_DF.count(),

    "agg_cash_collection":
        GOLD_AGG_CASH_PERSISTED_DF.count(),

    "agg_inventory_movement":
        GOLD_AGG_INVENTORY_PERSISTED_DF.count(),
}


# =============================================================================
# 8. ROW-COUNT RECONCILIATION
# =============================================================================

row_count_failures = {}

for (
    table_name,
    expected_count
) in EXPECTED_COUNTS.items():

    actual_count = ACTUAL_COUNTS[
        table_name
    ]

    if actual_count != expected_count:

        row_count_failures[
            table_name
        ] = {
            "expected":
                expected_count,

            "actual":
                actual_count,
        }


# =============================================================================
# 9. HELPER — DUPLICATE KEY COUNT
# =============================================================================

def duplicate_count(
    df,
    key_cols,
):

    return (
        df

        .groupBy(
            *key_cols
        )

        .count()

        .filter(
            F.col("count") > 1
        )

        .count()
    )


# =============================================================================
# 10. DIMENSION PRIMARY-KEY VALIDATION
# =============================================================================

dimension_key_failures = {

    "dim_date":
        duplicate_count(
            GOLD_DIM_DATE_PERSISTED_DF,
            [
                "date_key"
            ],
        ),

    "dim_company":
        duplicate_count(
            GOLD_DIM_COMPANY_PERSISTED_DF,
            [
                "company_key"
            ],
        ),

    "dim_customer":
        duplicate_count(
            GOLD_DIM_CUSTOMER_PERSISTED_DF,
            [
                "customer_key"
            ],
        ),

    "dim_product":
        duplicate_count(
            GOLD_DIM_PRODUCT_PERSISTED_DF,
            [
                "product_key"
            ],
        ),

    "dim_budget_category":
        duplicate_count(
            GOLD_DIM_BUDGET_CATEGORY_PERSISTED_DF,
            [
                "budget_category_key"
            ],
        ),
}


# =============================================================================
# 11. FACT PRIMARY-KEY VALIDATION
# =============================================================================

fact_key_failures = {

    "fact_sales":
        duplicate_count(
            GOLD_FACT_SALES_PERSISTED_DF,
            [
                "sales_order_line_key"
            ],
        ),

    "fact_invoice":
        duplicate_count(
            GOLD_FACT_INVOICE_PERSISTED_DF,
            [
                "invoice_line_key"
            ],
        ),

    "fact_payment":
        duplicate_count(
            GOLD_FACT_PAYMENT_PERSISTED_DF,
            [
                "payment_key"
            ],
        ),

    "fact_inventory":
        duplicate_count(
            GOLD_FACT_INVENTORY_PERSISTED_DF,
            [
                "inventory_transaction_key"
            ],
        ),

    "fact_budget":
        duplicate_count(
            GOLD_FACT_BUDGET_PERSISTED_DF,
            [
                "monthly_budget_key"
            ],
        ),
}


# =============================================================================
# 12. REQUIRED FOREIGN-KEY NULL VALIDATION
# =============================================================================

fk_null_failures = {

    "sales_company":
        GOLD_FACT_SALES_PERSISTED_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "sales_customer":
        GOLD_FACT_SALES_PERSISTED_DF
        .filter(
            F.col(
                "customer_key"
            ).isNull()
        )
        .count(),

    "sales_product":
        GOLD_FACT_SALES_PERSISTED_DF
        .filter(
            F.col(
                "product_key"
            ).isNull()
        )
        .count(),

    "sales_order_date":
        GOLD_FACT_SALES_PERSISTED_DF
        .filter(
            F.col(
                "order_date_key"
            ).isNull()
        )
        .count(),


    "invoice_company":
        GOLD_FACT_INVOICE_PERSISTED_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "invoice_customer":
        GOLD_FACT_INVOICE_PERSISTED_DF
        .filter(
            F.col(
                "customer_key"
            ).isNull()
        )
        .count(),

    "invoice_product":
        GOLD_FACT_INVOICE_PERSISTED_DF
        .filter(
            F.col(
                "product_key"
            ).isNull()
        )
        .count(),

    "invoice_date":
        GOLD_FACT_INVOICE_PERSISTED_DF
        .filter(
            F.col(
                "invoice_date_key"
            ).isNull()
        )
        .count(),


    "payment_company":
        GOLD_FACT_PAYMENT_PERSISTED_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "payment_customer":
        GOLD_FACT_PAYMENT_PERSISTED_DF
        .filter(
            F.col(
                "customer_key"
            ).isNull()
        )
        .count(),

    "payment_date":
        GOLD_FACT_PAYMENT_PERSISTED_DF
        .filter(
            F.col(
                "payment_date_key"
            ).isNull()
        )
        .count(),


    "inventory_company":
        GOLD_FACT_INVENTORY_PERSISTED_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "inventory_product":
        GOLD_FACT_INVENTORY_PERSISTED_DF
        .filter(
            F.col(
                "product_key"
            ).isNull()
        )
        .count(),

    "inventory_date":
        GOLD_FACT_INVENTORY_PERSISTED_DF
        .filter(
            F.col(
                "transaction_date_key"
            ).isNull()
        )
        .count(),


    "budget_company":
        GOLD_FACT_BUDGET_PERSISTED_DF
        .filter(
            F.col(
                "company_key"
            ).isNull()
        )
        .count(),

    "budget_date":
        GOLD_FACT_BUDGET_PERSISTED_DF
        .filter(
            F.col(
                "budget_month_date_key"
            ).isNull()
        )
        .count(),

    "budget_category":
        GOLD_FACT_BUDGET_PERSISTED_DF
        .filter(
            F.col(
                "budget_category_key"
            ).isNull()
        )
        .count(),
}


# =============================================================================
# 13. OPTIONAL SALES DELIVERY-DATE DIAGNOSTIC
# =============================================================================

optional_delivery_date_nulls = (
    GOLD_FACT_SALES_PERSISTED_DF

    .filter(
        F.col(
            "requested_delivery_date_key"
        ).isNull()
    )

    .count()
)


# =============================================================================
# 14. REFERENTIAL INTEGRITY — SALES
# =============================================================================

sales_orphan_company = (
    GOLD_FACT_SALES_PERSISTED_DF

    .select(
        "company_key"
    )

    .filter(
        F.col(
            "company_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_COMPANY_PERSISTED_DF
        .select(
            "company_key"
        )
        .distinct(),

        on="company_key",

        how="left_anti",
    )

    .count()
)


sales_orphan_customer = (
    GOLD_FACT_SALES_PERSISTED_DF

    .select(
        "customer_key"
    )

    .filter(
        F.col(
            "customer_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_CUSTOMER_PERSISTED_DF
        .select(
            "customer_key"
        )
        .distinct(),

        on="customer_key",

        how="left_anti",
    )

    .count()
)


sales_orphan_product = (
    GOLD_FACT_SALES_PERSISTED_DF

    .select(
        "product_key"
    )

    .filter(
        F.col(
            "product_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_PRODUCT_PERSISTED_DF
        .select(
            "product_key"
        )
        .distinct(),

        on="product_key",

        how="left_anti",
    )

    .count()
)


# =============================================================================
# 15. REFERENTIAL INTEGRITY — INVOICE
# =============================================================================

invoice_orphan_company = (
    GOLD_FACT_INVOICE_PERSISTED_DF

    .select(
        "company_key"
    )

    .filter(
        F.col(
            "company_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_COMPANY_PERSISTED_DF
        .select(
            "company_key"
        )
        .distinct(),

        on="company_key",

        how="left_anti",
    )

    .count()
)


invoice_orphan_customer = (
    GOLD_FACT_INVOICE_PERSISTED_DF

    .select(
        "customer_key"
    )

    .filter(
        F.col(
            "customer_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_CUSTOMER_PERSISTED_DF
        .select(
            "customer_key"
        )
        .distinct(),

        on="customer_key",

        how="left_anti",
    )

    .count()
)


invoice_orphan_product = (
    GOLD_FACT_INVOICE_PERSISTED_DF

    .select(
        "product_key"
    )

    .filter(
        F.col(
            "product_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_PRODUCT_PERSISTED_DF
        .select(
            "product_key"
        )
        .distinct(),

        on="product_key",

        how="left_anti",
    )

    .count()
)


# =============================================================================
# 16. REFERENTIAL INTEGRITY — PAYMENT
# =============================================================================

payment_orphan_company = (
    GOLD_FACT_PAYMENT_PERSISTED_DF

    .select(
        "company_key"
    )

    .filter(
        F.col(
            "company_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_COMPANY_PERSISTED_DF
        .select(
            "company_key"
        )
        .distinct(),

        on="company_key",

        how="left_anti",
    )

    .count()
)


payment_orphan_customer = (
    GOLD_FACT_PAYMENT_PERSISTED_DF

    .select(
        "customer_key"
    )

    .filter(
        F.col(
            "customer_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_CUSTOMER_PERSISTED_DF
        .select(
            "customer_key"
        )
        .distinct(),

        on="customer_key",

        how="left_anti",
    )

    .count()
)


# =============================================================================
# 17. REFERENTIAL INTEGRITY — INVENTORY
# =============================================================================

inventory_orphan_company = (
    GOLD_FACT_INVENTORY_PERSISTED_DF

    .select(
        "company_key"
    )

    .filter(
        F.col(
            "company_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_COMPANY_PERSISTED_DF
        .select(
            "company_key"
        )
        .distinct(),

        on="company_key",

        how="left_anti",
    )

    .count()
)


inventory_orphan_product = (
    GOLD_FACT_INVENTORY_PERSISTED_DF

    .select(
        "product_key"
    )

    .filter(
        F.col(
            "product_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_PRODUCT_PERSISTED_DF
        .select(
            "product_key"
        )
        .distinct(),

        on="product_key",

        how="left_anti",
    )

    .count()
)


# =============================================================================
# 18. REFERENTIAL INTEGRITY — BUDGET
# =============================================================================

budget_orphan_company = (
    GOLD_FACT_BUDGET_PERSISTED_DF

    .select(
        "company_key"
    )

    .filter(
        F.col(
            "company_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_COMPANY_PERSISTED_DF
        .select(
            "company_key"
        )
        .distinct(),

        on="company_key",

        how="left_anti",
    )

    .count()
)


budget_orphan_category = (
    GOLD_FACT_BUDGET_PERSISTED_DF

    .select(
        "budget_category_key"
    )

    .filter(
        F.col(
            "budget_category_key"
        ).isNotNull()
    )

    .distinct()

    .join(
        GOLD_DIM_BUDGET_CATEGORY_PERSISTED_DF
        .select(
            "budget_category_key"
        )
        .distinct(),

        on="budget_category_key",

        how="left_anti",
    )

    .count()
)


# =============================================================================
# 19. CORE PERSISTED BUSINESS-MEASURE TOTALS
# =============================================================================

persisted_invoice_revenue = (
    GOLD_FACT_INVOICE_PERSISTED_DF

    .agg(
        F.sum(
            "net_amount"
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


persisted_company_revenue = (
    GOLD_AGG_COMPANY_MONTHLY_PERSISTED_DF

    .agg(
        F.sum(
            "actual_revenue"
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


persisted_payment_total = (
    GOLD_FACT_PAYMENT_PERSISTED_DF

    .agg(
        F.sum(
            "payment_amount"
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


persisted_company_cash = (
    GOLD_AGG_COMPANY_MONTHLY_PERSISTED_DF

    .agg(
        F.sum(
            "cash_collected"
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


persisted_budget_total = (
    GOLD_FACT_BUDGET_PERSISTED_DF

    .agg(
        F.sum(
            "revenue_budget"
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


persisted_budget_agg_total = (
    GOLD_AGG_BUDGET_ACTUAL_PERSISTED_DF

    .agg(
        F.sum(
            "budget_revenue"
        ).alias(
            "amount"
        )
    )

    .first()[
        "amount"
    ]
)


# =============================================================================
# 20. SAFE AMOUNT COMPARISON
# =============================================================================

def amount_mismatch(
    a,
    b,
    tolerance=0.01,
):

    return (
        abs(
            float(
                a or 0
            )
            -
            float(
                b or 0
            )
        )
        >
        tolerance
    )


# =============================================================================
# 21. AGGREGATE RECONCILIATION
# =============================================================================

aggregate_failures = []


if amount_mismatch(
    persisted_invoice_revenue,
    persisted_company_revenue,
):

    aggregate_failures.append(
        "INVOICE_REVENUE_RECONCILIATION"
    )


if amount_mismatch(
    persisted_payment_total,
    persisted_company_cash,
):

    aggregate_failures.append(
        "PAYMENT_RECONCILIATION"
    )


if amount_mismatch(
    persisted_budget_total,
    persisted_budget_agg_total,
):

    aggregate_failures.append(
        "BUDGET_RECONCILIATION"
    )


# =============================================================================
# 22. BUILD CRITICAL FAILURES
# =============================================================================

critical_failures = []


if row_count_failures:

    critical_failures.append(
        "ROW_COUNT_VALIDATION"
    )


if any(
    value > 0
    for value
    in dimension_key_failures.values()
):

    critical_failures.append(
        "DIMENSION_DUPLICATE_KEYS"
    )


if any(
    value > 0
    for value
    in fact_key_failures.values()
):

    critical_failures.append(
        "FACT_DUPLICATE_KEYS"
    )


if any(
    value > 0
    for value
    in fk_null_failures.values()
):

    critical_failures.append(
        "NULL_REQUIRED_FOREIGN_KEYS"
    )


orphan_total = (
    sales_orphan_company
    +
    sales_orphan_customer
    +
    sales_orphan_product
    +
    invoice_orphan_company
    +
    invoice_orphan_customer
    +
    invoice_orphan_product
    +
    payment_orphan_company
    +
    payment_orphan_customer
    +
    inventory_orphan_company
    +
    inventory_orphan_product
    +
    budget_orphan_company
    +
    budget_orphan_category
)


if orphan_total > 0:

    critical_failures.append(
        "ORPHAN_FOREIGN_KEYS"
    )


if aggregate_failures:

    critical_failures.append(
        "AGGREGATE_RECONCILIATION"
    )


# =============================================================================
# 23. PRINT POST-LOAD SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD BUSINESS MODEL — POST-LOAD VALIDATION")
print("=" * 100)

print(
    f"Gold tables validated            : "
    f"{len(EXPECTED_COUNTS)}"
)

print(
    f"Row-count failures               : "
    f"{len(row_count_failures)}"
)

print(
    f"Dimension duplicate-key failures : "
    f"{sum(dimension_key_failures.values())}"
)

print(
    f"Fact duplicate-key failures      : "
    f"{sum(fact_key_failures.values())}"
)

print(
    f"Required FK null failures        : "
    f"{sum(fk_null_failures.values())}"
)

print(
    f"Orphan FK failures               : "
    f"{orphan_total}"
)

print(
    f"Aggregate reconciliation failures: "
    f"{len(aggregate_failures)}"
)

print(
    f"Optional delivery-date nulls     : "
    f"{optional_delivery_date_nulls}"
)

print(
    f"Critical validation failures     : "
    f"{len(critical_failures)}"
)

print("=" * 100)


# =============================================================================
# 24. DISPLAY DYNAMIC ROW-COUNT RECONCILIATION
# =============================================================================

for table_name in EXPECTED_COUNTS:

    status = (
        "PASSED"
        if (
            EXPECTED_COUNTS[
                table_name
            ]
            ==
            ACTUAL_COUNTS[
                table_name
            ]
        )
        else "FAILED"
    )

    print(
        f"{table_name:<40} | "
        f"expected={EXPECTED_COUNTS[table_name]:<8} | "
        f"actual={ACTUAL_COUNTS[table_name]:<8} | "
        f"status={status}"
    )


# =============================================================================
# 25. DISPLAY ROW-COUNT FAILURES IF PRESENT
# =============================================================================

if row_count_failures:

    print("=" * 100)
    print("GOLD ROW-COUNT RECONCILIATION FAILURES")
    print("=" * 100)

    for (
        table_name,
        result
    ) in row_count_failures.items():

        print(
            f"{table_name:<40} | "
            f"expected={result['expected']:<8} | "
            f"actual={result['actual']}"
        )

    print("=" * 100)


# =============================================================================
# 26. DISPLAY AGGREGATE FAILURES IF PRESENT
# =============================================================================

if aggregate_failures:

    print("=" * 100)
    print("GOLD AGGREGATE RECONCILIATION FAILURES")
    print("=" * 100)

    for failure_name in aggregate_failures:

        print(
            failure_name
        )

    print("=" * 100)


# =============================================================================
# 27. FAIL FAST
# =============================================================================

if critical_failures:

    raise RuntimeError(
        "GOLD POST-LOAD VALIDATION FAILED: "
        + ", ".join(
            critical_failures
        )
    )


# =============================================================================
# 28. FINAL STATUS
# =============================================================================

print("=" * 100)

print(
    "GOLD BUSINESS MODEL — POST-LOAD VALIDATION: SUCCEEDED"
)

print("=" * 100)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 12, Finished, Available, Finished, False)

GOLD BUSINESS MODEL — POST-LOAD VALIDATION
Gold tables validated            : 16
Row-count failures               : 0
Dimension duplicate-key failures : 0
Fact duplicate-key failures      : 0
Required FK null failures        : 0
Orphan FK failures               : 0
Aggregate reconciliation failures: 0
Optional delivery-date nulls     : 0
Critical validation failures     : 0
dim_date                                 | expected=5844     | actual=5844     | status=PASSED
dim_company                              | expected=3        | actual=3        | status=PASSED
dim_customer                             | expected=1961     | actual=1961     | status=PASSED
dim_product                              | expected=1353     | actual=1353     | status=PASSED
dim_budget_category                      | expected=9        | actual=9        | status=PASSED
fact_sales                               | expected=15817    | actual=15817    | status=PASSED
fact_invoice                             | expected=1

**GOLD OPERATIONAL MONITORING MODEL**

In [11]:
# =============================================================================
# CELL 11 — GOLD OPERATIONAL MONITORING MODEL
# Audit / Monitoring Source Discovery
#
# Purpose:
# - Discover operational metadata already available in Bronze, Silver and Gold.
# - Identify pipeline audit, reconciliation, DQ and load-monitoring tables.
# - Avoid fabricating monitoring information.
# - Run independently without relying on lakehouse variables from earlier cells.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    StructField,
    StructType,
)


# =============================================================================
# 1. ENVIRONMENT CONFIGURATION
# =============================================================================

WORKSPACE_NAME = "Global-Finance-Analytics-DEV"

BRONZE_LAKEHOUSE = "lh_global_finance_bronze"
BRONZE_SCHEMA = "dbo"

SILVER_LAKEHOUSE = "lh_global_finance_silver"
SILVER_SCHEMA = "dbo"

GOLD_LAKEHOUSE = "lh_global_finance_gold"
GOLD_SCHEMA = "dbo"


# =============================================================================
# 2. HELPER — FULLY QUALIFIED SCHEMA NAME
# =============================================================================

def qualified_schema(
    lakehouse_name: str,
    schema_name: str,
) -> str:
    """
    Return a fully qualified Fabric lakehouse schema identifier.

    Backticks are used because the workspace name contains hyphens.
    """

    return (
        f"`{WORKSPACE_NAME}`."
        f"`{lakehouse_name}`."
        f"`{schema_name}`"
    )


# =============================================================================
# 3. MONITORING SOURCE CONFIGURATION
# =============================================================================

MONITORING_SOURCE_SCHEMAS = [

    (
        "BRONZE",
        BRONZE_LAKEHOUSE,
        BRONZE_SCHEMA,
    ),

    (
        "SILVER",
        SILVER_LAKEHOUSE,
        SILVER_SCHEMA,
    ),

    (
        "GOLD",
        GOLD_LAKEHOUSE,
        GOLD_SCHEMA,
    ),
]


# =============================================================================
# 4. EXPLICIT DISCOVERY RESULT SCHEMA
# =============================================================================

discovered_objects_schema = StructType([

    StructField(
        "data_layer",
        StringType(),
        False,
    ),

    StructField(
        "lakehouse",
        StringType(),
        False,
    ),

    StructField(
        "schema_name",
        StringType(),
        False,
    ),

    StructField(
        "table_name",
        StringType(),
        False,
    ),
])


# =============================================================================
# 5. DISCOVER TABLES
# =============================================================================

discovered_objects = []

discovery_failures = []


for (
    layer,
    lakehouse,
    schema,
) in MONITORING_SOURCE_SCHEMAS:

    schema_identifier = qualified_schema(
        lakehouse,
        schema,
    )

    try:

        tables = (
            spark.sql(
                f"SHOW TABLES IN {schema_identifier}"
            )
            .collect()
        )

        for row in tables:

            if not row["isTemporary"]:

                table_name = row["tableName"]

                discovered_objects.append(
                    (
                        layer,
                        lakehouse,
                        schema,
                        table_name,
                    )
                )

    except Exception as exc:

        discovery_failures.append(
            {
                "data_layer": layer,
                "lakehouse": lakehouse,
                "schema_name": schema,
                "error_message": str(exc)[:2000],
            }
        )


# =============================================================================
# 6. FAIL IF ANY REQUIRED LAYER COULD NOT BE DISCOVERED
# =============================================================================

if discovery_failures:

    print("=" * 100)
    print("GOLD OPERATIONAL MONITORING — DISCOVERY FAILURES")
    print("=" * 100)

    for failure in discovery_failures:

        print(
            f"Layer={failure['data_layer']} | "
            f"Lakehouse={failure['lakehouse']} | "
            f"Schema={failure['schema_name']}"
        )

        print(
            f"Error={failure['error_message']}"
        )

        print("-" * 100)

    raise RuntimeError(
        "Operational monitoring source discovery failed for "
        f"{len(discovery_failures)} configured data layer(s)."
    )


# =============================================================================
# 7. CREATE DISCOVERY DATAFRAME
# =============================================================================

DISCOVERED_OBJECTS_DF = spark.createDataFrame(
    discovered_objects,
    schema=discovered_objects_schema,
)


# =============================================================================
# 8. IDENTIFY LIKELY MONITORING OBJECTS
# =============================================================================

MONITORING_CANDIDATES_DF = (
    DISCOVERED_OBJECTS_DF

    .withColumn(
        "table_name_upper",

        F.upper(
            F.col(
                "table_name"
            )
        ),
    )

    .filter(

        F.col(
            "table_name_upper"
        ).contains(
            "AUDIT"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "LOG"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "MONITOR"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "PIPELINE"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "RUN"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "LOAD"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "ERROR"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "REJECT"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "QUALITY"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "RECON"
        )

        |

        F.col(
            "table_name_upper"
        ).contains(
            "CONTROL"
        )
    )

    .drop(
        "table_name_upper"
    )

    .orderBy(
        "data_layer",
        "table_name",
    )
)


# =============================================================================
# 9. BUILD LAYER SUMMARY
# =============================================================================

DISCOVERY_LAYER_SUMMARY_DF = (
    DISCOVERED_OBJECTS_DF

    .groupBy(
        "data_layer"
    )

    .agg(
        F.count(
            "*"
        ).alias(
            "object_count"
        )
    )

    .orderBy(
        "data_layer"
    )
)


# =============================================================================
# 10. BUILD MONITORING-CANDIDATE SUMMARY
# =============================================================================

MONITORING_LAYER_SUMMARY_DF = (
    MONITORING_CANDIDATES_DF

    .groupBy(
        "data_layer"
    )

    .agg(
        F.count(
            "*"
        ).alias(
            "monitoring_candidate_count"
        )
    )

    .orderBy(
        "data_layer"
    )
)


# =============================================================================
# 11. OVERALL SUMMARY
# =============================================================================

total_objects = (
    DISCOVERED_OBJECTS_DF.count()
)

monitoring_candidates = (
    MONITORING_CANDIDATES_DF.count()
)


print("=" * 100)
print("GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY")
print("=" * 100)

print(
    f"Workspace                                 : "
    f"{WORKSPACE_NAME}"
)

print(
    f"Bronze source                             : "
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}"
)

print(
    f"Silver source                             : "
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}"
)

print(
    f"Gold source                               : "
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}"
)

print(
    f"Total Bronze/Silver/Gold objects discovered : "
    f"{total_objects}"
)

print(
    f"Potential monitoring/audit objects          : "
    f"{monitoring_candidates}"
)

print("=" * 100)


# =============================================================================
# 12. DISPLAY LAYER SUMMARY
# =============================================================================

print("=" * 100)
print("DATA PLATFORM OBJECT SUMMARY")
print("=" * 100)

display(
    DISCOVERY_LAYER_SUMMARY_DF
)


# =============================================================================
# 13. DISPLAY MONITORING-CANDIDATE SUMMARY
# =============================================================================

print("=" * 100)
print("MONITORING CANDIDATE SUMMARY")
print("=" * 100)

display(
    MONITORING_LAYER_SUMMARY_DF
)


# =============================================================================
# 14. DISPLAY MONITORING CANDIDATES
# =============================================================================

print("=" * 100)
print("MONITORING / AUDIT TABLE CANDIDATES")
print("=" * 100)

display(
    MONITORING_CANDIDATES_DF
)


# =============================================================================
# 15. DISPLAY ALL OBJECTS FOR TRACEABILITY
# =============================================================================

print("=" * 100)
print("ALL DATA PLATFORM OBJECTS")
print("=" * 100)

display(
    DISCOVERED_OBJECTS_DF

    .orderBy(
        "data_layer",
        "table_name",
    )
)


# =============================================================================
# 16. VALIDATE DISCOVERY COMPLETENESS
# =============================================================================

discovered_layer_count = (
    DISCOVERED_OBJECTS_DF

    .select(
        "data_layer"
    )

    .distinct()

    .count()
)


expected_layer_count = len(
    MONITORING_SOURCE_SCHEMAS
)


if discovered_layer_count != expected_layer_count:

    raise RuntimeError(
        "Operational monitoring discovery did not return objects "
        "from every configured data layer. "
        f"Expected layers={expected_layer_count}; "
        f"discovered layers={discovered_layer_count}."
    )


if total_objects == 0:

    raise RuntimeError(
        "Operational monitoring discovery returned no Bronze, "
        "Silver or Gold objects."
    )


# =============================================================================
# 17. STATUS
# =============================================================================

print("=" * 100)

print(
    "GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY: SUCCEEDED"
)

print("=" * 100)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 13, Finished, Available, Finished, False)

GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY
Workspace                                 : Global-Finance-Analytics-DEV
Bronze source                             : lh_global_finance_bronze.dbo
Silver source                             : lh_global_finance_silver.dbo
Gold source                               : lh_global_finance_gold.dbo
Total Bronze/Silver/Gold objects discovered : 143
Potential monitoring/audit objects          : 8
DATA PLATFORM OBJECT SUMMARY


SynapseWidget(Synapse.DataFrame, 35864771-99a6-4ee6-bdbe-92cfa50a222c)

MONITORING CANDIDATE SUMMARY


SynapseWidget(Synapse.DataFrame, 0c75d677-f657-4858-b823-5a9f7c06bd7b)

MONITORING / AUDIT TABLE CANDIDATES


SynapseWidget(Synapse.DataFrame, 7fdac85a-40c1-4a2b-9b5d-1a4ea30d7612)

ALL DATA PLATFORM OBJECTS


SynapseWidget(Synapse.DataFrame, 65a1b548-7c92-4628-af10-0c50edf22868)

GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY: SUCCEEDED


In [12]:
# =============================================================================
# CELL 11 — GOLD OPERATIONAL MONITORING MODEL
# Audit / Monitoring Source Discovery
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. DEFINE ALL LAKEHOUSES EXPLICITLY
# =============================================================================

BRONZE_LAKEHOUSE = "lh_global_finance_bronze"
SILVER_LAKEHOUSE = "lh_global_finance_silver"
GOLD_LAKEHOUSE   = "lh_global_finance_gold"

BRONZE_SCHEMA = "dbo"
SILVER_SCHEMA = "dbo"
GOLD_SCHEMA   = "dbo"


# =============================================================================
# 2. MONITORING SOURCE CONFIGURATION
# =============================================================================

MONITORING_SOURCE_SCHEMAS = [

    (
        "BRONZE",
        BRONZE_LAKEHOUSE,
        BRONZE_SCHEMA
    ),

    (
        "SILVER",
        SILVER_LAKEHOUSE,
        SILVER_SCHEMA
    ),

    (
        "GOLD",
        GOLD_LAKEHOUSE,
        GOLD_SCHEMA
    )
]


# =============================================================================
# 3. DISCOVER TABLES
# =============================================================================

discovered_objects = []


for layer, lakehouse, schema in MONITORING_SOURCE_SCHEMAS:

    tables = spark.sql(
        f"SHOW TABLES IN {lakehouse}.{schema}"
    ).collect()

    for row in tables:

        if not row["isTemporary"]:

            discovered_objects.append(
                (
                    layer,
                    lakehouse,
                    schema,
                    row["tableName"]
                )
            )


# =============================================================================
# 4. CREATE DISCOVERY DATAFRAME
# =============================================================================

DISCOVERED_OBJECTS_DF = spark.createDataFrame(

    discovered_objects,

    [
        "data_layer",
        "lakehouse",
        "schema_name",
        "table_name"
    ]
)


# =============================================================================
# 5. FIND MONITORING / AUDIT CANDIDATES
# =============================================================================

MONITORING_CANDIDATES_DF = (

    DISCOVERED_OBJECTS_DF

    .withColumn(
        "_table_upper",
        F.upper(
            F.col("table_name")
        )
    )

    .filter(

        F.col("_table_upper").contains("AUDIT")
        |
        F.col("_table_upper").contains("LOG")
        |
        F.col("_table_upper").contains("MONITOR")
        |
        F.col("_table_upper").contains("PIPELINE")
        |
        F.col("_table_upper").contains("RUN")
        |
        F.col("_table_upper").contains("LOAD")
        |
        F.col("_table_upper").contains("ERROR")
        |
        F.col("_table_upper").contains("REJECT")
        |
        F.col("_table_upper").contains("QUALITY")
        |
        F.col("_table_upper").contains("RECON")
        |
        F.col("_table_upper").contains("CONTROL")
    )

    .drop(
        "_table_upper"
    )

    .orderBy(
        "data_layer",
        "table_name"
    )
)


# =============================================================================
# 6. COUNTS
# =============================================================================

total_objects = (
    DISCOVERED_OBJECTS_DF.count()
)

monitoring_candidate_count = (
    MONITORING_CANDIDATES_DF.count()
)


# =============================================================================
# 7. SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY")
print("=" * 100)

print(
    f"Total Bronze/Silver/Gold objects discovered : "
    f"{total_objects}"
)

print(
    f"Potential monitoring/audit objects          : "
    f"{monitoring_candidate_count}"
)

print("=" * 100)


# =============================================================================
# 8. DISPLAY MONITORING CANDIDATES
# =============================================================================

print("=" * 100)
print("MONITORING / AUDIT TABLE CANDIDATES")
print("=" * 100)

display(
    MONITORING_CANDIDATES_DF
)


# =============================================================================
# 9. DISPLAY ALL OBJECTS
# =============================================================================

print("=" * 100)
print("ALL DATA PLATFORM OBJECTS")
print("=" * 100)

display(
    DISCOVERED_OBJECTS_DF
    .orderBy(
        "data_layer",
        "table_name"
    )
)


# =============================================================================
# 10. FINAL STATUS
# =============================================================================

print("=" * 100)

print(
    "GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY: SUCCEEDED"
)

print("=" * 100)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 14, Finished, Available, Finished, False)

GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY
Total Bronze/Silver/Gold objects discovered : 143
Potential monitoring/audit objects          : 8
MONITORING / AUDIT TABLE CANDIDATES


SynapseWidget(Synapse.DataFrame, 4ad0e75a-e514-43fc-831a-17bb7960978a)

ALL DATA PLATFORM OBJECTS


SynapseWidget(Synapse.DataFrame, 1a260379-f798-44d7-9612-8cf3254fe567)

GOLD OPERATIONAL MONITORING — SOURCE DISCOVERY: SUCCEEDED


In [13]:
# =============================================================================
# CELL 12 — GOLD OPERATIONAL MONITORING MODEL
# Audit / Monitoring Source Profiling
#
# Purpose:
# - Profile the four real operational audit tables.
# - Inspect schema, row counts, nulls, date/timestamp coverage and samples.
# - Determine the correct monitoring fact design from actual metadata.
#
# No Gold monitoring tables are written in this cell.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DateType, TimestampType


# =============================================================================
# 1. DEFINE MONITORING SOURCE TABLES
# =============================================================================

MONITORING_SOURCE_TABLES = {

    "qbo_es_cdc_change_log":
        f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_cdc_change_log",

    "qbo_es_incremental_merge_audit":
        f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_incremental_merge_audit",

    "qbo_es_incremental_validation_audit":
        f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_incremental_validation_audit",

    "qbo_es_ingestion_audit":
        f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_ingestion_audit",
}


# =============================================================================
# 2. VALIDATE EXISTENCE
# =============================================================================

missing_monitoring_tables = [

    table_name

    for table_name, full_name
    in MONITORING_SOURCE_TABLES.items()

    if not spark.catalog.tableExists(
        full_name
    )
]


if missing_monitoring_tables:

    raise RuntimeError(
        "Monitoring source table(s) missing: "
        + ", ".join(
            missing_monitoring_tables
        )
    )


# =============================================================================
# 3. PROFILE EACH TABLE
# =============================================================================

monitoring_profiles = []


for table_name, full_name in MONITORING_SOURCE_TABLES.items():

    df = spark.table(
        full_name
    )

    row_count = (
        df.count()
    )

    column_count = (
        len(
            df.columns
        )
    )


    # -------------------------------------------------------------------------
    # Date / timestamp columns
    # -------------------------------------------------------------------------

    temporal_columns = [

        field.name

        for field in df.schema.fields

        if isinstance(
            field.dataType,
            (
                DateType,
                TimestampType
            )
        )
    ]


    # -------------------------------------------------------------------------
    # Null counts
    # -------------------------------------------------------------------------

    if df.columns:

        null_expressions = [

            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    1
                ).otherwise(
                    0
                )
            ).alias(
                column_name
            )

            for column_name in df.columns
        ]


        null_result = (
            df
            .agg(
                *null_expressions
            )
            .first()
            .asDict()
        )

    else:

        null_result = {}


    total_null_values = (
        sum(
            int(value or 0)
            for value in null_result.values()
        )
    )


    # -------------------------------------------------------------------------
    # Temporal ranges
    # -------------------------------------------------------------------------

    temporal_ranges = {}


    for temporal_column in temporal_columns:

        result = (
            df

            .agg(
                F.min(
                    F.col(
                        temporal_column
                    )
                ).alias(
                    "min_value"
                ),

                F.max(
                    F.col(
                        temporal_column
                    )
                ).alias(
                    "max_value"
                )
            )

            .first()
        )


        temporal_ranges[
            temporal_column
        ] = {

            "min":
                result[
                    "min_value"
                ],

            "max":
                result[
                    "max_value"
                ]
        }


    monitoring_profiles.append({

        "table_name":
            table_name,

        "full_name":
            full_name,

        "row_count":
            row_count,

        "column_count":
            column_count,

        "temporal_columns":
            temporal_columns,

        "total_null_values":
            total_null_values,

        "temporal_ranges":
            temporal_ranges
    })


# =============================================================================
# 4. PRINT SUMMARY
# =============================================================================

print("=" * 100)
print("GOLD OPERATIONAL MONITORING — SOURCE PROFILING")
print("=" * 100)


for profile in monitoring_profiles:

    print()

    print("-" * 100)

    print(
        f"TABLE               : "
        f"{profile['table_name']}"
    )

    print(
        f"ROWS                : "
        f"{profile['row_count']}"
    )

    print(
        f"COLUMNS             : "
        f"{profile['column_count']}"
    )

    print(
        f"TOTAL NULL VALUES   : "
        f"{profile['total_null_values']}"
    )

    print(
        f"TEMPORAL COLUMNS    : "
        f"{', '.join(profile['temporal_columns']) if profile['temporal_columns'] else 'NONE'}"
    )


    for (
        temporal_column,
        ranges
    ) in profile[
        "temporal_ranges"
    ].items():

        print(
            f"  {temporal_column:<30} | "
            f"min={ranges['min']} | "
            f"max={ranges['max']}"
        )


print("=" * 100)


# =============================================================================
# 5. PRINT COMPLETE SCHEMAS
# =============================================================================

for table_name, full_name in MONITORING_SOURCE_TABLES.items():

    print()

    print("=" * 100)

    print(
        f"SCHEMA — "
        f"{table_name.upper()}"
    )

    print("=" * 100)

    spark.table(
        full_name
    ).printSchema()


# =============================================================================
# 6. DISPLAY SAMPLE RECORDS
# =============================================================================

for table_name, full_name in MONITORING_SOURCE_TABLES.items():

    print()

    print("=" * 100)

    print(
        f"SAMPLE — "
        f"{table_name.upper()}"
    )

    print("=" * 100)


    display(
        spark.table(
            full_name
        )
        .limit(
            20
        )
    )


# =============================================================================
# 7. DISTINCT VALUE PROFILING FOR LIKELY STATUS / TYPE COLUMNS
# =============================================================================

LIKELY_CATEGORICAL_KEYWORDS = [

    "status",
    "type",
    "operation",
    "action",
    "entity",
    "table",
    "source",
    "result",
    "validation"
]


for table_name, full_name in MONITORING_SOURCE_TABLES.items():

    df = spark.table(
        full_name
    )


    candidate_columns = [

        column_name

        for column_name in df.columns

        if any(
            keyword in column_name.lower()
            for keyword in LIKELY_CATEGORICAL_KEYWORDS
        )
    ]


    if candidate_columns:

        print()

        print("=" * 100)

        print(
            f"CATEGORICAL PROFILE — "
            f"{table_name.upper()}"
        )

        print("=" * 100)


    for column_name in candidate_columns:

        distinct_count = (
            df
            .select(
                column_name
            )
            .distinct()
            .count()
        )


        print(
            f"{column_name:<35} | "
            f"distinct values = "
            f"{distinct_count}"
        )


        if distinct_count <= 25:

            display(
                df
                .groupBy(
                    column_name
                )
                .count()
                .orderBy(
                    F.col(
                        "count"
                    ).desc()
                )
            )


# =============================================================================
# 8. FINAL STATUS
# =============================================================================

print("=" * 100)

print(
    "GOLD OPERATIONAL MONITORING — SOURCE PROFILING: SUCCEEDED"
)

print("=" * 100)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 15, Finished, Available, Finished, False)

GOLD OPERATIONAL MONITORING — SOURCE PROFILING

----------------------------------------------------------------------------------------------------
TABLE               : qbo_es_cdc_change_log
ROWS                : 4790
COLUMNS             : 16
TOTAL NULL VALUES   : 9580
TEMPORAL COLUMNS    : previous_watermark_utc, extraction_start_utc, extraction_end_utc, ingested_utc
  previous_watermark_utc         | min=None | max=None
  extraction_start_utc           | min=2026-07-06 04:11:37.380154 | max=2026-07-16 12:33:21.509536
  extraction_end_utc             | min=2026-08-04 04:16:37.380154 | max=2026-08-14 12:38:21.509536
  ingested_utc                   | min=2026-08-04 04:20:32.210815 | max=2026-08-14 12:39:05.058422

----------------------------------------------------------------------------------------------------
TABLE               : qbo_es_incremental_merge_audit
ROWS                : 345
COLUMNS             : 13
TOTAL NULL VALUES   : 529
TEMPORAL COLUMNS    : started_utc, complete

SynapseWidget(Synapse.DataFrame, b0705984-6886-4557-a063-e4fa63cae6dd)


SAMPLE — QBO_ES_INCREMENTAL_MERGE_AUDIT


SynapseWidget(Synapse.DataFrame, 0cc7fc30-9cf9-4f7c-8eb4-10fe2f770090)

...

target_line_table                   | distinct values = 8


SynapseWidget(Synapse.DataFrame, fd269f8c-6723-48bd-a43c-8d821d40be49)

status                              | distinct values = 1


SynapseWidget(Synapse.DataFrame, 8e6f9c85-7dbc-4dc6-b30b-0f61afad6cc0)

GOLD OPERATIONAL MONITORING — SOURCE PROFILING: SUCCEEDED


In [14]:
# =============================================================================
# CELL 13 — GOLD OPERATIONAL MONITORING MODEL
# Unified Multi-Source Monitoring Model
#
# Sources:
# - platform_ingestion_audit
# - qbo_es_ingestion_audit
# - qbo_es_incremental_merge_audit
# - qbo_es_incremental_validation_audit
# - qbo_es_cdc_change_log
#
# Outputs prepared in memory:
# - GOLD_DIM_PIPELINE_DF
# - GOLD_DIM_DATA_OBJECT_DF
# - GOLD_FACT_PIPELINE_RUN_DF
# - GOLD_FACT_DATA_QUALITY_DF
# - GOLD_FACT_CDC_ACTIVITY_DF
# - GOLD_AGG_SOURCE_FRESHNESS_DF
#
# No Gold tables are written in this cell.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. CONSTANTS / CANONICAL CODES
# =============================================================================

QBO_CANONICAL_SOURCE_SYSTEM = "QBO_ES"
QBO_CANONICAL_SOURCE_COMPANY = "ES01"
QBO_PIPELINE_NAME = "QBO_ES_INCREMENTAL_INGESTION"


# =============================================================================
# 2. KEY HELPERS
# =============================================================================

def qbo_data_object_key(source_object_column):
    """
    Preserve the established QBO Gold monitoring key convention:

        SHA2(QBO_ES | source_object)

    Do not use raw QBO source_system because CDC emits QBO rather than QBO_ES.
    """

    return F.sha2(
        F.concat_ws(
            "|",
            F.lit(QBO_CANONICAL_SOURCE_SYSTEM),
            F.upper(
                F.trim(
                    source_object_column
                )
            )
        ),
        256
    )


def generic_data_object_key(
    source_system_column,
    source_object_column
):
    """
    Generic enterprise monitoring key:

        SHA2(source_system | source_object)

    Company is intentionally not included because the existing semantic
    monitoring model identifies a data object by source system + object.
    """

    return F.sha2(
        F.concat_ws(
            "|",
            F.upper(
                F.trim(
                    source_system_column
                )
            ),
            F.upper(
                F.trim(
                    source_object_column
                )
            )
        ),
        256
    )


def pipeline_key(
    source_system_column,
    source_company_column,
    pipeline_name_column
):
    """
    One key per logical pipeline, not per execution.
    """

    return F.sha2(
        F.concat_ws(
            "|",

            F.upper(
                F.trim(
                    F.coalesce(
                        source_system_column,
                        F.lit("UNKNOWN")
                    )
                )
            ),

            F.upper(
                F.trim(
                    F.coalesce(
                        source_company_column,
                        F.lit("UNKNOWN")
                    )
                )
            ),

            F.upper(
                F.trim(
                    F.coalesce(
                        pipeline_name_column,
                        F.lit("UNKNOWN")
                    )
                )
            )
        ),
        256
    )


# =============================================================================
# 3. LOAD MONITORING SOURCES
# =============================================================================

PLATFORM_INGESTION_AUDIT_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.platform_ingestion_audit"
)

INGESTION_AUDIT_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_ingestion_audit"
)

MERGE_AUDIT_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_incremental_merge_audit"
)

VALIDATION_AUDIT_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_incremental_validation_audit"
)

CDC_CHANGE_LOG_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_cdc_change_log"
)


print("=" * 100)
print("MONITORING SOURCE COUNTS")
print("=" * 100)

print(
    f"platform_ingestion_audit          : "
    f"{PLATFORM_INGESTION_AUDIT_DF.count():,}"
)

print(
    f"qbo_es_ingestion_audit            : "
    f"{INGESTION_AUDIT_DF.count():,}"
)

print(
    f"qbo_es_incremental_merge_audit    : "
    f"{MERGE_AUDIT_DF.count():,}"
)

print(
    f"qbo_es_incremental_validation     : "
    f"{VALIDATION_AUDIT_DF.count():,}"
)

print(
    f"qbo_es_cdc_change_log             : "
    f"{CDC_CHANGE_LOG_DF.count():,}"
)

print("=" * 100)


# =============================================================================
# 4. STANDARDISE GENERIC PLATFORM AUDIT
# =============================================================================

GENERIC_PLATFORM_AUDIT_DF = (
    PLATFORM_INGESTION_AUDIT_DF

    # QBO already has richer specialist monitoring.
    .filter(
        ~F.upper(
            F.coalesce(
                F.col("source_system"),
                F.lit("")
            )
        ).isin(
            "QBO",
            "QBO_ES",
            "QUICKBOOKS",
            "QUICKBOOKS_ES"
        )
    )

    .withColumn(
        "source_system",
        F.upper(
            F.trim(
                F.col("source_system")
            )
        )
    )

    .withColumn(
        "source_company",
        F.upper(
            F.trim(
                F.col("source_company")
            )
        )
    )

    .withColumn(
        "source_object",
        F.trim(
            F.col("source_object")
        )
    )

    .withColumn(
        "pipeline_name",
        F.trim(
            F.col("pipeline_name")
        )
    )

    .withColumn(
        "status_standardised",

        F.when(
            F.upper(
                F.col("status")
            ).isin(
                "FAILED",
                "FAIL",
                "ERROR"
            ),
            F.lit("FAILED")
        )

        .when(
            F.upper(
                F.col("status")
            ).isin(
                "WARNING",
                "WARN"
            ),
            F.lit("WARNING")
        )

        .when(
            F.upper(
                F.col("status")
            ).isin(
                "SUCCEEDED",
                "SUCCESS"
            ),
            F.lit("SUCCEEDED")
        )

        .otherwise(
            F.upper(
                F.col("status")
            )
        )
    )
)


# =============================================================================
# 5. DIM_PIPELINE
#
# Grain:
# One row per logical pipeline.
# =============================================================================

QBO_PIPELINE_DIM_DF = (
    INGESTION_AUDIT_DF

    .limit(1)

    .select(
        F.lit(
            QBO_PIPELINE_NAME
        ).alias(
            "pipeline_name"
        ),

        F.lit(
            "BRONZE"
        ).alias(
            "pipeline_layer"
        ),

        F.lit(
            "INCREMENTAL_CDC"
        ).alias(
            "pipeline_type"
        ),

        F.lit(
            QBO_CANONICAL_SOURCE_SYSTEM
        ).alias(
            "source_system"
        ),

        F.lit(
            QBO_CANONICAL_SOURCE_COMPANY
        ).alias(
            "source_company"
        ),

        F.lit(1)
        .cast("int")
        .alias(
            "is_active"
        )
    )
)


GENERIC_PIPELINE_DIM_DF = (
    GENERIC_PLATFORM_AUDIT_DF

    .select(
        "pipeline_name",
        "source_system",
        "source_company"
    )

    .filter(
        F.col("pipeline_name").isNotNull()
    )

    .dropDuplicates(
        [
            "pipeline_name",
            "source_system",
            "source_company"
        ]
    )

    .withColumn(
        "pipeline_layer",
        F.lit("BRONZE")
    )

    .withColumn(
        "pipeline_type",

        F.when(
            F.lower(
                F.col("pipeline_name")
            ).contains(
                "incremental"
            ),
            F.lit("INCREMENTAL")
        )

        .otherwise(
            F.lit("INGESTION")
        )
    )

    .withColumn(
        "is_active",
        F.lit(1).cast("int")
    )

    .select(
        "pipeline_name",
        "pipeline_layer",
        "pipeline_type",
        "source_system",
        "source_company",
        "is_active"
    )
)


GOLD_DIM_PIPELINE_DF = (
    QBO_PIPELINE_DIM_DF

    .unionByName(
        GENERIC_PIPELINE_DIM_DF,
        allowMissingColumns=True
    )

    .filter(
        F.col("pipeline_name").isNotNull()
    )

    .dropDuplicates(
        [
            "pipeline_name",
            "source_system",
            "source_company"
        ]
    )

    .withColumn(
        "pipeline_key",

        pipeline_key(
            F.col("source_system"),
            F.col("source_company"),
            F.col("pipeline_name")
        )
    )

    .select(
        "pipeline_key",
        "pipeline_name",
        "pipeline_layer",
        "pipeline_type",
        "source_system",
        "source_company",
        "is_active"
    )
)


# =============================================================================
# 6. BUILD QBO DATA-OBJECT INVENTORY FROM ALL QBO MONITORING SOURCES
# =============================================================================

QBO_OBJECTS_INGESTION_DF = (
    INGESTION_AUDIT_DF

    .select(
        F.trim(
            F.col("source_object")
        ).alias(
            "source_object"
        ),

        F.col(
            "target_table"
        ),

        F.col(
            "target_line_table"
        )
    )

    .filter(
        F.col("source_object").isNotNull()
    )
)


QBO_OBJECTS_MERGE_DF = (
    MERGE_AUDIT_DF

    .select(
        F.trim(
            F.col("source_object")
        ).alias(
            "source_object"
        ),

        F.col(
            "target_table"
        ),

        F.col(
            "target_line_table"
        )
    )

    .filter(
        F.col("source_object").isNotNull()
    )
)


QBO_OBJECTS_VALIDATION_DF = (
    VALIDATION_AUDIT_DF

    .select(
        F.trim(
            F.col("object_name")
        ).alias(
            "source_object"
        ),

        F.lit(None)
        .cast("string")
        .alias(
            "target_table"
        ),

        F.lit(None)
        .cast("string")
        .alias(
            "target_line_table"
        )
    )

    .filter(
        F.col("source_object").isNotNull()
    )
)


QBO_OBJECTS_CDC_DF = (
    CDC_CHANGE_LOG_DF

    .select(
        F.trim(
            F.col("source_object")
        ).alias(
            "source_object"
        ),

        F.lit(None)
        .cast("string")
        .alias(
            "target_table"
        ),

        F.lit(None)
        .cast("string")
        .alias(
            "target_line_table"
        )
    )

    .filter(
        F.col("source_object").isNotNull()
    )
)


QBO_OBJECT_RAW_DF = (
    QBO_OBJECTS_INGESTION_DF

    .unionByName(
        QBO_OBJECTS_MERGE_DF,
        allowMissingColumns=True
    )

    .unionByName(
        QBO_OBJECTS_VALIDATION_DF,
        allowMissingColumns=True
    )

    .unionByName(
        QBO_OBJECTS_CDC_DF,
        allowMissingColumns=True
    )

    .withColumn(
        "_object_match_key",

        F.upper(
            F.trim(
                F.col("source_object")
            )
        )
    )
)


QBO_OBJECT_DIM_DF = (
    QBO_OBJECT_RAW_DF

    .groupBy(
        "_object_match_key"
    )

    .agg(
        F.first(
            "source_object",
            ignorenulls=True
        ).alias(
            "source_object"
        ),

        F.first(
            "target_table",
            ignorenulls=True
        ).alias(
            "target_table"
        ),

        F.first(
            "target_line_table",
            ignorenulls=True
        ).alias(
            "target_line_table"
        )
    )

    .withColumn(
        "source_system",
        F.lit(
            QBO_CANONICAL_SOURCE_SYSTEM
        )
    )

    .withColumn(
        "source_company",
        F.lit(
            QBO_CANONICAL_SOURCE_COMPANY
        )
    )

    .withColumn(
        "data_layer",
        F.lit("BRONZE")
    )

    .withColumn(
        "data_object_key",

        qbo_data_object_key(
            F.col("source_object")
        )
    )

    .drop(
        "_object_match_key"
    )
)


# =============================================================================
# 7. GENERIC SAP / CZECH DATA-OBJECT INVENTORY
# =============================================================================

GENERIC_OBJECT_DIM_DF = (
    GENERIC_PLATFORM_AUDIT_DF

    .select(
        "source_system",
        "source_company",
        "source_object",
        "target_table"
    )

    .filter(
        F.col("source_object").isNotNull()
    )

    .dropDuplicates(
        [
            "source_system",
            "source_object"
        ]
    )

    .withColumn(
        "target_line_table",
        F.lit(None).cast("string")
    )

    .withColumn(
        "data_layer",
        F.lit("BRONZE")
    )

    .withColumn(
        "data_object_key",

        generic_data_object_key(
            F.col("source_system"),
            F.col("source_object")
        )
    )
)


# =============================================================================
# 8. UNIFIED DIM_DATA_OBJECT
# =============================================================================

GOLD_DIM_DATA_OBJECT_DF = (
    QBO_OBJECT_DIM_DF

    .select(
        "data_object_key",
        "source_system",
        "source_company",
        "data_layer",
        "source_object",
        "target_table",
        "target_line_table"
    )

    .unionByName(
        GENERIC_OBJECT_DIM_DF.select(
            "data_object_key",
            "source_system",
            "source_company",
            "data_layer",
            "source_object",
            "target_table",
            "target_line_table"
        ),
        allowMissingColumns=True
    )

    .dropDuplicates(
        ["data_object_key"]
    )
)


# =============================================================================
# 9. QBO INGESTION RUN AGGREGATION
# =============================================================================

QBO_INGESTION_RUN_DF = (
    INGESTION_AUDIT_DF

    .groupBy(
        "pipeline_run_id",
        "source_object"
    )

    .agg(

        F.min(
            "started_utc"
        ).alias(
            "ingestion_started_utc"
        ),

        F.max(
            "completed_utc"
        ).alias(
            "ingestion_completed_utc"
        ),

        F.sum(
            F.coalesce(
                F.col("pages_read"),
                F.lit(0)
            )
        ).alias(
            "pages_read"
        ),

        F.sum(
            F.coalesce(
                F.col("header_rows_written"),
                F.lit(0)
            )
        ).alias(
            "header_rows_written"
        ),

        F.sum(
            F.coalesce(
                F.col("line_rows_written"),
                F.lit(0)
            )
        ).alias(
            "line_rows_written"
        ),

        F.max(
            F.when(
                F.upper(
                    F.col("status")
                ).isin(
                    "FAILED",
                    "FAIL",
                    "ERROR"
                ),
                1
            ).otherwise(0)
        ).alias(
            "ingestion_failed_flag"
        ),

        F.concat_ws(
            " | ",
            F.collect_set(
                F.when(
                    F.col(
                        "error_message"
                    ).isNotNull(),
                    F.col(
                        "error_message"
                    )
                )
            )
        ).alias(
            "ingestion_error_message"
        )
    )
)


# =============================================================================
# 10. QBO MERGE RUN AGGREGATION
# =============================================================================

QBO_MERGE_RUN_DF = (
    MERGE_AUDIT_DF

    .groupBy(
        "pipeline_run_id",
        "source_object"
    )

    .agg(

        F.min(
            "started_utc"
        ).alias(
            "merge_started_utc"
        ),

        F.max(
            "completed_utc"
        ).alias(
            "merge_completed_utc"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "changed_records_received"
                ),
                F.lit(0)
            )
        ).alias(
            "changed_records_received"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "headers_upserted"
                ),
                F.lit(0)
            )
        ).alias(
            "headers_upserted"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "headers_deleted"
                ),
                F.lit(0)
            )
        ).alias(
            "headers_deleted"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "lines_replaced"
                ),
                F.lit(0)
            )
        ).alias(
            "lines_replaced"
        ),

        F.max(
            F.when(
                F.upper(
                    F.col("status")
                ).isin(
                    "FAILED",
                    "FAIL",
                    "ERROR"
                ),
                1
            ).otherwise(0)
        ).alias(
            "merge_failed_flag"
        ),

        F.concat_ws(
            " | ",
            F.collect_set(
                F.when(
                    F.col(
                        "error_message"
                    ).isNotNull(),
                    F.col(
                        "error_message"
                    )
                )
            )
        ).alias(
            "merge_error_message"
        )
    )
)


# =============================================================================
# 11. QBO VALIDATION AGGREGATION
# =============================================================================

QBO_VALIDATION_RUN_DF = (
    VALIDATION_AUDIT_DF

    .groupBy(
        "pipeline_run_id",
        "object_name"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "validation_check_count"
        ),

        F.sum(
            F.when(
                F.upper(
                    F.col("status")
                ).isin(
                    "FAILED",
                    "FAIL",
                    "ERROR"
                ),
                1
            ).otherwise(0)
        ).alias(
            "validation_failed_count"
        ),

        F.sum(
            F.when(
                F.upper(
                    F.col("status")
                ).isin(
                    "WARNING",
                    "WARN"
                ),
                1
            ).otherwise(0)
        ).alias(
            "validation_warning_count"
        ),

        F.max(
            "validated_utc"
        ).alias(
            "latest_validation_utc"
        )
    )

    .withColumnRenamed(
        "object_name",
        "source_object"
    )
)


# =============================================================================
# 12. QBO FACT_PIPELINE_RUN
# =============================================================================

QBO_FACT_PIPELINE_RUN_DF = (
    QBO_INGESTION_RUN_DF

    .join(
        QBO_MERGE_RUN_DF,
        on=[
            "pipeline_run_id",
            "source_object"
        ],
        how="full_outer"
    )

    .join(
        QBO_VALIDATION_RUN_DF,
        on=[
            "pipeline_run_id",
            "source_object"
        ],
        how="left"
    )

    .withColumn(
        "pipeline_name",
        F.lit(
            QBO_PIPELINE_NAME
        )
    )

    .withColumn(
        "source_system",
        F.lit(
            QBO_CANONICAL_SOURCE_SYSTEM
        )
    )

    .withColumn(
        "source_company",
        F.lit(
            QBO_CANONICAL_SOURCE_COMPANY
        )
    )

    .withColumn(
        "pipeline_key",

        pipeline_key(
            F.col("source_system"),
            F.col("source_company"),
            F.col("pipeline_name")
        )
    )

    .withColumn(
        "data_object_key",

        qbo_data_object_key(
            F.col("source_object")
        )
    )

    .withColumn(
        "pipeline_run_key",

        F.sha2(
            F.concat_ws(
                "|",
                F.lit(
                    QBO_CANONICAL_SOURCE_SYSTEM
                ),
                F.col(
                    "pipeline_run_id"
                ),
                F.upper(
                    F.trim(
                        F.col(
                            "source_object"
                        )
                    )
                )
            ),
            256
        )
    )

    .withColumn(
        "run_started_utc",

        F.least(
            F.col(
                "ingestion_started_utc"
            ),
            F.col(
                "merge_started_utc"
            )
        )
    )

    .withColumn(
        "run_completed_utc",

        F.greatest(
            F.col(
                "ingestion_completed_utc"
            ),
            F.col(
                "merge_completed_utc"
            ),
            F.col(
                "latest_validation_utc"
            )
        )
    )

    .withColumn(
        "duration_seconds",

        F.when(
            F.col(
                "run_started_utc"
            ).isNotNull()
            &
            F.col(
                "run_completed_utc"
            ).isNotNull(),

            F.col(
                "run_completed_utc"
            ).cast("long")
            -
            F.col(
                "run_started_utc"
            ).cast("long")
        )
    )

    .withColumn(
        "rows_written",

        F.coalesce(
            F.col(
                "header_rows_written"
            ),
            F.lit(0)
        )
        +
        F.coalesce(
            F.col(
                "line_rows_written"
            ),
            F.lit(0)
        )
    )

    .withColumn(
        "records_changed",

        F.coalesce(
            F.col(
                "changed_records_received"
            ),
            F.lit(0)
        )
    )

    .withColumn(
        "records_upserted",

        F.coalesce(
            F.col(
                "headers_upserted"
            ),
            F.lit(0)
        )
    )

    .withColumn(
        "records_deleted",

        F.coalesce(
            F.col(
                "headers_deleted"
            ),
            F.lit(0)
        )
    )

    .withColumn(
        "run_status",

        F.when(
            (
                F.coalesce(
                    F.col(
                        "ingestion_failed_flag"
                    ),
                    F.lit(0)
                ) > 0
            )
            |
            (
                F.coalesce(
                    F.col(
                        "merge_failed_flag"
                    ),
                    F.lit(0)
                ) > 0
            )
            |
            (
                F.coalesce(
                    F.col(
                        "validation_failed_count"
                    ),
                    F.lit(0)
                ) > 0
            ),
            F.lit("FAILED")
        )

        .when(
            F.coalesce(
                F.col(
                    "validation_warning_count"
                ),
                F.lit(0)
            ) > 0,
            F.lit("WARNING")
        )

        .otherwise(
            F.lit("SUCCEEDED")
        )
    )

    .withColumn(
        "error_message",

        F.when(
            F.length(
                F.trim(
                    F.concat_ws(
                        " | ",
                        F.col(
                            "ingestion_error_message"
                        ),
                        F.col(
                            "merge_error_message"
                        )
                    )
                )
            ) > 0,

            F.concat_ws(
                " | ",
                F.col(
                    "ingestion_error_message"
                ),
                F.col(
                    "merge_error_message"
                )
            )
        )
    )

    .select(
        "pipeline_run_key",
        "pipeline_key",
        "pipeline_run_id",
        "data_object_key",

        "pipeline_name",
        "source_system",
        "source_company",
        "source_object",

        "run_started_utc",
        "run_completed_utc",
        "duration_seconds",
        "run_status",

        "pages_read",
        "header_rows_written",
        "line_rows_written",
        "rows_written",

        "records_changed",
        "records_upserted",
        "records_deleted",
        "lines_replaced",

        "validation_check_count",
        "validation_failed_count",
        "validation_warning_count",

        "error_message"
    )
)


# =============================================================================
# 13. GENERIC SAP / CZECH FACT_PIPELINE_RUN
# =============================================================================

GENERIC_FACT_PIPELINE_RUN_DF = (
    GENERIC_PLATFORM_AUDIT_DF

    .groupBy(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object"
    )

    .agg(

        F.min(
            "started_utc"
        ).alias(
            "run_started_utc"
        ),

        F.max(
            "completed_utc"
        ).alias(
            "run_completed_utc"
        ),

        F.max(
            F.coalesce(
                F.col(
                    "duration_seconds"
                ),
                F.lit(0)
            )
        ).cast("long").alias(
            "logged_duration_seconds"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "rows_written"
                ),
                F.lit(0)
            )
        ).cast("long").alias(
            "rows_written"
        ),

        F.max(
            F.when(
                F.col(
                    "status_standardised"
                ) == "FAILED",
                3
            )

            .when(
                F.col(
                    "status_standardised"
                ) == "WARNING",
                2
            )

            .when(
                F.col(
                    "status_standardised"
                ) == "SUCCEEDED",
                1
            )

            .otherwise(
                0
            )
        ).alias(
            "status_priority"
        ),

        F.concat_ws(
            " | ",
            F.collect_set(
                F.when(
                    F.col(
                        "error_message"
                    ).isNotNull(),
                    F.col(
                        "error_message"
                    )
                )
            )
        ).alias(
            "generic_error_message"
        )
    )

    .withColumn(
        "run_status",

        F.when(
            F.col(
                "status_priority"
            ) == 3,
            F.lit("FAILED")
        )

        .when(
            F.col(
                "status_priority"
            ) == 2,
            F.lit("WARNING")
        )

        .when(
            F.col(
                "status_priority"
            ) == 1,
            F.lit("SUCCEEDED")
        )

        .otherwise(
            F.lit("UNKNOWN")
        )
    )

    .withColumn(
        "duration_seconds",

        F.when(
            F.col(
                "logged_duration_seconds"
            ) > 0,
            F.col(
                "logged_duration_seconds"
            )
        )

        .when(
            F.col(
                "run_started_utc"
            ).isNotNull()
            &
            F.col(
                "run_completed_utc"
            ).isNotNull(),

            F.col(
                "run_completed_utc"
            ).cast("long")
            -
            F.col(
                "run_started_utc"
            ).cast("long")
        )

        .otherwise(
            F.lit(0).cast("long")
        )
    )

    .withColumn(
        "pipeline_key",

        pipeline_key(
            F.col("source_system"),
            F.col("source_company"),
            F.col("pipeline_name")
        )
    )

    .withColumn(
        "data_object_key",

        generic_data_object_key(
            F.col("source_system"),
            F.col("source_object")
        )
    )

    .withColumn(
        "pipeline_run_key",

        F.sha2(
            F.concat_ws(
                "|",

                F.upper(
                    F.trim(
                        F.col(
                            "source_system"
                        )
                    )
                ),

                F.col(
                    "pipeline_run_id"
                ),

                F.upper(
                    F.trim(
                        F.col(
                            "source_object"
                        )
                    )
                )
            ),
            256
        )
    )

    .withColumn(
        "pages_read",
        F.lit(None).cast("long")
    )

    .withColumn(
        "header_rows_written",
        F.lit(None).cast("long")
    )

    .withColumn(
        "line_rows_written",
        F.lit(None).cast("long")
    )

    .withColumn(
        "records_changed",
        F.lit(None).cast("long")
    )

    .withColumn(
        "records_upserted",
        F.lit(None).cast("long")
    )

    .withColumn(
        "records_deleted",
        F.lit(None).cast("long")
    )

    .withColumn(
        "lines_replaced",
        F.lit(None).cast("long")
    )

    .withColumn(
        "validation_check_count",
        F.lit(None).cast("long")
    )

    .withColumn(
        "validation_failed_count",
        F.lit(None).cast("long")
    )

    .withColumn(
        "validation_warning_count",
        F.lit(None).cast("long")
    )

    .withColumn(
        "error_message",

        F.when(
            F.length(
                F.trim(
                    F.col(
                        "generic_error_message"
                    )
                )
            ) > 0,

            F.col(
                "generic_error_message"
            )
        )
    )

    .select(
        "pipeline_run_key",
        "pipeline_key",
        "pipeline_run_id",
        "data_object_key",

        "pipeline_name",
        "source_system",
        "source_company",
        "source_object",

        "run_started_utc",
        "run_completed_utc",
        "duration_seconds",
        "run_status",

        "pages_read",
        "header_rows_written",
        "line_rows_written",
        "rows_written",

        "records_changed",
        "records_upserted",
        "records_deleted",
        "lines_replaced",

        "validation_check_count",
        "validation_failed_count",
        "validation_warning_count",

        "error_message"
    )
)


# =============================================================================
# 14. UNIFIED FACT_PIPELINE_RUN
# =============================================================================

GOLD_FACT_PIPELINE_RUN_DF = (
    QBO_FACT_PIPELINE_RUN_DF

    .unionByName(
        GENERIC_FACT_PIPELINE_RUN_DF,
        allowMissingColumns=True
    )

    .dropDuplicates(
        ["pipeline_run_key"]
    )
)


# =============================================================================
# 15. FACT_DATA_QUALITY
#
# QBO specialist validation telemetry.
# =============================================================================

GOLD_FACT_DATA_QUALITY_DF = (
    VALIDATION_AUDIT_DF

    .withColumn(
        "pipeline_name",
        F.lit(
            QBO_PIPELINE_NAME
        )
    )

    .withColumn(
        "source_system",
        F.lit(
            QBO_CANONICAL_SOURCE_SYSTEM
        )
    )

    .withColumn(
        "source_company",
        F.lit(
            QBO_CANONICAL_SOURCE_COMPANY
        )
    )

    .withColumn(
        "pipeline_key",

        pipeline_key(
            F.col("source_system"),
            F.col("source_company"),
            F.col("pipeline_name")
        )
    )

    .withColumn(
        "data_quality_key",

        F.sha2(
            F.concat_ws(
                "|",

                F.coalesce(
                    F.col(
                        "pipeline_run_id"
                    ),
                    F.lit("UNKNOWN")
                ),

                F.upper(
                    F.trim(
                        F.coalesce(
                            F.col(
                                "object_name"
                            ),
                            F.lit("UNKNOWN")
                        )
                    )
                ),

                F.coalesce(
                    F.col(
                        "check_name"
                    ),
                    F.lit("UNKNOWN")
                ),

                F.coalesce(
                    F.col(
                        "validated_utc"
                    ).cast("string"),
                    F.lit("UNKNOWN")
                )
            ),
            256
        )
    )

    .withColumn(
        "data_object_key",

        qbo_data_object_key(
            F.col(
                "object_name"
            )
        )
    )

    .withColumn(
        "validation_date_key",

        F.date_format(
            F.col(
                "validated_utc"
            ),
            "yyyyMMdd"
        ).cast("int")
    )

    .select(
        "data_quality_key",
        "pipeline_key",
        "pipeline_run_id",
        "data_object_key",

        F.col(
            "object_name"
        ).alias(
            "source_object"
        ),

        "check_name",
        "status",
        "observed_value",
        "expected_value",
        "details",
        "validated_utc",
        "validation_date_key"
    )
)


# =============================================================================
# 16. FACT_CDC_ACTIVITY
# =============================================================================

GOLD_FACT_CDC_ACTIVITY_DF = (
    CDC_CHANGE_LOG_DF

    .withColumn(
        "source_system",
        F.lit(
            QBO_CANONICAL_SOURCE_SYSTEM
        )
    )

    .withColumn(
        "source_company",

        F.coalesce(
            F.col(
                "source_company"
            ),
            F.lit(
                QBO_CANONICAL_SOURCE_COMPANY
            )
        )
    )

    .withColumn(
        "pipeline_name",
        F.lit(
            QBO_PIPELINE_NAME
        )
    )

    .withColumn(
        "pipeline_key",

        pipeline_key(
            F.col("source_system"),
            F.col("source_company"),
            F.col("pipeline_name")
        )
    )

    .withColumn(
        "cdc_activity_key",

        F.sha2(
            F.concat_ws(
                "|",

                F.coalesce(
                    F.col(
                        "pipeline_run_id"
                    ),
                    F.lit("UNKNOWN")
                ),

                F.upper(
                    F.trim(
                        F.coalesce(
                            F.col(
                                "source_object"
                            ),
                            F.lit("UNKNOWN")
                        )
                    )
                ),

                F.coalesce(
                    F.col(
                        "source_record_id"
                    ),
                    F.lit("UNKNOWN")
                ),

                F.coalesce(
                    F.col(
                        "cdc_operation"
                    ),
                    F.lit("UNKNOWN")
                ),

                F.coalesce(
                    F.col(
                        "ingested_utc"
                    ).cast("string"),
                    F.lit("UNKNOWN")
                )
            ),
            256
        )
    )

    .withColumn(
        "data_object_key",

        qbo_data_object_key(
            F.col(
                "source_object"
            )
        )
    )

    .withColumn(
        "ingestion_date_key",

        F.date_format(
            F.col(
                "ingested_utc"
            ),
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "extraction_duration_seconds",

        F.when(
            F.col(
                "extraction_start_utc"
            ).isNotNull()
            &
            F.col(
                "extraction_end_utc"
            ).isNotNull(),

            F.col(
                "extraction_end_utc"
            ).cast("long")
            -
            F.col(
                "extraction_start_utc"
            ).cast("long")
        )
    )

    .select(
        "cdc_activity_key",
        "pipeline_key",
        "pipeline_run_id",
        "ingestion_id",
        "data_object_key",

        "source_system",
        "source_company",
        "source_environment",
        "source_object",
        "source_record_id",

        "cdc_operation",
        "cdc_status",

        "source_last_updated_time",

        "previous_watermark_utc",
        "extraction_start_utc",
        "extraction_end_utc",
        "extraction_duration_seconds",

        "ingested_utc",
        "ingestion_date_key",

        "payload_hash"
    )
)


# =============================================================================
# 17. QBO SOURCE FRESHNESS
# =============================================================================

QBO_SOURCE_FRESHNESS_DF = (
    CDC_CHANGE_LOG_DF

    .withColumn(
        "source_system",
        F.lit(
            QBO_CANONICAL_SOURCE_SYSTEM
        )
    )

    .withColumn(
        "source_company",

        F.coalesce(
            F.col(
                "source_company"
            ),
            F.lit(
                QBO_CANONICAL_SOURCE_COMPANY
            )
        )
    )

    .groupBy(
        "source_system",
        "source_company",
        "source_object"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "cdc_record_count"
        ),

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "pipeline_run_count"
        ),

        F.max(
            "extraction_end_utc"
        ).alias(
            "last_extraction_utc"
        ),

        F.max(
            "ingested_utc"
        ).alias(
            "last_ingested_utc"
        ),

        F.max(
            "source_last_updated_time"
        ).alias(
            "latest_source_update_value"
        )
    )

    .withColumn(
        "data_object_key",

        qbo_data_object_key(
            F.col(
                "source_object"
            )
        )
    )
)


# =============================================================================
# 18. GENERIC SAP / CZECH SOURCE FRESHNESS
# =============================================================================

GENERIC_SOURCE_FRESHNESS_DF = (
    GENERIC_PLATFORM_AUDIT_DF

    .groupBy(
        "source_system",
        "source_company",
        "source_object"
    )

    .agg(

        F.lit(None)
        .cast("long")
        .alias(
            "cdc_record_count"
        ),

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "pipeline_run_count"
        ),

        F.max(
            "completed_utc"
        ).alias(
            "last_extraction_utc"
        ),

        F.max(
            F.coalesce(
                F.col(
                    "audit_logged_utc"
                ),
                F.col(
                    "completed_utc"
                )
            )
        ).alias(
            "last_ingested_utc"
        ),

        F.lit(None)
        .cast("timestamp")
        .alias(
            "latest_source_update_value"
        )
    )

    .withColumn(
        "data_object_key",

        generic_data_object_key(
            F.col("source_system"),
            F.col("source_object")
        )
    )
)


# =============================================================================
# 19. UNIFIED SOURCE FRESHNESS
# =============================================================================

GOLD_AGG_SOURCE_FRESHNESS_DF = (
    QBO_SOURCE_FRESHNESS_DF

    .unionByName(
        GENERIC_SOURCE_FRESHNESS_DF,
        allowMissingColumns=True
    )

    .withColumn(
        "freshness_age_hours",

        F.when(
            F.col(
                "last_ingested_utc"
            ).isNotNull(),

            (
                F.current_timestamp()
                .cast("long")
                -
                F.col(
                    "last_ingested_utc"
                ).cast("long")
            )
            /
            F.lit(3600.0)
        )
    )

    .withColumn(
        "freshness_status",

        F.when(
            F.col(
                "last_ingested_utc"
            ).isNull(),
            F.lit("UNKNOWN")
        )

        .when(
            F.col(
                "freshness_age_hours"
            ) <= 24,
            F.lit("FRESH")
        )

        .when(
            F.col(
                "freshness_age_hours"
            ) <= 48,
            F.lit("WARNING")
        )

        .otherwise(
            F.lit("STALE")
        )
    )

    .dropDuplicates(
        ["data_object_key"]
    )

    .select(
        "data_object_key",
        "source_system",
        "source_company",
        "source_object",

        "cdc_record_count",
        "pipeline_run_count",

        "last_extraction_utc",
        "last_ingested_utc",
        "latest_source_update_value",

        "freshness_age_hours",
        "freshness_status"
    )
)


# =============================================================================
# 20. DUPLICATE VALIDATION
# =============================================================================

pipeline_dimension_duplicate_count = (
    GOLD_DIM_PIPELINE_DF

    .groupBy(
        "pipeline_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


data_object_dimension_duplicate_count = (
    GOLD_DIM_DATA_OBJECT_DF

    .groupBy(
        "data_object_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


pipeline_run_duplicate_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .groupBy(
        "pipeline_run_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


dq_duplicate_count = (
    GOLD_FACT_DATA_QUALITY_DF

    .groupBy(
        "data_quality_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


cdc_duplicate_count = (
    GOLD_FACT_CDC_ACTIVITY_DF

    .groupBy(
        "cdc_activity_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


freshness_duplicate_count = (
    GOLD_AGG_SOURCE_FRESHNESS_DF

    .groupBy(
        "data_object_key"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


# =============================================================================
# 21. ORPHAN VALIDATION
# =============================================================================

DIM_PIPELINE_KEYS_DF = (
    GOLD_DIM_PIPELINE_DF

    .select(
        "pipeline_key"
    )

    .distinct()
)


DIM_OBJECT_KEYS_DF = (
    GOLD_DIM_DATA_OBJECT_DF

    .select(
        "data_object_key"
    )

    .distinct()
)


def pipeline_orphan_count(df):

    return (
        df

        .select(
            "pipeline_key"
        )

        .filter(
            F.col(
                "pipeline_key"
            ).isNotNull()
        )

        .distinct()

        .join(
            DIM_PIPELINE_KEYS_DF,
            on="pipeline_key",
            how="left_anti"
        )

        .count()
    )


def data_object_orphan_count(df):

    return (
        df

        .select(
            "data_object_key"
        )

        .filter(
            F.col(
                "data_object_key"
            ).isNotNull()
        )

        .distinct()

        .join(
            DIM_OBJECT_KEYS_DF,
            on="data_object_key",
            how="left_anti"
        )

        .count()
    )


pipeline_dimension_orphans = (
    pipeline_orphan_count(
        GOLD_FACT_PIPELINE_RUN_DF
    )
)


pipeline_object_orphans = (
    data_object_orphan_count(
        GOLD_FACT_PIPELINE_RUN_DF
    )
)


dq_pipeline_orphans = (
    pipeline_orphan_count(
        GOLD_FACT_DATA_QUALITY_DF
    )
)


dq_object_orphans = (
    data_object_orphan_count(
        GOLD_FACT_DATA_QUALITY_DF
    )
)


cdc_pipeline_orphans = (
    pipeline_orphan_count(
        GOLD_FACT_CDC_ACTIVITY_DF
    )
)


cdc_object_orphans = (
    data_object_orphan_count(
        GOLD_FACT_CDC_ACTIVITY_DF
    )
)


freshness_object_orphans = (
    data_object_orphan_count(
        GOLD_AGG_SOURCE_FRESHNESS_DF
    )
)


# =============================================================================
# 22. ROW COUNTS / KPIs
# =============================================================================

pipeline_dimension_count = (
    GOLD_DIM_PIPELINE_DF.count()
)

data_object_dimension_count = (
    GOLD_DIM_DATA_OBJECT_DF.count()
)

pipeline_run_count = (
    GOLD_FACT_PIPELINE_RUN_DF.count()
)

data_quality_count = (
    GOLD_FACT_DATA_QUALITY_DF.count()
)

cdc_activity_count = (
    GOLD_FACT_CDC_ACTIVITY_DF.count()
)

freshness_count = (
    GOLD_AGG_SOURCE_FRESHNESS_DF.count()
)


distinct_pipeline_executions = (
    GOLD_FACT_PIPELINE_RUN_DF

    .select(
        "pipeline_run_id"
    )

    .filter(
        F.col(
            "pipeline_run_id"
        ).isNotNull()
    )

    .distinct()

    .count()
)


successful_pipeline_objects = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "run_status"
        ) == "SUCCEEDED"
    )

    .count()
)


failed_pipeline_objects = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "run_status"
        ) == "FAILED"
    )

    .count()
)


warning_pipeline_objects = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "run_status"
        ) == "WARNING"
    )

    .count()
)


failed_dq_checks = (
    GOLD_FACT_DATA_QUALITY_DF

    .filter(
        F.upper(
            F.col("status")
        ).isin(
            "FAILED",
            "FAIL",
            "ERROR"
        )
    )

    .count()
)


warning_dq_checks = (
    GOLD_FACT_DATA_QUALITY_DF

    .filter(
        F.upper(
            F.col("status")
        ).isin(
            "WARNING",
            "WARN"
        )
    )

    .count()
)


# =============================================================================
# 23. SUMMARY
# =============================================================================

print("=" * 110)
print("GOLD OPERATIONAL MONITORING — UNIFIED MODEL PREPARATION")
print("=" * 110)

print(
    f"dim_pipeline rows                   : "
    f"{pipeline_dimension_count:,}"
)

print(
    f"dim_data_object rows                : "
    f"{data_object_dimension_count:,}"
)

print(
    f"fact_pipeline_run rows              : "
    f"{pipeline_run_count:,}"
)

print(
    f"fact_data_quality rows              : "
    f"{data_quality_count:,}"
)

print(
    f"fact_cdc_activity rows              : "
    f"{cdc_activity_count:,}"
)

print(
    f"agg_source_freshness rows           : "
    f"{freshness_count:,}"
)

print("-" * 110)

print(
    f"Distinct pipeline executions        : "
    f"{distinct_pipeline_executions:,}"
)

print(
    f"Successful pipeline objects         : "
    f"{successful_pipeline_objects:,}"
)

print(
    f"Failed pipeline objects             : "
    f"{failed_pipeline_objects:,}"
)

print(
    f"Warning pipeline objects            : "
    f"{warning_pipeline_objects:,}"
)

print(
    f"Failed DQ checks                    : "
    f"{failed_dq_checks:,}"
)

print(
    f"Warning DQ checks                   : "
    f"{warning_dq_checks:,}"
)

print("-" * 110)

print(
    f"dim_pipeline duplicate keys         : "
    f"{pipeline_dimension_duplicate_count}"
)

print(
    f"dim_data_object duplicate keys      : "
    f"{data_object_dimension_duplicate_count}"
)

print(
    f"fact_pipeline_run duplicate keys    : "
    f"{pipeline_run_duplicate_count}"
)

print(
    f"fact_data_quality duplicate keys    : "
    f"{dq_duplicate_count}"
)

print(
    f"fact_cdc_activity duplicate keys    : "
    f"{cdc_duplicate_count}"
)

print(
    f"source freshness duplicate keys     : "
    f"{freshness_duplicate_count}"
)

print("-" * 110)

print(
    f"fact_pipeline_run → dim_pipeline    : "
    f"{pipeline_dimension_orphans}"
)

print(
    f"fact_pipeline_run → dim_data_object : "
    f"{pipeline_object_orphans}"
)

print(
    f"fact_data_quality → dim_pipeline    : "
    f"{dq_pipeline_orphans}"
)

print(
    f"fact_data_quality → dim_data_object : "
    f"{dq_object_orphans}"
)

print(
    f"fact_cdc_activity → dim_pipeline    : "
    f"{cdc_pipeline_orphans}"
)

print(
    f"fact_cdc_activity → dim_data_object : "
    f"{cdc_object_orphans}"
)

print(
    f"freshness → dim_data_object         : "
    f"{freshness_object_orphans}"
)

print("=" * 110)


# =============================================================================
# 24. FAIL FAST
# =============================================================================

monitoring_failures = []


if pipeline_dimension_duplicate_count > 0:
    monitoring_failures.append(
        "DIM_PIPELINE_DUPLICATE_KEY"
    )

if data_object_dimension_duplicate_count > 0:
    monitoring_failures.append(
        "DIM_DATA_OBJECT_DUPLICATE_KEY"
    )

if pipeline_run_duplicate_count > 0:
    monitoring_failures.append(
        "PIPELINE_RUN_DUPLICATE_KEY"
    )

if dq_duplicate_count > 0:
    monitoring_failures.append(
        "DQ_DUPLICATE_KEY"
    )

if cdc_duplicate_count > 0:
    monitoring_failures.append(
        "CDC_DUPLICATE_KEY"
    )

if freshness_duplicate_count > 0:
    monitoring_failures.append(
        "FRESHNESS_DUPLICATE_KEY"
    )

if pipeline_dimension_orphans > 0:
    monitoring_failures.append(
        "PIPELINE_DIMENSION_ORPHAN"
    )

if pipeline_object_orphans > 0:
    monitoring_failures.append(
        "PIPELINE_OBJECT_ORPHAN"
    )

if dq_pipeline_orphans > 0:
    monitoring_failures.append(
        "DQ_PIPELINE_ORPHAN"
    )

if dq_object_orphans > 0:
    monitoring_failures.append(
        "DQ_OBJECT_ORPHAN"
    )

if cdc_pipeline_orphans > 0:
    monitoring_failures.append(
        "CDC_PIPELINE_ORPHAN"
    )

if cdc_object_orphans > 0:
    monitoring_failures.append(
        "CDC_OBJECT_ORPHAN"
    )

if freshness_object_orphans > 0:
    monitoring_failures.append(
        "FRESHNESS_OBJECT_ORPHAN"
    )


if monitoring_failures:

    raise RuntimeError(
        "GOLD MONITORING MODEL PREPARATION FAILED: "
        + ", ".join(
            monitoring_failures
        )
    )


# =============================================================================
# 25. DISPLAY PIPELINES
# =============================================================================

print("=" * 110)
print("DIM_PIPELINE — LOGICAL PIPELINES")
print("=" * 110)

display(
    GOLD_DIM_PIPELINE_DF

    .orderBy(
        "source_system",
        "pipeline_name"
    )
)


# =============================================================================
# 26. DISPLAY LATEST PIPELINE EVENTS
# =============================================================================

print("=" * 110)
print("FACT_PIPELINE_RUN — LATEST EXECUTION EVENTS")
print("=" * 110)

display(
    GOLD_FACT_PIPELINE_RUN_DF

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object",
        "run_started_utc",
        "run_completed_utc",
        "duration_seconds",
        "run_status",
        "rows_written",
        "error_message"
    )

    .orderBy(
        F.col(
            "run_started_utc"
        ).desc_nulls_last()
    )

    .limit(
        200
    )
)


# =============================================================================
# 27. DISPLAY FAILED EVENTS
# =============================================================================

print("=" * 110)
print("FAILED PIPELINE OBJECTS")
print("=" * 110)

display(
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "run_status"
        ) == "FAILED"
    )

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object",
        "run_started_utc",
        "run_completed_utc",
        "duration_seconds",
        "rows_written",
        "error_message"
    )

    .orderBy(
        F.col(
            "run_started_utc"
        ).desc_nulls_last()
    )
)


# =============================================================================
# 28. DISPLAY SOURCE FRESHNESS
# =============================================================================

print("=" * 110)
print("AGG_SOURCE_FRESHNESS")
print("=" * 110)

display(
    GOLD_AGG_SOURCE_FRESHNESS_DF

    .orderBy(
        F.col(
            "freshness_age_hours"
        ).desc_nulls_last()
    )
)


# =============================================================================
# 29. FINAL STATUS
# =============================================================================

print("=" * 110)
print(
    "GOLD OPERATIONAL MONITORING — "
    "UNIFIED MODEL PREPARATION: SUCCEEDED"
)
print("=" * 110)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 16, Finished, Available, Finished, False)

MONITORING SOURCE COUNTS
platform_ingestion_audit          : 57
qbo_es_ingestion_audit            : 176
qbo_es_incremental_merge_audit    : 345
qbo_es_incremental_validation     : 3,586
qbo_es_cdc_change_log             : 4,790
GOLD OPERATIONAL MONITORING — UNIFIED MODEL PREPARATION
dim_pipeline rows                   : 3
dim_data_object rows                : 69
fact_pipeline_run rows              : 578
fact_data_quality rows              : 3,586
fact_cdc_activity rows              : 4,790
agg_source_freshness rows           : 36
--------------------------------------------------------------------------------------------------------------
Distinct pipeline executions        : 40
Successful pipeline objects         : 570
Failed pipeline objects             : 8
Warning pipeline objects            : 0
Failed DQ checks                    : 0
Warning DQ checks                   : 0
--------------------------------------------------------------------------------------------------------------

SynapseWidget(Synapse.DataFrame, 8db60a15-1f0d-46ae-9ebc-3a87fe7501b0)

FACT_PIPELINE_RUN — LATEST EXECUTION EVENTS


SynapseWidget(Synapse.DataFrame, 946dcc8a-dd64-43fb-bd08-1f3c5fe84ef7)

FAILED PIPELINE OBJECTS


SynapseWidget(Synapse.DataFrame, 28b36359-2cdd-4395-8118-c99ad4be3e38)

AGG_SOURCE_FRESHNESS


SynapseWidget(Synapse.DataFrame, e5828b8f-b4d5-48ec-83b7-154f2f440545)

GOLD OPERATIONAL MONITORING — UNIFIED MODEL PREPARATION: SUCCEEDED


In [15]:
# =============================================================================
# CELL 14 — GOLD OPERATIONAL MONITORING
# Unified Multi-Source Pre-Write Validation
#
# PURPOSE
# -------
# Validate the unified monitoring model produced by Cell 13 before any Gold
# monitoring tables are persisted.
#
# Current monitored pipelines:
#   - QuickBooks Spain incremental ingestion
#   - SAP Business One HANA UK ingestion
#   - Czech ERP ingestion
#
# Validates:
#   - logical pipeline grain
#   - data-object grain
#   - fact primary-key uniqueness
#   - pipeline foreign-key integrity
#   - data-object foreign-key integrity
#   - QBO DQ source-to-Gold reconciliation
#   - QBO CDC source-to-Gold reconciliation
#   - timestamps and durations
#   - run-status values
#   - freshness values
#   - operational KPI preparation
#
# IMPORTANT:
#   - Cell 13 already created the correct pipeline_key and data_object_key.
#   - Do NOT recreate dim_pipeline here.
#   - Do NOT overwrite pipeline_key here.
#   - No Gold tables are written in this cell.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. CORE ROW COUNTS
# =============================================================================

monitoring_counts = {

    "dim_pipeline":
        GOLD_DIM_PIPELINE_DF.count(),

    "dim_data_object":
        GOLD_DIM_DATA_OBJECT_DF.count(),

    "fact_pipeline_run":
        GOLD_FACT_PIPELINE_RUN_DF.count(),

    "fact_data_quality":
        GOLD_FACT_DATA_QUALITY_DF.count(),

    "fact_cdc_activity":
        GOLD_FACT_CDC_ACTIVITY_DF.count(),

    "agg_source_freshness":
        GOLD_AGG_SOURCE_FRESHNESS_DF.count()
}


# =============================================================================
# 2. EXPECTED LOGICAL PIPELINE COUNT
#
# Dynamic rule:
#
#   QBO specialist monitoring pipeline
#   +
#   distinct generic pipelines from platform_ingestion_audit
#
# This avoids hard-coding dim_pipeline = 1 or dim_pipeline = 3.
# =============================================================================

generic_logical_pipeline_count = (
    GENERIC_PLATFORM_AUDIT_DF

    .select(
        "pipeline_name",
        "source_system",
        "source_company"
    )

    .filter(
        F.col("pipeline_name").isNotNull()
    )

    .distinct()

    .count()
)


qbo_logical_pipeline_count = (
    1
    if INGESTION_AUDIT_DF.limit(1).count() > 0
    else 0
)


expected_logical_pipeline_count = (
    generic_logical_pipeline_count
    +
    qbo_logical_pipeline_count
)


actual_logical_pipeline_count = (
    GOLD_DIM_PIPELINE_DF.count()
)


logical_pipeline_count_difference = (
    actual_logical_pipeline_count
    -
    expected_logical_pipeline_count
)


# =============================================================================
# 3. SOURCE-TO-GOLD ROW RECONCILIATION
#
# QBO DQ and CDC are event-grain facts.
# Therefore source row counts must reconcile exactly.
# =============================================================================

source_validation_rows = (
    VALIDATION_AUDIT_DF.count()
)


gold_validation_rows = (
    GOLD_FACT_DATA_QUALITY_DF.count()
)


dq_row_count_difference = (
    gold_validation_rows
    -
    source_validation_rows
)


source_cdc_rows = (
    CDC_CHANGE_LOG_DF.count()
)


gold_cdc_rows = (
    GOLD_FACT_CDC_ACTIVITY_DF.count()
)


cdc_row_count_difference = (
    gold_cdc_rows
    -
    source_cdc_rows
)


# =============================================================================
# 4. PIPELINE FACT SOURCE RECONCILIATION
#
# Generic platform audit is normalised to:
#
# pipeline_run_id
# + pipeline_name
# + source_system
# + source_company
# + source_object
#
# QBO pipeline fact is built independently from specialist QBO audits.
#
# We therefore validate that every generic source grain exists in the unified
# Gold fact.
# =============================================================================

GENERIC_EXPECTED_PIPELINE_GRAIN_DF = (
    GENERIC_PLATFORM_AUDIT_DF

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object"
    )

    .filter(
        F.col("pipeline_run_id").isNotNull()
    )

    .filter(
        F.col("pipeline_name").isNotNull()
    )

    .filter(
        F.col("source_object").isNotNull()
    )

    .distinct()
)


GENERIC_GOLD_PIPELINE_GRAIN_DF = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col("source_system") != "QBO_ES"
    )

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object"
    )

    .distinct()
)


generic_pipeline_missing_from_gold_count = (
    GENERIC_EXPECTED_PIPELINE_GRAIN_DF

    .join(
        GENERIC_GOLD_PIPELINE_GRAIN_DF,

        on=[
            "pipeline_run_id",
            "pipeline_name",
            "source_system",
            "source_company",
            "source_object"
        ],

        how="left_anti"
    )

    .count()
)


# =============================================================================
# 5. PRIMARY KEY DUPLICATE CHECK HELPER
# =============================================================================

def duplicate_key_count(
    df,
    key_columns
):

    return (
        df

        .groupBy(
            *key_columns
        )

        .count()

        .filter(
            F.col("count") > 1
        )

        .count()
    )


# =============================================================================
# 6. PRIMARY KEY DUPLICATE CHECKS
# =============================================================================

pipeline_dimension_duplicate_count = (
    duplicate_key_count(
        GOLD_DIM_PIPELINE_DF,
        ["pipeline_key"]
    )
)


data_object_duplicate_count = (
    duplicate_key_count(
        GOLD_DIM_DATA_OBJECT_DF,
        ["data_object_key"]
    )
)


pipeline_run_duplicate_count = (
    duplicate_key_count(
        GOLD_FACT_PIPELINE_RUN_DF,
        ["pipeline_run_key"]
    )
)


dq_duplicate_count = (
    duplicate_key_count(
        GOLD_FACT_DATA_QUALITY_DF,
        ["data_quality_key"]
    )
)


cdc_duplicate_count = (
    duplicate_key_count(
        GOLD_FACT_CDC_ACTIVITY_DF,
        ["cdc_activity_key"]
    )
)


freshness_duplicate_count = (
    duplicate_key_count(
        GOLD_AGG_SOURCE_FRESHNESS_DF,
        ["data_object_key"]
    )
)


# =============================================================================
# 7. NULL PRIMARY KEY CHECK HELPER
# =============================================================================

def null_key_count(
    df,
    key_column
):

    return (
        df

        .filter(
            F.col(
                key_column
            ).isNull()
        )

        .count()
    )


# =============================================================================
# 8. NULL PRIMARY KEY CHECKS
# =============================================================================

pipeline_dimension_null_key_count = (
    null_key_count(
        GOLD_DIM_PIPELINE_DF,
        "pipeline_key"
    )
)


data_object_null_key_count = (
    null_key_count(
        GOLD_DIM_DATA_OBJECT_DF,
        "data_object_key"
    )
)


pipeline_run_null_key_count = (
    null_key_count(
        GOLD_FACT_PIPELINE_RUN_DF,
        "pipeline_run_key"
    )
)


dq_null_key_count = (
    null_key_count(
        GOLD_FACT_DATA_QUALITY_DF,
        "data_quality_key"
    )
)


cdc_null_key_count = (
    null_key_count(
        GOLD_FACT_CDC_ACTIVITY_DF,
        "cdc_activity_key"
    )
)


# =============================================================================
# 9. DIMENSION KEY INVENTORIES
# =============================================================================

DIM_PIPELINE_KEYS_DF = (
    GOLD_DIM_PIPELINE_DF

    .select(
        "pipeline_key"
    )

    .distinct()
)


DIM_DATA_OBJECT_KEYS_DF = (
    GOLD_DIM_DATA_OBJECT_DF

    .select(
        "data_object_key"
    )

    .distinct()
)


# =============================================================================
# 10. PIPELINE FK ORPHAN HELPER
# =============================================================================

def pipeline_fk_orphan_count(df):

    return (
        df

        .select(
            "pipeline_key"
        )

        .filter(
            F.col(
                "pipeline_key"
            ).isNotNull()
        )

        .distinct()

        .join(
            DIM_PIPELINE_KEYS_DF,
            on="pipeline_key",
            how="left_anti"
        )

        .count()
    )


# =============================================================================
# 11. DATA-OBJECT FK ORPHAN HELPER
# =============================================================================

def data_object_fk_orphan_count(df):

    return (
        df

        .select(
            "data_object_key"
        )

        .filter(
            F.col(
                "data_object_key"
            ).isNotNull()
        )

        .distinct()

        .join(
            DIM_DATA_OBJECT_KEYS_DF,
            on="data_object_key",
            how="left_anti"
        )

        .count()
    )


# =============================================================================
# 12. PIPELINE FK ORPHAN CHECKS
# =============================================================================

pipeline_run_orphan_pipeline_count = (
    pipeline_fk_orphan_count(
        GOLD_FACT_PIPELINE_RUN_DF
    )
)


dq_orphan_pipeline_count = (
    pipeline_fk_orphan_count(
        GOLD_FACT_DATA_QUALITY_DF
    )
)


cdc_orphan_pipeline_count = (
    pipeline_fk_orphan_count(
        GOLD_FACT_CDC_ACTIVITY_DF
    )
)


# =============================================================================
# 13. DATA-OBJECT FK ORPHAN CHECKS
# =============================================================================

pipeline_run_orphan_object_count = (
    data_object_fk_orphan_count(
        GOLD_FACT_PIPELINE_RUN_DF
    )
)


dq_orphan_object_count = (
    data_object_fk_orphan_count(
        GOLD_FACT_DATA_QUALITY_DF
    )
)


cdc_orphan_object_count = (
    data_object_fk_orphan_count(
        GOLD_FACT_CDC_ACTIVITY_DF
    )
)


freshness_orphan_object_count = (
    data_object_fk_orphan_count(
        GOLD_AGG_SOURCE_FRESHNESS_DF
    )
)


# =============================================================================
# 14. REQUIRED PIPELINE IDENTIFIERS
# =============================================================================

null_pipeline_run_id_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "pipeline_run_id"
        ).isNull()
    )

    .count()
)


null_pipeline_name_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "pipeline_name"
        ).isNull()
    )

    .count()
)


null_source_system_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "source_system"
        ).isNull()
    )

    .count()
)


null_source_object_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "source_object"
        ).isNull()
    )

    .count()
)


# =============================================================================
# 15. TIMESTAMP / DURATION VALIDATION
# =============================================================================

negative_duration_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "duration_seconds"
        ) < 0
    )

    .count()
)


completion_before_start_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "run_completed_utc"
        )
        <
        F.col(
            "run_started_utc"
        )
    )

    .count()
)


cdc_negative_duration_count = (
    GOLD_FACT_CDC_ACTIVITY_DF

    .filter(
        F.col(
            "extraction_duration_seconds"
        ) < 0
    )

    .count()
)


# =============================================================================
# 16. ZERO-DURATION MONITORING
#
# Zero durations are not treated as a critical failure because historical
# generic audit records were initially logged with identical start/end
# timestamps.
#
# The count is retained as an operational quality indicator.
# =============================================================================

zero_duration_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.coalesce(
            F.col(
                "duration_seconds"
            ),
            F.lit(0)
        ) == 0
    )

    .count()
)


# =============================================================================
# 17. RUN STATUS VALIDATION
# =============================================================================

ALLOWED_PIPELINE_STATUSES = [
    "SUCCEEDED",
    "FAILED",
    "WARNING",
    "UNKNOWN"
]


invalid_pipeline_status_count = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "run_status"
        ).isNull()
        |
        (
            ~F.col(
                "run_status"
            ).isin(
                ALLOWED_PIPELINE_STATUSES
            )
        )
    )

    .count()
)


# =============================================================================
# 18. RUN STATUS DISTRIBUTION
# =============================================================================

PIPELINE_STATUS_SUMMARY_DF = (
    GOLD_FACT_PIPELINE_RUN_DF

    .groupBy(
        "run_status"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "run_object_count"
        ),

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "pipeline_run_count"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "rows_written"
                ),
                F.lit(0)
            )
        ).alias(
            "rows_written"
        )
    )

    .orderBy(
        "run_status"
    )
)


# =============================================================================
# 19. PIPELINE DISTRIBUTION
# =============================================================================

PIPELINE_EXECUTION_SUMMARY_DF = (
    GOLD_FACT_PIPELINE_RUN_DF

    .groupBy(
        "pipeline_key",
        "pipeline_name",
        "source_system",
        "source_company"
    )

    .agg(

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "pipeline_run_count"
        ),

        F.count(
            "*"
        ).alias(
            "object_event_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "SUCCEEDED",
                1
            ).otherwise(0)
        ).alias(
            "succeeded_object_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_object_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "WARNING",
                1
            ).otherwise(0)
        ).alias(
            "warning_object_count"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "rows_written"
                ),
                F.lit(0)
            )
        ).alias(
            "rows_written"
        ),

        F.avg(
            "duration_seconds"
        ).alias(
            "average_duration_seconds"
        )
    )

    .orderBy(
        "source_system",
        "pipeline_name"
    )
)


# =============================================================================
# 20. DATA QUALITY STATUS DISTRIBUTION
# =============================================================================

DQ_STATUS_SUMMARY_DF = (
    GOLD_FACT_DATA_QUALITY_DF

    .groupBy(
        "status"
    )

    .agg(
        F.count(
            "*"
        ).alias(
            "check_count"
        )
    )

    .orderBy(
        F.col(
            "check_count"
        ).desc()
    )
)


# =============================================================================
# 21. SOURCE FRESHNESS VALIDATION
# =============================================================================

null_last_ingested_count = (
    GOLD_AGG_SOURCE_FRESHNESS_DF

    .filter(
        F.col(
            "last_ingested_utc"
        ).isNull()
    )

    .count()
)


invalid_freshness_status_count = (
    GOLD_AGG_SOURCE_FRESHNESS_DF

    .filter(
        F.col(
            "freshness_status"
        ).isNull()
        |
        (
            ~F.col(
                "freshness_status"
            ).isin(
                "FRESH",
                "WARNING",
                "STALE",
                "UNKNOWN"
            )
        )
    )

    .count()
)


negative_freshness_age_count = (
    GOLD_AGG_SOURCE_FRESHNESS_DF

    .filter(
        F.col(
            "freshness_age_hours"
        ) < 0
    )

    .count()
)


# =============================================================================
# 22. DISTINCT PIPELINE EXECUTIONS
# =============================================================================

distinct_pipeline_run_ids = (
    GOLD_FACT_PIPELINE_RUN_DF

    .select(
        "pipeline_run_id"
    )

    .filter(
        F.col(
            "pipeline_run_id"
        ).isNotNull()
    )

    .distinct()

    .count()
)


# =============================================================================
# 23. OPERATIONAL KPI SUMMARY
# =============================================================================

MONITORING_KPI_SUMMARY_DF = (
    GOLD_FACT_PIPELINE_RUN_DF

    .agg(

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "total_pipeline_runs"
        ),

        F.count(
            "*"
        ).alias(
            "run_object_events"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "SUCCEEDED",
                1
            ).otherwise(0)
        ).alias(
            "successful_run_objects"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_run_objects"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "WARNING",
                1
            ).otherwise(0)
        ).alias(
            "warning_run_objects"
        ),

        F.avg(
            "duration_seconds"
        ).alias(
            "average_duration_seconds"
        ),

        F.max(
            "duration_seconds"
        ).alias(
            "maximum_duration_seconds"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "rows_written"
                ),
                F.lit(0)
            )
        ).alias(
            "total_rows_written"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "records_changed"
                ),
                F.lit(0)
            )
        ).alias(
            "total_records_changed"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "records_upserted"
                ),
                F.lit(0)
            )
        ).alias(
            "total_records_upserted"
        )
    )
)


# =============================================================================
# 24. FAILED PIPELINE OBJECT SUMMARY
# =============================================================================

FAILED_PIPELINE_OBJECTS_DF = (
    GOLD_FACT_PIPELINE_RUN_DF

    .filter(
        F.col(
            "run_status"
        ) == "FAILED"
    )

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object",
        "run_started_utc",
        "run_completed_utc",
        "duration_seconds",
        "rows_written",
        "error_message"
    )

    .orderBy(
        F.col(
            "run_started_utc"
        ).desc_nulls_last()
    )
)


# =============================================================================
# 25. PRINT PRE-WRITE VALIDATION SUMMARY
# =============================================================================

print("=" * 110)
print("GOLD OPERATIONAL MONITORING — UNIFIED PRE-WRITE VALIDATION")
print("=" * 110)


for table_name, row_count in monitoring_counts.items():

    print(
        f"{table_name:<35} : "
        f"{row_count:,}"
    )


print("-" * 110)


print(
    f"Expected logical pipelines          : "
    f"{expected_logical_pipeline_count:,}"
)


print(
    f"Actual logical pipelines            : "
    f"{actual_logical_pipeline_count:,}"
)


print(
    f"Logical pipeline count difference   : "
    f"{logical_pipeline_count_difference:,}"
)


print(
    f"Distinct pipeline executions        : "
    f"{distinct_pipeline_run_ids:,}"
)


print("-" * 110)


print(
    f"DQ source rows                      : "
    f"{source_validation_rows:,}"
)


print(
    f"DQ Gold rows                        : "
    f"{gold_validation_rows:,}"
)


print(
    f"DQ source/Gold difference           : "
    f"{dq_row_count_difference:,}"
)


print(
    f"CDC source rows                     : "
    f"{source_cdc_rows:,}"
)


print(
    f"CDC Gold rows                       : "
    f"{gold_cdc_rows:,}"
)


print(
    f"CDC source/Gold difference          : "
    f"{cdc_row_count_difference:,}"
)


print(
    f"Generic pipeline grains missing     : "
    f"{generic_pipeline_missing_from_gold_count:,}"
)


print("-" * 110)


print(
    f"dim_pipeline duplicate keys         : "
    f"{pipeline_dimension_duplicate_count:,}"
)


print(
    f"dim_data_object duplicate keys      : "
    f"{data_object_duplicate_count:,}"
)


print(
    f"fact_pipeline_run duplicate keys    : "
    f"{pipeline_run_duplicate_count:,}"
)


print(
    f"fact_data_quality duplicate keys    : "
    f"{dq_duplicate_count:,}"
)


print(
    f"fact_cdc_activity duplicate keys    : "
    f"{cdc_duplicate_count:,}"
)


print(
    f"freshness duplicate keys            : "
    f"{freshness_duplicate_count:,}"
)


print("-" * 110)


print(
    f"dim_pipeline null keys              : "
    f"{pipeline_dimension_null_key_count:,}"
)


print(
    f"dim_data_object null keys           : "
    f"{data_object_null_key_count:,}"
)


print(
    f"fact_pipeline_run null keys         : "
    f"{pipeline_run_null_key_count:,}"
)


print(
    f"fact_data_quality null keys         : "
    f"{dq_null_key_count:,}"
)


print(
    f"fact_cdc_activity null keys         : "
    f"{cdc_null_key_count:,}"
)


print("-" * 110)


print(
    f"Pipeline → dim_pipeline orphans     : "
    f"{pipeline_run_orphan_pipeline_count:,}"
)


print(
    f"DQ → dim_pipeline orphans           : "
    f"{dq_orphan_pipeline_count:,}"
)


print(
    f"CDC → dim_pipeline orphans          : "
    f"{cdc_orphan_pipeline_count:,}"
)


print(
    f"Pipeline → dim_object orphans       : "
    f"{pipeline_run_orphan_object_count:,}"
)


print(
    f"DQ → dim_object orphans             : "
    f"{dq_orphan_object_count:,}"
)


print(
    f"CDC → dim_object orphans            : "
    f"{cdc_orphan_object_count:,}"
)


print(
    f"Freshness → dim_object orphans      : "
    f"{freshness_orphan_object_count:,}"
)


print("-" * 110)


print(
    f"Null pipeline_run_id values         : "
    f"{null_pipeline_run_id_count:,}"
)


print(
    f"Null pipeline_name values           : "
    f"{null_pipeline_name_count:,}"
)


print(
    f"Null source_system values           : "
    f"{null_source_system_count:,}"
)


print(
    f"Null source_object values           : "
    f"{null_source_object_count:,}"
)


print("-" * 110)


print(
    f"Negative pipeline durations         : "
    f"{negative_duration_count:,}"
)


print(
    f"Completion before start             : "
    f"{completion_before_start_count:,}"
)


print(
    f"Negative CDC durations              : "
    f"{cdc_negative_duration_count:,}"
)


print(
    f"Zero-duration pipeline events       : "
    f"{zero_duration_count:,}"
)


print(
    f"Invalid pipeline statuses           : "
    f"{invalid_pipeline_status_count:,}"
)


print("-" * 110)


print(
    f"Null freshness timestamps           : "
    f"{null_last_ingested_count:,}"
)


print(
    f"Invalid freshness statuses          : "
    f"{invalid_freshness_status_count:,}"
)


print(
    f"Negative freshness ages             : "
    f"{negative_freshness_age_count:,}"
)


print("=" * 110)


# =============================================================================
# 26. BUILD CRITICAL FAILURE LIST
# =============================================================================

monitoring_validation_failures = []


if logical_pipeline_count_difference != 0:

    monitoring_validation_failures.append(
        "LOGICAL_PIPELINE_RECONCILIATION"
    )


if dq_row_count_difference != 0:

    monitoring_validation_failures.append(
        "DQ_ROW_RECONCILIATION"
    )


if cdc_row_count_difference != 0:

    monitoring_validation_failures.append(
        "CDC_ROW_RECONCILIATION"
    )


if generic_pipeline_missing_from_gold_count > 0:

    monitoring_validation_failures.append(
        "GENERIC_PIPELINE_GRAIN_RECONCILIATION"
    )


if pipeline_dimension_duplicate_count > 0:

    monitoring_validation_failures.append(
        "PIPELINE_DIMENSION_DUPLICATE"
    )


if data_object_duplicate_count > 0:

    monitoring_validation_failures.append(
        "DATA_OBJECT_DUPLICATE"
    )


if pipeline_run_duplicate_count > 0:

    monitoring_validation_failures.append(
        "PIPELINE_RUN_DUPLICATE"
    )


if dq_duplicate_count > 0:

    monitoring_validation_failures.append(
        "DQ_DUPLICATE"
    )


if cdc_duplicate_count > 0:

    monitoring_validation_failures.append(
        "CDC_DUPLICATE"
    )


if freshness_duplicate_count > 0:

    monitoring_validation_failures.append(
        "FRESHNESS_DUPLICATE"
    )


if pipeline_dimension_null_key_count > 0:

    monitoring_validation_failures.append(
        "PIPELINE_DIMENSION_NULL_KEY"
    )


if data_object_null_key_count > 0:

    monitoring_validation_failures.append(
        "DATA_OBJECT_NULL_KEY"
    )


if pipeline_run_null_key_count > 0:

    monitoring_validation_failures.append(
        "PIPELINE_RUN_NULL_KEY"
    )


if dq_null_key_count > 0:

    monitoring_validation_failures.append(
        "DQ_NULL_KEY"
    )


if cdc_null_key_count > 0:

    monitoring_validation_failures.append(
        "CDC_NULL_KEY"
    )


if pipeline_run_orphan_pipeline_count > 0:

    monitoring_validation_failures.append(
        "PIPELINE_RUN_ORPHAN_PIPELINE"
    )


if dq_orphan_pipeline_count > 0:

    monitoring_validation_failures.append(
        "DQ_ORPHAN_PIPELINE"
    )


if cdc_orphan_pipeline_count > 0:

    monitoring_validation_failures.append(
        "CDC_ORPHAN_PIPELINE"
    )


if pipeline_run_orphan_object_count > 0:

    monitoring_validation_failures.append(
        "PIPELINE_RUN_ORPHAN_OBJECT"
    )


if dq_orphan_object_count > 0:

    monitoring_validation_failures.append(
        "DQ_ORPHAN_OBJECT"
    )


if cdc_orphan_object_count > 0:

    monitoring_validation_failures.append(
        "CDC_ORPHAN_OBJECT"
    )


if freshness_orphan_object_count > 0:

    monitoring_validation_failures.append(
        "FRESHNESS_ORPHAN_OBJECT"
    )


if null_pipeline_run_id_count > 0:

    monitoring_validation_failures.append(
        "NULL_PIPELINE_RUN_ID"
    )


if null_pipeline_name_count > 0:

    monitoring_validation_failures.append(
        "NULL_PIPELINE_NAME"
    )


if null_source_system_count > 0:

    monitoring_validation_failures.append(
        "NULL_SOURCE_SYSTEM"
    )


if null_source_object_count > 0:

    monitoring_validation_failures.append(
        "NULL_SOURCE_OBJECT"
    )


if negative_duration_count > 0:

    monitoring_validation_failures.append(
        "NEGATIVE_PIPELINE_DURATION"
    )


if completion_before_start_count > 0:

    monitoring_validation_failures.append(
        "INVALID_PIPELINE_TIMESTAMPS"
    )


if cdc_negative_duration_count > 0:

    monitoring_validation_failures.append(
        "NEGATIVE_CDC_DURATION"
    )


if invalid_pipeline_status_count > 0:

    monitoring_validation_failures.append(
        "INVALID_PIPELINE_STATUS"
    )


if invalid_freshness_status_count > 0:

    monitoring_validation_failures.append(
        "INVALID_FRESHNESS_STATUS"
    )


if negative_freshness_age_count > 0:

    monitoring_validation_failures.append(
        "NEGATIVE_FRESHNESS_AGE"
    )


# =============================================================================
# 27. FAIL FAST
# =============================================================================

if monitoring_validation_failures:

    raise RuntimeError(
        "GOLD OPERATIONAL MONITORING PRE-WRITE VALIDATION FAILED: "
        +
        ", ".join(
            monitoring_validation_failures
        )
    )


# =============================================================================
# 28. DISPLAY LOGICAL PIPELINES
# =============================================================================

print("=" * 110)
print("LOGICAL PIPELINES")
print("=" * 110)

display(
    GOLD_DIM_PIPELINE_DF

    .select(
        "pipeline_key",
        "pipeline_name",
        "pipeline_layer",
        "pipeline_type",
        "source_system",
        "source_company",
        "is_active"
    )

    .orderBy(
        "source_system",
        "pipeline_name"
    )
)


# =============================================================================
# 29. DISPLAY PIPELINE STATUS DISTRIBUTION
# =============================================================================

print("=" * 110)
print("PIPELINE STATUS DISTRIBUTION")
print("=" * 110)

display(
    PIPELINE_STATUS_SUMMARY_DF
)


# =============================================================================
# 30. DISPLAY PIPELINE EXECUTION SUMMARY
# =============================================================================

print("=" * 110)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 110)

display(
    PIPELINE_EXECUTION_SUMMARY_DF
)


# =============================================================================
# 31. DISPLAY FAILED PIPELINE OBJECTS
# =============================================================================

print("=" * 110)
print("FAILED PIPELINE OBJECTS")
print("=" * 110)

display(
    FAILED_PIPELINE_OBJECTS_DF
)


# =============================================================================
# 32. DISPLAY DATA QUALITY STATUS
# =============================================================================

print("=" * 110)
print("DATA QUALITY STATUS DISTRIBUTION")
print("=" * 110)

display(
    DQ_STATUS_SUMMARY_DF
)


# =============================================================================
# 33. DISPLAY MONITORING KPI SUMMARY
# =============================================================================

print("=" * 110)
print("MONITORING KPI SUMMARY")
print("=" * 110)

display(
    MONITORING_KPI_SUMMARY_DF
)


# =============================================================================
# 34. FINAL STATUS
# =============================================================================

print("=" * 110)

print(
    "GOLD OPERATIONAL MONITORING — "
    "UNIFIED PRE-WRITE VALIDATION: SUCCEEDED"
)

print("=" * 110)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 17, Finished, Available, Finished, False)

GOLD OPERATIONAL MONITORING — UNIFIED PRE-WRITE VALIDATION
dim_pipeline                        : 3
dim_data_object                     : 69
fact_pipeline_run                   : 578
fact_data_quality                   : 3,586
fact_cdc_activity                   : 4,790
agg_source_freshness                : 36
--------------------------------------------------------------------------------------------------------------
Expected logical pipelines          : 3
Actual logical pipelines            : 3
Logical pipeline count difference   : 0
Distinct pipeline executions        : 40
--------------------------------------------------------------------------------------------------------------
DQ source rows                      : 3,586
DQ Gold rows                        : 3,586
DQ source/Gold difference           : 0
CDC source rows                     : 4,790
CDC Gold rows                       : 4,790
CDC source/Gold difference          : 0
Generic pipeline grains missing     : 0
----------

SynapseWidget(Synapse.DataFrame, aa303237-753e-4264-8d9f-0b5b0a8a9a37)

PIPELINE STATUS DISTRIBUTION


SynapseWidget(Synapse.DataFrame, 1b8e6e8d-23c0-4c3d-837a-870ae7022b44)

...

DATA QUALITY STATUS DISTRIBUTION


SynapseWidget(Synapse.DataFrame, 722f6aff-4ef1-4475-ba73-7aa11336b81c)

MONITORING KPI SUMMARY


SynapseWidget(Synapse.DataFrame, 006af63a-2255-4dc5-a91e-4f624a78a910)

GOLD OPERATIONAL MONITORING — UNIFIED PRE-WRITE VALIDATION: SUCCEEDED


In [16]:
# =============================================================================
# CELL 15 — GOLD OPERATIONAL MONITORING
# Persist Unified Multi-Source Monitoring Model
#
# PURPOSE
# -------
# Persist the validated monitoring DataFrames produced by Cells 13–14.
#
# Current monitoring coverage:
#   - QuickBooks Spain incremental ingestion
#   - SAP Business One HANA UK ingestion
#   - Czech ERP ingestion
#
# Gold tables:
#   - dim_pipeline
#   - dim_data_object
#   - fact_pipeline_run
#   - fact_data_quality
#   - fact_cdc_activity
#   - agg_source_freshness
#
# IMPORTANT
# ---------
# Cell 14 must have completed successfully before this cell is run.
#
# Write strategy:
#   1. Dimensions
#   2. Facts
#   3. Aggregate
#
# Each persisted table is reconciled against its validated pre-write
# DataFrame row count.
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. DEFINE GOLD MONITORING TABLES IN CONTROLLED WRITE ORDER
# =============================================================================

MONITORING_GOLD_TABLES = [

    (
        "dim_pipeline",
        GOLD_DIM_PIPELINE_DF
    ),

    (
        "dim_data_object",
        GOLD_DIM_DATA_OBJECT_DF
    ),

    (
        "fact_pipeline_run",
        GOLD_FACT_PIPELINE_RUN_DF
    ),

    (
        "fact_data_quality",
        GOLD_FACT_DATA_QUALITY_DF
    ),

    (
        "fact_cdc_activity",
        GOLD_FACT_CDC_ACTIVITY_DF
    ),

    (
        "agg_source_freshness",
        GOLD_AGG_SOURCE_FRESHNESS_DF
    )
]


# =============================================================================
# 2. CAPTURE VALIDATED PRE-WRITE ROW COUNTS
# =============================================================================

EXPECTED_MONITORING_COUNTS = {
    table_name: df.count()
    for table_name, df in MONITORING_GOLD_TABLES
}


# =============================================================================
# 3. PRE-WRITE SAFETY CHECK
#
# Prevent an accidental empty DataFrame from overwriting an existing Gold
# monitoring table.
# =============================================================================

empty_monitoring_tables = [

    table_name

    for table_name, row_count
    in EXPECTED_MONITORING_COUNTS.items()

    if row_count == 0
]


if empty_monitoring_tables:

    raise RuntimeError(
        "GOLD MONITORING WRITE ABORTED. "
        "The following validated DataFrames are empty: "
        +
        ", ".join(
            empty_monitoring_tables
        )
    )


# =============================================================================
# 4. DISPLAY PRE-WRITE COUNTS
# =============================================================================

print("=" * 110)
print("GOLD OPERATIONAL MONITORING — PRE-WRITE COUNTS")
print("=" * 110)


for table_name, expected_rows in EXPECTED_MONITORING_COUNTS.items():

    print(
        f"{table_name:<30} : "
        f"{expected_rows:,}"
    )


print("=" * 110)


# =============================================================================
# 5. WRITE GOLD MONITORING TABLES
# =============================================================================

write_results = []


print("=" * 110)
print("GOLD OPERATIONAL MONITORING — TABLE WRITE")
print("=" * 110)


for table_name, df in MONITORING_GOLD_TABLES:

    target_table = (
        f"{GOLD_LAKEHOUSE}."
        f"{GOLD_SCHEMA}."
        f"{table_name}"
    )

    expected_rows = (
        EXPECTED_MONITORING_COUNTS[
            table_name
        ]
    )

    print(
        f"Writing {target_table} ..."
    )


    (
        df

        .write

        .format(
            "delta"
        )

        .mode(
            "overwrite"
        )

        .option(
            "overwriteSchema",
            "true"
        )

        .saveAsTable(
            target_table
        )
    )


    actual_rows = (
        spark.table(
            target_table
        )
        .count()
    )


    status = (
        "SUCCEEDED"
        if actual_rows == expected_rows
        else "FAILED"
    )


    write_results.append(
        {
            "table_name":
                table_name,

            "target_table":
                target_table,

            "expected_rows":
                expected_rows,

            "actual_rows":
                actual_rows,

            "status":
                status
        }
    )


# =============================================================================
# 6. DISPLAY WRITE RESULTS
# =============================================================================

print("=" * 110)
print("GOLD MONITORING TABLE WRITE RESULTS")
print("=" * 110)


for result in write_results:

    print(
        f"{result['table_name']:<30} | "
        f"expected={result['expected_rows']:<8,} | "
        f"actual={result['actual_rows']:<8,} | "
        f"{result['status']}"
    )


print("=" * 110)


# =============================================================================
# 7. FAIL ON ROW-COUNT RECONCILIATION ERROR
# =============================================================================

failed_writes = [

    result

    for result in write_results

    if result[
        "status"
    ] != "SUCCEEDED"
]


if failed_writes:

    raise RuntimeError(
        "GOLD MONITORING TABLE WRITE FAILED: "
        +
        ", ".join(
            result[
                "table_name"
            ]
            for result in failed_writes
        )
    )


# =============================================================================
# 8. DISCOVER PERSISTED GOLD TABLES
# =============================================================================

gold_tables_discovered = [

    row[
        "tableName"
    ]

    for row in spark.sql(
        f"""
        SHOW TABLES IN
        {GOLD_LAKEHOUSE}.{GOLD_SCHEMA}
        """
    ).collect()

    if not row[
        "isTemporary"
    ]
]


expected_monitoring_table_names = [

    table_name

    for table_name, _
    in MONITORING_GOLD_TABLES
]


missing_monitoring_tables = [

    table_name

    for table_name
    in expected_monitoring_table_names

    if table_name
    not in gold_tables_discovered
]


if missing_monitoring_tables:

    raise RuntimeError(
        "Persisted monitoring table(s) "
        "not discoverable after write: "
        +
        ", ".join(
            missing_monitoring_tables
        )
    )


# =============================================================================
# 9. RELOAD PERSISTED GOLD MONITORING TABLES
#
# Important:
# Subsequent cells should validate the persisted tables, not only the
# in-memory DataFrames.
# =============================================================================

DIM_PIPELINE_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}."
    f"{GOLD_SCHEMA}."
    f"dim_pipeline"
)


DIM_DATA_OBJECT_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}."
    f"{GOLD_SCHEMA}."
    f"dim_data_object"
)


FACT_PIPELINE_RUN_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}."
    f"{GOLD_SCHEMA}."
    f"fact_pipeline_run"
)


FACT_DATA_QUALITY_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}."
    f"{GOLD_SCHEMA}."
    f"fact_data_quality"
)


FACT_CDC_ACTIVITY_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}."
    f"{GOLD_SCHEMA}."
    f"fact_cdc_activity"
)


AGG_SOURCE_FRESHNESS_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}."
    f"{GOLD_SCHEMA}."
    f"agg_source_freshness"
)


# =============================================================================
# 10. POST-WRITE ROW COUNTS
# =============================================================================

POST_WRITE_COUNTS = {

    "dim_pipeline":
        DIM_PIPELINE_GOLD_DF.count(),

    "dim_data_object":
        DIM_DATA_OBJECT_GOLD_DF.count(),

    "fact_pipeline_run":
        FACT_PIPELINE_RUN_GOLD_DF.count(),

    "fact_data_quality":
        FACT_DATA_QUALITY_GOLD_DF.count(),

    "fact_cdc_activity":
        FACT_CDC_ACTIVITY_GOLD_DF.count(),

    "agg_source_freshness":
        AGG_SOURCE_FRESHNESS_GOLD_DF.count()
}


# =============================================================================
# 11. FINAL PERSISTENCE RECONCILIATION
# =============================================================================

post_write_reconciliation_failures = []


for table_name in EXPECTED_MONITORING_COUNTS:

    expected_rows = (
        EXPECTED_MONITORING_COUNTS[
            table_name
        ]
    )

    actual_rows = (
        POST_WRITE_COUNTS[
            table_name
        ]
    )

    if expected_rows != actual_rows:

        post_write_reconciliation_failures.append(
            table_name
        )


if post_write_reconciliation_failures:

    raise RuntimeError(
        "POST-WRITE GOLD MONITORING "
        "ROW RECONCILIATION FAILED: "
        +
        ", ".join(
            post_write_reconciliation_failures
        )
    )


# =============================================================================
# 12. VERIFY EXPECTED LOGICAL PIPELINES AFTER WRITE
# =============================================================================

persisted_pipeline_count = (
    DIM_PIPELINE_GOLD_DF.count()
)


if (
    persisted_pipeline_count
    !=
    expected_logical_pipeline_count
):

    raise RuntimeError(
        "Persisted dim_pipeline count does not match "
        "the validated logical pipeline count. "
        f"Expected={expected_logical_pipeline_count}, "
        f"Actual={persisted_pipeline_count}"
    )


# =============================================================================
# 13. VERIFY FACT → PIPELINE REFERENTIAL INTEGRITY AFTER WRITE
# =============================================================================

post_write_pipeline_orphans = (

    FACT_PIPELINE_RUN_GOLD_DF

    .select(
        "pipeline_key"
    )

    .filter(
        F.col(
            "pipeline_key"
        ).isNotNull()
    )

    .distinct()

    .join(

        DIM_PIPELINE_GOLD_DF

        .select(
            "pipeline_key"
        )

        .distinct(),

        on="pipeline_key",

        how="left_anti"
    )

    .count()
)


if post_write_pipeline_orphans > 0:

    raise RuntimeError(
        "Persisted fact_pipeline_run contains "
        f"{post_write_pipeline_orphans} "
        "pipeline_key orphan(s)."
    )


# =============================================================================
# 14. VERIFY FACT → DATA OBJECT REFERENTIAL INTEGRITY AFTER WRITE
# =============================================================================

post_write_object_orphans = (

    FACT_PIPELINE_RUN_GOLD_DF

    .select(
        "data_object_key"
    )

    .filter(
        F.col(
            "data_object_key"
        ).isNotNull()
    )

    .distinct()

    .join(

        DIM_DATA_OBJECT_GOLD_DF

        .select(
            "data_object_key"
        )

        .distinct(),

        on="data_object_key",

        how="left_anti"
    )

    .count()
)


if post_write_object_orphans > 0:

    raise RuntimeError(
        "Persisted fact_pipeline_run contains "
        f"{post_write_object_orphans} "
        "data_object_key orphan(s)."
    )


# =============================================================================
# 15. PRINT FINAL WRITE SUMMARY
# =============================================================================

print("=" * 110)
print("GOLD OPERATIONAL MONITORING — WRITE SUMMARY")
print("=" * 110)


print(
    f"Monitoring tables expected      : "
    f"{len(MONITORING_GOLD_TABLES)}"
)


print(
    f"Monitoring tables written       : "
    f"{len(write_results)}"
)


print(
    f"Write failures                  : "
    f"{len(failed_writes)}"
)


print(
    f"Missing after write             : "
    f"{len(missing_monitoring_tables)}"
)


print(
    f"Logical pipelines persisted     : "
    f"{persisted_pipeline_count}"
)


print(
    f"Pipeline FK orphans after write : "
    f"{post_write_pipeline_orphans}"
)


print(
    f"Object FK orphans after write   : "
    f"{post_write_object_orphans}"
)


print("-" * 110)


for table_name, actual_rows in POST_WRITE_COUNTS.items():

    print(
        f"{table_name:<30} : "
        f"{actual_rows:,} rows"
    )


print("=" * 110)


# =============================================================================
# 16. DISPLAY PERSISTED PIPELINE DIMENSION
# =============================================================================

print("=" * 110)
print("PERSISTED DIM_PIPELINE")
print("=" * 110)


display(

    DIM_PIPELINE_GOLD_DF

    .orderBy(
        "source_system",
        "pipeline_name"
    )
)


# =============================================================================
# 17. DISPLAY LATEST PERSISTED PIPELINE EVENTS
# =============================================================================

print("=" * 110)
print("LATEST PERSISTED PIPELINE EVENTS")
print("=" * 110)


display(

    FACT_PIPELINE_RUN_GOLD_DF

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object",
        "run_started_utc",
        "run_completed_utc",
        "duration_seconds",
        "run_status",
        "rows_written",
        "error_message"
    )

    .orderBy(
        F.col(
            "run_started_utc"
        ).desc_nulls_last()
    )

    .limit(
        100
    )
)


# =============================================================================
# 18. FINAL STATUS
# =============================================================================

print("=" * 110)

print(
    "GOLD OPERATIONAL MONITORING — "
    "UNIFIED TABLE WRITE: SUCCEEDED"
)

print("=" * 110)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 18, Finished, Available, Finished, False)

GOLD OPERATIONAL MONITORING — PRE-WRITE COUNTS
dim_pipeline                   : 3
dim_data_object                : 69
fact_pipeline_run              : 578
fact_data_quality              : 3,586
fact_cdc_activity              : 4,790
agg_source_freshness           : 36
GOLD OPERATIONAL MONITORING — TABLE WRITE
Writing lh_global_finance_gold.dbo.dim_pipeline ...
Writing lh_global_finance_gold.dbo.dim_data_object ...
Writing lh_global_finance_gold.dbo.fact_pipeline_run ...
Writing lh_global_finance_gold.dbo.fact_data_quality ...
Writing lh_global_finance_gold.dbo.fact_cdc_activity ...
Writing lh_global_finance_gold.dbo.agg_source_freshness ...
GOLD MONITORING TABLE WRITE RESULTS
dim_pipeline                   | expected=3        | actual=3        | SUCCEEDED
dim_data_object                | expected=69       | actual=69       | SUCCEEDED
fact_pipeline_run              | expected=578      | actual=578      | SUCCEEDED
fact_data_quality              | expected=3,586    | actual=3,586    | S

SynapseWidget(Synapse.DataFrame, c4c49d25-2251-4f9b-a208-ec56a73e57e3)

LATEST PERSISTED PIPELINE EVENTS


SynapseWidget(Synapse.DataFrame, daa6079e-aabf-4ece-a154-3d46005dde75)

GOLD OPERATIONAL MONITORING — UNIFIED TABLE WRITE: SUCCEEDED


In [17]:
# =============================================================================
# CELL 16 — GOLD OPERATIONAL MONITORING
# Unified Multi-Source Final Post-Load Validation
#
# PURPOSE
# -------
# Perform final validation of the monitoring tables persisted by Cell 15.
#
# IMPORTANT
# ---------
# This cell DOES NOT:
#   - rebuild dimensions
#   - re-key facts
#   - overwrite Gold tables
#   - force QBO-only source-system codes
#
# It validates the persisted multi-source model exactly as written.
#
# Current monitoring coverage:
#   - QuickBooks Spain incremental ingestion
#   - SAP Business One HANA UK ingestion
#   - Czech ERP ingestion
#
# Persisted tables:
#   - dim_pipeline
#   - dim_data_object
#   - fact_pipeline_run
#   - fact_data_quality
#   - fact_cdc_activity
#   - agg_source_freshness
# =============================================================================

from pyspark.sql import functions as F


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

BRONZE_LAKEHOUSE = "lh_global_finance_bronze"
GOLD_LAKEHOUSE   = "lh_global_finance_gold"

BRONZE_SCHEMA = "dbo"
GOLD_SCHEMA   = "dbo"


# =============================================================================
# 2. RELOAD PERSISTED GOLD MONITORING TABLES
# =============================================================================

DIM_PIPELINE_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.dim_pipeline"
)

DIM_DATA_OBJECT_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.dim_data_object"
)

FACT_PIPELINE_RUN_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_pipeline_run"
)

FACT_DATA_QUALITY_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_data_quality"
)

FACT_CDC_ACTIVITY_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.fact_cdc_activity"
)

AGG_SOURCE_FRESHNESS_GOLD_DF = spark.table(
    f"{GOLD_LAKEHOUSE}.{GOLD_SCHEMA}.agg_source_freshness"
)


# =============================================================================
# 3. RELOAD BRONZE AUDIT SOURCES FOR RECONCILIATION
# =============================================================================

PLATFORM_INGESTION_AUDIT_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.platform_ingestion_audit"
)

INGESTION_AUDIT_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_ingestion_audit"
)

VALIDATION_AUDIT_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_incremental_validation_audit"
)

CDC_CHANGE_LOG_DF = spark.table(
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.qbo_es_cdc_change_log"
)


# =============================================================================
# 4. STANDARDISE GENERIC PLATFORM AUDIT
#
# QBO remains excluded because it has its own specialist monitoring model.
# =============================================================================

GENERIC_PLATFORM_AUDIT_DF = (
    PLATFORM_INGESTION_AUDIT_DF

    .filter(
        ~F.upper(
            F.coalesce(
                F.col("source_system"),
                F.lit("")
            )
        ).isin(
            "QBO",
            "QBO_ES",
            "QUICKBOOKS",
            "QUICKBOOKS_ES"
        )
    )

    .withColumn(
        "source_system",
        F.upper(
            F.trim(
                F.col("source_system")
            )
        )
    )

    .withColumn(
        "source_company",
        F.upper(
            F.trim(
                F.col("source_company")
            )
        )
    )

    .withColumn(
        "pipeline_name",
        F.trim(
            F.col("pipeline_name")
        )
    )

    .withColumn(
        "source_object",
        F.trim(
            F.col("source_object")
        )
    )
)


# =============================================================================
# 5. ACTUAL PERSISTED ROW COUNTS
# =============================================================================

ACTUAL_COUNTS = {

    "dim_pipeline":
        DIM_PIPELINE_GOLD_DF.count(),

    "dim_data_object":
        DIM_DATA_OBJECT_GOLD_DF.count(),

    "fact_pipeline_run":
        FACT_PIPELINE_RUN_GOLD_DF.count(),

    "fact_data_quality":
        FACT_DATA_QUALITY_GOLD_DF.count(),

    "fact_cdc_activity":
        FACT_CDC_ACTIVITY_GOLD_DF.count(),

    "agg_source_freshness":
        AGG_SOURCE_FRESHNESS_GOLD_DF.count()
}


# =============================================================================
# 6. EXPECTED COUNTS
#
# Prefer validated counts carried forward from Cell 15.
# If the notebook was restarted, fall back to persisted counts for the
# structural tables while independently reconciling source-grain facts below.
# =============================================================================

if "EXPECTED_MONITORING_COUNTS" in globals():

    EXPECTED_COUNTS = dict(
        EXPECTED_MONITORING_COUNTS
    )

else:

    EXPECTED_COUNTS = {

        "dim_pipeline":
            ACTUAL_COUNTS["dim_pipeline"],

        "dim_data_object":
            ACTUAL_COUNTS["dim_data_object"],

        "fact_pipeline_run":
            ACTUAL_COUNTS["fact_pipeline_run"],

        "fact_data_quality":
            VALIDATION_AUDIT_DF.count(),

        "fact_cdc_activity":
            CDC_CHANGE_LOG_DF.count(),

        "agg_source_freshness":
            ACTUAL_COUNTS["agg_source_freshness"]
    }


# =============================================================================
# 7. ROW-COUNT RECONCILIATION
# =============================================================================

row_count_failures = {

    table_name: {
        "expected": EXPECTED_COUNTS[table_name],
        "actual": ACTUAL_COUNTS[table_name]
    }

    for table_name in EXPECTED_COUNTS

    if (
        EXPECTED_COUNTS[table_name]
        !=
        ACTUAL_COUNTS[table_name]
    )
}


# =============================================================================
# 8. EXPECTED LOGICAL PIPELINES — DYNAMIC
# =============================================================================

generic_logical_pipeline_count = (
    GENERIC_PLATFORM_AUDIT_DF

    .select(
        "pipeline_name",
        "source_system",
        "source_company"
    )

    .filter(
        F.col("pipeline_name").isNotNull()
    )

    .distinct()

    .count()
)


qbo_logical_pipeline_count = (
    1
    if INGESTION_AUDIT_DF.limit(1).count() > 0
    else 0
)


expected_logical_pipeline_count = (
    generic_logical_pipeline_count
    +
    qbo_logical_pipeline_count
)


actual_logical_pipeline_count = (
    DIM_PIPELINE_GOLD_DF.count()
)


logical_pipeline_difference = (
    actual_logical_pipeline_count
    -
    expected_logical_pipeline_count
)


# =============================================================================
# 9. PRIMARY KEY DUPLICATE HELPER
# =============================================================================

def duplicate_count(
    df,
    key_columns
):

    return (
        df

        .groupBy(
            *key_columns
        )

        .count()

        .filter(
            F.col("count") > 1
        )

        .count()
    )


# =============================================================================
# 10. DUPLICATE KEY VALIDATION
# =============================================================================

duplicate_failures = {

    "dim_pipeline":
        duplicate_count(
            DIM_PIPELINE_GOLD_DF,
            ["pipeline_key"]
        ),

    "dim_data_object":
        duplicate_count(
            DIM_DATA_OBJECT_GOLD_DF,
            ["data_object_key"]
        ),

    "fact_pipeline_run":
        duplicate_count(
            FACT_PIPELINE_RUN_GOLD_DF,
            ["pipeline_run_key"]
        ),

    "fact_data_quality":
        duplicate_count(
            FACT_DATA_QUALITY_GOLD_DF,
            ["data_quality_key"]
        ),

    "fact_cdc_activity":
        duplicate_count(
            FACT_CDC_ACTIVITY_GOLD_DF,
            ["cdc_activity_key"]
        ),

    "agg_source_freshness":
        duplicate_count(
            AGG_SOURCE_FRESHNESS_GOLD_DF,
            ["data_object_key"]
        )
}


# =============================================================================
# 11. NULL PRIMARY KEY HELPER
# =============================================================================

def null_key_count(
    df,
    key_column
):

    return (
        df

        .filter(
            F.col(
                key_column
            ).isNull()
        )

        .count()
    )


# =============================================================================
# 12. NULL PRIMARY KEY VALIDATION
# =============================================================================

null_key_failures = {

    "dim_pipeline":
        null_key_count(
            DIM_PIPELINE_GOLD_DF,
            "pipeline_key"
        ),

    "dim_data_object":
        null_key_count(
            DIM_DATA_OBJECT_GOLD_DF,
            "data_object_key"
        ),

    "fact_pipeline_run":
        null_key_count(
            FACT_PIPELINE_RUN_GOLD_DF,
            "pipeline_run_key"
        ),

    "fact_data_quality":
        null_key_count(
            FACT_DATA_QUALITY_GOLD_DF,
            "data_quality_key"
        ),

    "fact_cdc_activity":
        null_key_count(
            FACT_CDC_ACTIVITY_GOLD_DF,
            "cdc_activity_key"
        )
}


# =============================================================================
# 13. PIPELINE DIMENSION KEY INVENTORY
# =============================================================================

PIPELINE_KEYS_DF = (
    DIM_PIPELINE_GOLD_DF

    .select(
        "pipeline_key"
    )

    .distinct()
)


# =============================================================================
# 14. PIPELINE FK ORPHAN HELPER
# =============================================================================

def pipeline_orphan_count(df):

    return (
        df

        .select(
            "pipeline_key"
        )

        .filter(
            F.col(
                "pipeline_key"
            ).isNotNull()
        )

        .distinct()

        .join(
            PIPELINE_KEYS_DF,
            on="pipeline_key",
            how="left_anti"
        )

        .count()
    )


# =============================================================================
# 15. PIPELINE FK VALIDATION
# =============================================================================

pipeline_run_pipeline_orphans = (
    pipeline_orphan_count(
        FACT_PIPELINE_RUN_GOLD_DF
    )
)

dq_pipeline_orphans = (
    pipeline_orphan_count(
        FACT_DATA_QUALITY_GOLD_DF
    )
)

cdc_pipeline_orphans = (
    pipeline_orphan_count(
        FACT_CDC_ACTIVITY_GOLD_DF
    )
)


total_pipeline_fk_orphans = (
    pipeline_run_pipeline_orphans
    +
    dq_pipeline_orphans
    +
    cdc_pipeline_orphans
)


# =============================================================================
# 16. DATA-OBJECT DIMENSION KEY INVENTORY
# =============================================================================

DATA_OBJECT_KEYS_DF = (
    DIM_DATA_OBJECT_GOLD_DF

    .select(
        "data_object_key"
    )

    .distinct()
)


# =============================================================================
# 17. DATA-OBJECT FK ORPHAN HELPER
# =============================================================================

def data_object_orphan_count(df):

    return (
        df

        .select(
            "data_object_key"
        )

        .filter(
            F.col(
                "data_object_key"
            ).isNotNull()
        )

        .distinct()

        .join(
            DATA_OBJECT_KEYS_DF,
            on="data_object_key",
            how="left_anti"
        )

        .count()
    )


# =============================================================================
# 18. DATA-OBJECT FK VALIDATION
# =============================================================================

pipeline_object_orphans = (
    data_object_orphan_count(
        FACT_PIPELINE_RUN_GOLD_DF
    )
)

dq_object_orphans = (
    data_object_orphan_count(
        FACT_DATA_QUALITY_GOLD_DF
    )
)

cdc_object_orphans = (
    data_object_orphan_count(
        FACT_CDC_ACTIVITY_GOLD_DF
    )
)

freshness_object_orphans = (
    data_object_orphan_count(
        AGG_SOURCE_FRESHNESS_GOLD_DF
    )
)


total_data_object_fk_orphans = (
    pipeline_object_orphans
    +
    dq_object_orphans
    +
    cdc_object_orphans
    +
    freshness_object_orphans
)


# =============================================================================
# 19. QBO SOURCE-TO-GOLD RECONCILIATION
# =============================================================================

dq_source_rows = (
    VALIDATION_AUDIT_DF.count()
)


dq_gold_rows = (
    FACT_DATA_QUALITY_GOLD_DF.count()
)


dq_source_gold_difference = (
    dq_gold_rows
    -
    dq_source_rows
)


cdc_source_rows = (
    CDC_CHANGE_LOG_DF.count()
)


cdc_gold_rows = (
    FACT_CDC_ACTIVITY_GOLD_DF.count()
)


cdc_source_gold_difference = (
    cdc_gold_rows
    -
    cdc_source_rows
)


# =============================================================================
# 20. GENERIC PLATFORM AUDIT GRAIN RECONCILIATION
#
# Every generic pipeline-run/source-object grain must exist in persisted Gold.
# =============================================================================

GENERIC_EXPECTED_GRAIN_DF = (
    GENERIC_PLATFORM_AUDIT_DF

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object"
    )

    .filter(
        F.col("pipeline_run_id").isNotNull()
    )

    .filter(
        F.col("pipeline_name").isNotNull()
    )

    .filter(
        F.col("source_object").isNotNull()
    )

    .distinct()
)


GENERIC_GOLD_GRAIN_DF = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "source_system"
        ) != "QBO_ES"
    )

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object"
    )

    .distinct()
)


generic_grains_missing_from_gold = (
    GENERIC_EXPECTED_GRAIN_DF

    .join(
        GENERIC_GOLD_GRAIN_DF,

        on=[
            "pipeline_run_id",
            "pipeline_name",
            "source_system",
            "source_company",
            "source_object"
        ],

        how="left_anti"
    )

    .count()
)


# =============================================================================
# 21. REQUIRED OPERATIONAL IDENTIFIERS
# =============================================================================

null_pipeline_run_ids = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "pipeline_run_id"
        ).isNull()
    )

    .count()
)


null_pipeline_names = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "pipeline_name"
        ).isNull()
    )

    .count()
)


null_source_systems = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "source_system"
        ).isNull()
    )

    .count()
)


null_source_objects = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "source_object"
        ).isNull()
    )

    .count()
)


# =============================================================================
# 22. TIMING VALIDATION
# =============================================================================

negative_pipeline_durations = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "duration_seconds"
        ) < 0
    )

    .count()
)


invalid_pipeline_timestamps = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "run_started_utc"
        ).isNotNull()
        &
        F.col(
            "run_completed_utc"
        ).isNotNull()
        &
        (
            F.col(
                "run_completed_utc"
            )
            <
            F.col(
                "run_started_utc"
            )
        )
    )

    .count()
)


negative_cdc_durations = (
    FACT_CDC_ACTIVITY_GOLD_DF

    .filter(
        F.col(
            "extraction_duration_seconds"
        ) < 0
    )

    .count()
)


zero_pipeline_durations = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.coalesce(
            F.col(
                "duration_seconds"
            ),
            F.lit(0)
        ) == 0
    )

    .count()
)


# =============================================================================
# 23. PIPELINE STATUS VALIDATION
# =============================================================================

ALLOWED_PIPELINE_STATUSES = [
    "SUCCEEDED",
    "FAILED",
    "WARNING",
    "UNKNOWN"
]


invalid_pipeline_statuses = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "run_status"
        ).isNull()
        |
        (
            ~F.col(
                "run_status"
            ).isin(
                ALLOWED_PIPELINE_STATUSES
            )
        )
    )

    .count()
)


# =============================================================================
# 24. FRESHNESS VALIDATION
# =============================================================================

invalid_freshness_statuses = (
    AGG_SOURCE_FRESHNESS_GOLD_DF

    .filter(
        F.col(
            "freshness_status"
        ).isNull()
        |
        (
            ~F.col(
                "freshness_status"
            ).isin(
                "FRESH",
                "WARNING",
                "STALE",
                "UNKNOWN"
            )
        )
    )

    .count()
)


null_freshness_timestamps = (
    AGG_SOURCE_FRESHNESS_GOLD_DF

    .filter(
        F.col(
            "last_ingested_utc"
        ).isNull()
    )

    .count()
)


negative_freshness_ages = (
    AGG_SOURCE_FRESHNESS_GOLD_DF

    .filter(
        F.col(
            "freshness_age_hours"
        ) < 0
    )

    .count()
)


# =============================================================================
# 25. PIPELINE STATUS DISTRIBUTION
# =============================================================================

PIPELINE_STATUS_SUMMARY_DF = (
    FACT_PIPELINE_RUN_GOLD_DF

    .groupBy(
        "run_status"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "object_event_count"
        ),

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "pipeline_run_count"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "rows_written"
                ),
                F.lit(0)
            )
        ).alias(
            "rows_written"
        )
    )

    .orderBy(
        "run_status"
    )
)


# =============================================================================
# 26. PIPELINE EXECUTION SUMMARY
# =============================================================================

PIPELINE_EXECUTION_SUMMARY_DF = (
    FACT_PIPELINE_RUN_GOLD_DF

    .groupBy(
        "pipeline_key",
        "pipeline_name",
        "source_system",
        "source_company"
    )

    .agg(

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "pipeline_run_count"
        ),

        F.count(
            "*"
        ).alias(
            "object_event_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "SUCCEEDED",
                1
            ).otherwise(0)
        ).alias(
            "succeeded_object_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_object_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "WARNING",
                1
            ).otherwise(0)
        ).alias(
            "warning_object_count"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "rows_written"
                ),
                F.lit(0)
            )
        ).alias(
            "rows_written"
        ),

        F.avg(
            "duration_seconds"
        ).alias(
            "average_duration_seconds"
        ),

        F.max(
            "duration_seconds"
        ).alias(
            "maximum_duration_seconds"
        )
    )

    .orderBy(
        "source_system",
        "pipeline_name"
    )
)


# =============================================================================
# 27. FAILED PIPELINE OBJECTS
# =============================================================================

FAILED_PIPELINE_OBJECTS_DF = (
    FACT_PIPELINE_RUN_GOLD_DF

    .filter(
        F.col(
            "run_status"
        ) == "FAILED"
    )

    .select(
        "pipeline_run_id",
        "pipeline_name",
        "source_system",
        "source_company",
        "source_object",
        "run_started_utc",
        "run_completed_utc",
        "duration_seconds",
        "rows_written",
        "error_message"
    )

    .orderBy(
        F.col(
            "run_started_utc"
        ).desc_nulls_last()
    )
)


# =============================================================================
# 28. DATA QUALITY STATUS DISTRIBUTION
# =============================================================================

DQ_STATUS_SUMMARY_DF = (
    FACT_DATA_QUALITY_GOLD_DF

    .groupBy(
        "status"
    )

    .agg(
        F.count(
            "*"
        ).alias(
            "check_count"
        )
    )

    .orderBy(
        F.col(
            "check_count"
        ).desc()
    )
)


# =============================================================================
# 29. OPERATIONAL KPI SUMMARY
# =============================================================================

MONITORING_KPI_SUMMARY_DF = (
    FACT_PIPELINE_RUN_GOLD_DF

    .agg(

        F.countDistinct(
            "pipeline_run_id"
        ).alias(
            "total_pipeline_runs"
        ),

        F.count(
            "*"
        ).alias(
            "run_object_events"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "SUCCEEDED",
                1
            ).otherwise(0)
        ).alias(
            "successful_run_objects"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_run_objects"
        ),

        F.sum(
            F.when(
                F.col(
                    "run_status"
                ) == "WARNING",
                1
            ).otherwise(0)
        ).alias(
            "warning_run_objects"
        ),

        F.avg(
            "duration_seconds"
        ).alias(
            "average_duration_seconds"
        ),

        F.max(
            "duration_seconds"
        ).alias(
            "maximum_duration_seconds"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "rows_written"
                ),
                F.lit(0)
            )
        ).alias(
            "total_rows_written"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "records_changed"
                ),
                F.lit(0)
            )
        ).alias(
            "total_records_changed"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "records_upserted"
                ),
                F.lit(0)
            )
        ).alias(
            "total_records_upserted"
        )
    )
)


# =============================================================================
# 30. FINAL VALIDATION SUMMARY
# =============================================================================

print("=" * 110)
print("GOLD OPERATIONAL MONITORING — FINAL POST-LOAD VALIDATION")
print("=" * 110)


for table_name, actual_count in ACTUAL_COUNTS.items():

    print(
        f"{table_name:<35} : "
        f"{actual_count:,}"
    )


print("-" * 110)


print(
    f"Expected logical pipelines          : "
    f"{expected_logical_pipeline_count:,}"
)

print(
    f"Actual logical pipelines            : "
    f"{actual_logical_pipeline_count:,}"
)

print(
    f"Logical pipeline difference         : "
    f"{logical_pipeline_difference:,}"
)


print("-" * 110)


print(
    f"Row-count failures                  : "
    f"{len(row_count_failures)}"
)

print(
    f"Duplicate-key failures              : "
    f"{sum(duplicate_failures.values())}"
)

print(
    f"Null-key failures                   : "
    f"{sum(null_key_failures.values())}"
)


print("-" * 110)


print(
    f"Pipeline FK orphans                 : "
    f"{total_pipeline_fk_orphans}"
)

print(
    f"Data-object FK orphans              : "
    f"{total_data_object_fk_orphans}"
)


print("-" * 110)


print(
    f"DQ source/Gold difference           : "
    f"{dq_source_gold_difference}"
)

print(
    f"CDC source/Gold difference          : "
    f"{cdc_source_gold_difference}"
)

print(
    f"Generic grains missing from Gold    : "
    f"{generic_grains_missing_from_gold}"
)


print("-" * 110)


print(
    f"Null pipeline_run_id                : "
    f"{null_pipeline_run_ids}"
)

print(
    f"Null pipeline_name                  : "
    f"{null_pipeline_names}"
)

print(
    f"Null source_system                  : "
    f"{null_source_systems}"
)

print(
    f"Null source_object                  : "
    f"{null_source_objects}"
)


print("-" * 110)


print(
    f"Negative pipeline durations         : "
    f"{negative_pipeline_durations}"
)

print(
    f"Invalid pipeline timestamps         : "
    f"{invalid_pipeline_timestamps}"
)

print(
    f"Negative CDC durations              : "
    f"{negative_cdc_durations}"
)

print(
    f"Zero-duration pipeline events       : "
    f"{zero_pipeline_durations}"
)


print("-" * 110)


print(
    f"Invalid pipeline statuses           : "
    f"{invalid_pipeline_statuses}"
)

print(
    f"Invalid freshness statuses          : "
    f"{invalid_freshness_statuses}"
)

print(
    f"Null freshness timestamps           : "
    f"{null_freshness_timestamps}"
)

print(
    f"Negative freshness ages             : "
    f"{negative_freshness_ages}"
)


print("=" * 110)


# =============================================================================
# 31. BUILD FINAL FAILURE LIST
# =============================================================================

FINAL_FAILURES = []


if row_count_failures:

    FINAL_FAILURES.append(
        "ROW_COUNT"
    )


if logical_pipeline_difference != 0:

    FINAL_FAILURES.append(
        "LOGICAL_PIPELINE_COUNT"
    )


if any(
    value > 0
    for value in duplicate_failures.values()
):

    FINAL_FAILURES.append(
        "DUPLICATE_KEYS"
    )


if any(
    value > 0
    for value in null_key_failures.values()
):

    FINAL_FAILURES.append(
        "NULL_KEYS"
    )


if total_pipeline_fk_orphans > 0:

    FINAL_FAILURES.append(
        "PIPELINE_ORPHANS"
    )


if total_data_object_fk_orphans > 0:

    FINAL_FAILURES.append(
        "DATA_OBJECT_ORPHANS"
    )


if dq_source_gold_difference != 0:

    FINAL_FAILURES.append(
        "DQ_SOURCE_RECONCILIATION"
    )


if cdc_source_gold_difference != 0:

    FINAL_FAILURES.append(
        "CDC_SOURCE_RECONCILIATION"
    )


if generic_grains_missing_from_gold > 0:

    FINAL_FAILURES.append(
        "GENERIC_SOURCE_RECONCILIATION"
    )


if null_pipeline_run_ids > 0:

    FINAL_FAILURES.append(
        "NULL_PIPELINE_RUN_ID"
    )


if null_pipeline_names > 0:

    FINAL_FAILURES.append(
        "NULL_PIPELINE_NAME"
    )


if null_source_systems > 0:

    FINAL_FAILURES.append(
        "NULL_SOURCE_SYSTEM"
    )


if null_source_objects > 0:

    FINAL_FAILURES.append(
        "NULL_SOURCE_OBJECT"
    )


if negative_pipeline_durations > 0:

    FINAL_FAILURES.append(
        "NEGATIVE_PIPELINE_DURATION"
    )


if invalid_pipeline_timestamps > 0:

    FINAL_FAILURES.append(
        "INVALID_PIPELINE_TIMESTAMPS"
    )


if negative_cdc_durations > 0:

    FINAL_FAILURES.append(
        "NEGATIVE_CDC_DURATION"
    )


if invalid_pipeline_statuses > 0:

    FINAL_FAILURES.append(
        "INVALID_PIPELINE_STATUS"
    )


if invalid_freshness_statuses > 0:

    FINAL_FAILURES.append(
        "INVALID_FRESHNESS_STATUS"
    )


if negative_freshness_ages > 0:

    FINAL_FAILURES.append(
        "NEGATIVE_FRESHNESS_AGE"
    )


# =============================================================================
# 32. FAIL FAST
# =============================================================================

if FINAL_FAILURES:

    raise RuntimeError(
        "GOLD MONITORING FINAL VALIDATION FAILED: "
        +
        ", ".join(
            FINAL_FAILURES
        )
    )


# =============================================================================
# 33. DISPLAY PERSISTED PIPELINES
# =============================================================================

print("=" * 110)
print("PERSISTED LOGICAL PIPELINES")
print("=" * 110)

display(
    DIM_PIPELINE_GOLD_DF

    .orderBy(
        "source_system",
        "pipeline_name"
    )
)


# =============================================================================
# 34. DISPLAY PIPELINE STATUS DISTRIBUTION
# =============================================================================

print("=" * 110)
print("PIPELINE STATUS DISTRIBUTION")
print("=" * 110)

display(
    PIPELINE_STATUS_SUMMARY_DF
)


# =============================================================================
# 35. DISPLAY PIPELINE EXECUTION SUMMARY
# =============================================================================

print("=" * 110)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 110)

display(
    PIPELINE_EXECUTION_SUMMARY_DF
)


# =============================================================================
# 36. DISPLAY FAILED PIPELINE OBJECTS
# =============================================================================

print("=" * 110)
print("FAILED PIPELINE OBJECTS")
print("=" * 110)

display(
    FAILED_PIPELINE_OBJECTS_DF
)


# =============================================================================
# 37. DISPLAY DATA QUALITY STATUS
# =============================================================================

print("=" * 110)
print("DATA QUALITY STATUS DISTRIBUTION")
print("=" * 110)

display(
    DQ_STATUS_SUMMARY_DF
)


# =============================================================================
# 38. DISPLAY OPERATIONAL KPI SUMMARY
# =============================================================================

print("=" * 110)
print("MONITORING KPI SUMMARY")
print("=" * 110)

display(
    MONITORING_KPI_SUMMARY_DF
)


# =============================================================================
# 39. FINAL SUCCESS
# =============================================================================

print("=" * 110)

print(
    "GOLD OPERATIONAL MONITORING — "
    "FINAL MULTI-SOURCE POST-LOAD VALIDATION: SUCCEEDED"
)

print("=" * 110)

StatementMeta(, 697e6fce-ed6c-4ea7-a5db-8cc0f9c213bf, 19, Finished, Available, Finished, False)

GOLD OPERATIONAL MONITORING — FINAL POST-LOAD VALIDATION
dim_pipeline                        : 3
dim_data_object                     : 69
fact_pipeline_run                   : 578
fact_data_quality                   : 3,586
fact_cdc_activity                   : 4,790
agg_source_freshness                : 36
--------------------------------------------------------------------------------------------------------------
Expected logical pipelines          : 3
Actual logical pipelines            : 3
Logical pipeline difference         : 0
--------------------------------------------------------------------------------------------------------------
Row-count failures                  : 0
Duplicate-key failures              : 0
Null-key failures                   : 0
--------------------------------------------------------------------------------------------------------------
Pipeline FK orphans                 : 0
Data-object FK orphans              : 0
--------------------------------------

SynapseWidget(Synapse.DataFrame, 59c91c0b-ab75-4bff-998c-6dfb9855c51b)

PIPELINE STATUS DISTRIBUTION


SynapseWidget(Synapse.DataFrame, 5cd11496-d865-4e58-9f7f-087ed3473062)

...

DATA QUALITY STATUS DISTRIBUTION


SynapseWidget(Synapse.DataFrame, 08d0f1b3-f330-4979-9b21-0483973817aa)

MONITORING KPI SUMMARY


SynapseWidget(Synapse.DataFrame, 0b4ba498-0319-4f6a-a9c0-68882a5e9124)

GOLD OPERATIONAL MONITORING — FINAL MULTI-SOURCE POST-LOAD VALIDATION: SUCCEEDED
